<a href="https://colab.research.google.com/github/ankitkumarsahu454554-svg/Rare_Disease_Ai/blob/main/Rare_Disease_Prediction_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

required_files = [
    DATA / "clean_hpo_annotations.csv",
    DATA / "disease_level_split.csv",
    DATA / "three_model_comparison_summary.csv",
    DATA / "rare_disease_training_graph.graphml",
    DATA / "rare_disease_gnn_training_only.pt",
    PROJECT / "hp.obo",
]

print("\nPROJECT FILE CHECK")
missing = []

for path in required_files:
    status = "OK" if path.exists() else "MISSING"
    print(f"{status}: {path}")
    if not path.exists():
        missing.append(path.name)

if missing:
    print("\nMissing files:", missing)
else:
    print("\nAll required files are available.")


Mounted at /content/drive

PROJECT FILE CHECK
OK: /content/drive/MyDrive/Rare Disease ai/processed/clean_hpo_annotations.csv
OK: /content/drive/MyDrive/Rare Disease ai/processed/disease_level_split.csv
OK: /content/drive/MyDrive/Rare Disease ai/processed/three_model_comparison_summary.csv
OK: /content/drive/MyDrive/Rare Disease ai/processed/rare_disease_training_graph.graphml
OK: /content/drive/MyDrive/Rare Disease ai/processed/rare_disease_gnn_training_only.pt
OK: /content/drive/MyDrive/Rare Disease ai/hp.obo

All required files are available.


In [ ]:

import pandas as pd
import numpy as np

annotations = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False,
    dtype={"database_id": "string", "hpo_id": "string"}
).dropna(subset=["database_id", "hpo_id"])

split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype={"disease_id": "string"}
)

comparison = pd.read_csv(
    DATA / "three_model_comparison_summary.csv"
)

split["split"] = split["split"].str.lower().str.strip()

train_ids = set(split.loc[split["split"] == "train", "disease_id"])
test_ids = set(split.loc[split["split"] == "test", "disease_id"])

print("ANNOTATION ROWS:", len(annotations))
print("DISEASES:", annotations["database_id"].nunique())
print("PHENOTYPES:", annotations["hpo_id"].nunique())
print("TRAINING DISEASES:", len(train_ids))
print("HELD-OUT DISEASES:", len(test_ids))
print("TRAIN/TEST OVERLAP:", len(train_ids & test_ids))

print("\nEXISTING BENCHMARK")
print(comparison.to_string(index=False))

assert not (train_ids & test_ids), "Train/test leakage detected!"
print("\nVALIDATION CHECK PASSED")


ANNOTATION ROWS: 286047
DISEASES: 12879
PHENOTYPES: 11658
TRAINING DISEASES: 7340
HELD-OUT DISEASES: 1836
TRAIN/TEST OVERLAP: 0

EXISTING BENCHMARK
                   Model  Profiles  Mean Recall@10  Mean MRR
          Phenotype-only      1827        0.213532  0.520367
           Gene-informed      1827        0.133764  0.379349
GNN phenotype embeddings      1827        0.086528  0.278151

VALIDATION CHECK PASSED


In [ ]:

from collections import defaultdict
import numpy as np
import pandas as pd

# Build disease phenotype profiles from the curated knowledge base.
# This is for knowledge-base disease ranking, not unseen-disease generalization.
profiles = (
    annotations.groupby("database_id")["hpo_id"]
    .apply(lambda x: set(x.dropna().astype(str)))
    .to_dict()
)

# Information content: rarity of each phenotype across disease profiles.
term_count = defaultdict(int)
for terms in profiles.values():
    for term in terms:
        term_count[term] += 1

n_diseases = len(profiles)
ic = {
    term: -np.log(count / n_diseases)
    for term, count in term_count.items()
    if count > 0
}

def rank_diseases(input_phenotypes, top_k=10):
    """Rank known diseases by information-weighted phenotype overlap."""
    input_terms = set(input_phenotypes) & set(ic)
    if not input_terms:
        return pd.DataFrame(columns=["disease_id", "score"])

    denominator = sum(ic[t] for t in input_terms)
    if denominator <= 0:
        denominator = len(input_terms)

    results = []
    for disease_id, disease_terms in profiles.items():
        overlap = input_terms & disease_terms
        score = sum(ic[t] for t in overlap) / denominator
        results.append((disease_id, score))

    results.sort(key=lambda x: x[1], reverse=True)
    return pd.DataFrame(
        results[:top_k], columns=["disease_id", "score"]
    )

# Simulate patient profiles by hiding one phenotype from each eligible
# held-out disease. The full curated profiles remain the candidate database.
rng = np.random.default_rng(42)
test_disease_ids = sorted(test_ids)
evaluation = []

for disease_id in test_disease_ids:
    full_terms = profiles.get(disease_id, set())
    known_terms = sorted(full_terms & set(ic))

    if len(known_terms) < 2:
        continue

    # Hold out one phenotype as the simulated unknown finding.
    hidden_term = known_terms[int(rng.integers(0, len(known_terms)))]
    input_terms = set(known_terms) - {hidden_term}

    ranked = rank_diseases(input_terms, top_k=10)
    ranked_ids = ranked["disease_id"].tolist()

    # The correct disease is the target; report disease-level ranking metrics.
    full_ranking = rank_diseases(input_terms, top_k=len(profiles))
    all_ranked_ids = full_ranking["disease_id"].tolist()

    rank = (
        all_ranked_ids.index(disease_id) + 1
        if disease_id in all_ranked_ids else None
    )

    evaluation.append({
        "disease_id": disease_id,
        "input_phenotype_count": len(input_terms),
        "hidden_phenotype": hidden_term,
        "correct_disease_rank": rank,
        "recall_at_1": int(rank is not None and rank <= 1),
        "recall_at_5": int(rank is not None and rank <= 5),
        "recall_at_10": int(rank is not None and rank <= 10),
        "reciprocal_rank": 1 / rank if rank else 0.0,
    })

eval_df = pd.DataFrame(evaluation)
results_path = DATA / "phenotype_disease_ranking_evaluation.csv"
eval_df.to_csv(results_path, index=False)

print("Evaluation profiles:", len(eval_df))
print("Results saved to:", results_path)

if len(eval_df):
    print("\nDisease-ranking metrics")
    print("Recall@1: ", round(eval_df["recall_at_1"].mean(), 4))
    print("Recall@5: ", round(eval_df["recall_at_5"].mean(), 4))
    print("Recall@10:", round(eval_df["recall_at_10"].mean(), 4))
    print("MRR:      ", round(eval_df["reciprocal_rank"].mean(), 4))
else:
    print("No eligible profiles; check the disease and phenotype data.")


Evaluation profiles: 1832
Results saved to: /content/drive/MyDrive/Rare Disease ai/processed/phenotype_disease_ranking_evaluation.csv

Disease-ranking metrics
Recall@1:  0.9301
Recall@5:  0.9733
Recall@10: 0.9776
MRR:       0.9487


In [ ]:

import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

split = pd.read_csv(DATA / "disease_level_split.csv")
results = pd.read_csv(
    DATA / "phenotype_disease_ranking_evaluation.csv"
)

print("Split columns:", split.columns.tolist())
print("Split counts:")
print(split["split"].value_counts().to_string())

print("\nEvaluation columns:", results.columns.tolist())
print("Evaluation profiles:", len(results))
print("Unique evaluated diseases:",
      results["disease_id"].nunique())

print("\nSaved results are preliminary; do not interpret them as diagnostic accuracy.")


Split columns: ['disease_id', 'split']
Split counts:
split
train    7340
test     1836

Evaluation columns: ['disease_id', 'input_phenotype_count', 'hidden_phenotype', 'correct_disease_rank', 'recall_at_1', 'recall_at_5', 'recall_at_10', 'reciprocal_rank']
Evaluation profiles: 1832
Unique evaluated diseases: 1832

Saved results are preliminary; do not interpret them as diagnostic accuracy.


In [ ]:

import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False,
    dtype={"database_id": "string", "hpo_id": "string"}
).dropna(subset=["database_id", "hpo_id"])

split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype={"disease_id": "string"}
)
split["split"] = split["split"].str.lower().str.strip()

train_ids = set(split.loc[split["split"] == "train", "disease_id"])
test_ids = set(split.loc[split["split"] == "test", "disease_id"])

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()

profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()
train_profiles = {d: profiles[d] for d in train_ids if d in profiles}

# Weight phenotypes by rarity across training disease profiles only.
counts = defaultdict(int)
for terms in train_profiles.values():
    for term in terms:
        counts[term] += 1

n = max(len(train_profiles), 1)
ic = {t: -np.log(c / n) for t, c in counts.items() if c > 0}

# Test only diseases with enough phenotypes to form input and target.
rng = np.random.default_rng(42)
rows = []

for disease in sorted(test_ids):
    terms = sorted(profiles.get(disease, set()) & set(ic))
    if len(terms) < 2:
        continue

    # Randomly hide 20% of terms, leaving at least one input term.
    n_hide = max(1, int(round(0.2 * len(terms))))
    n_hide = min(n_hide, len(terms) - 1)
    hidden_idx = set(rng.choice(len(terms), size=n_hide, replace=False))
    hidden = {terms[i] for i in hidden_idx}
    observed = set(terms) - hidden

    # Candidate list contains training diseases only.
    scored = []
    denom = sum(ic[t] for t in observed) or 1.0

    for candidate, candidate_terms in train_profiles.items():
        overlap = observed & candidate_terms
        score = sum(ic[t] for t in overlap) / denom
        scored.append((candidate, score))

    scored.sort(key=lambda x: x[1], reverse=True)

    # Because test disease IDs are excluded from candidates, direct
    # correct-disease rank is undefined for this split.
    # Record similarity to the nearest training disease for diagnostics.
    best_candidate, best_score = scored[0] if scored else (None, 0.0)

    rows.append({
        "test_disease_id": disease,
        "input_phenotypes": len(observed),
        "hidden_phenotypes": len(hidden),
        "nearest_training_disease": best_candidate,
        "nearest_training_score": best_score,
    })

out = pd.DataFrame(rows)
path = DATA / "heldout_profile_nearest_training_disease.csv"
out.to_csv(path, index=False)

print("Held-out profiles evaluated:", len(out))
print("Saved:", path)
print("\nThis is a nearest-training-disease diagnostic only.")
print("It does not provide Recall@K or MRR for the correct held-out disease,")
print("because the held-out disease is intentionally absent from candidates.")


Held-out profiles evaluated: 1829
Saved: /content/drive/MyDrive/Rare Disease ai/processed/heldout_profile_nearest_training_disease.csv

This is a nearest-training-disease diagnostic only.
It does not provide Recall@K or MRR for the correct held-out disease,
because the held-out disease is intentionally absent from candidates.


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False,
    dtype={"database_id": "string", "hpo_id": "string"}
).dropna(subset=["database_id", "hpo_id"])

split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype={"disease_id": "string"}
)
split["split"] = split["split"].str.lower().str.strip()

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()

train_ids = set(split.loc[split["split"] == "train", "disease_id"])
test_ids = set(split.loc[split["split"] == "test", "disease_id"])

profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()
train_profiles = {d: profiles[d] for d in train_ids if d in profiles}
test_profiles = {d: profiles[d] for d in test_ids if d in profiles}

# Estimate phenotype information content using training diseases only.
freq = {}
for terms in train_profiles.values():
    for term in terms:
        freq[term] = freq.get(term, 0) + 1

n_train = max(len(train_profiles), 1)
ic = {t: -np.log(c / n_train) for t, c in freq.items()}

# Score all known disease profiles using the same phenotype-based rule.
# Candidate annotations are used for retrieval; no test disease identity
# embeddings are trained.
candidate_profiles = {
    **train_profiles,
    **{d: test_profiles[d] for d in test_profiles}
}

rng = np.random.default_rng(42)
rows = []

for disease_id, full_terms in sorted(test_profiles.items()):
    terms = sorted(set(full_terms) & set(ic))
    if len(terms) < 2:
        continue

    # Hide 20% of known phenotypes to simulate an incomplete profile.
    n_hide = min(max(1, round(0.2 * len(terms))), len(terms) - 1)
    hidden_idx = set(rng.choice(len(terms), size=n_hide, replace=False))
    observed = {t for i, t in enumerate(terms) if i not in hidden_idx}

    weights = {t: ic[t] for t in observed}
    denom = sum(weights.values()) or 1.0

    scores = []
    for candidate_id, candidate_terms in candidate_profiles.items():
        overlap = observed & candidate_terms
        score = sum(weights[t] for t in overlap) / denom
        scores.append((candidate_id, score))

    # Stable deterministic tie handling.
    scores.sort(key=lambda x: (-x[1], x[0]))
    ranking = [candidate for candidate, _ in scores]
    rank = ranking.index(disease_id) + 1

    rows.append({
        "disease_id": disease_id,
        "rank": rank,
        "recall_at_1": int(rank <= 1),
        "recall_at_5": int(rank <= 5),
        "recall_at_10": int(rank <= 10),
        "reciprocal_rank": 1 / rank
    })

metrics = pd.DataFrame(rows)
out_file = DATA / "disease_ranking_retrieval_metrics.csv"
metrics.to_csv(out_file, index=False)

print("Evaluated profiles:", len(metrics))
print("Recall@1:", round(metrics["recall_at_1"].mean(), 4))
print("Recall@5:", round(metrics["recall_at_5"].mean(), 4))
print("Recall@10:", round(metrics["recall_at_10"].mean(), 4))
print("MRR:", round(metrics["reciprocal_rank"].mean(), 4))
print("Saved:", out_file)
print("\nInterpretation: known-profile retrieval only; not clinical diagnostic accuracy.")


Evaluated profiles: 1829
Recall@1: 0.9333
Recall@5: 0.9727
Recall@10: 0.9787
MRR: 0.9503
Saved: /content/drive/MyDrive/Rare Disease ai/processed/disease_ranking_retrieval_metrics.csv

Interpretation: known-profile retrieval only; not clinical diagnostic accuracy.


In [ ]:

import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

results = pd.read_csv(DATA / "disease_ranking_retrieval_metrics.csv")

print("Profiles:", len(results))
print("Duplicate disease IDs:", results["disease_id"].duplicated().sum())
print("\nMetric summary:")
print(results[[
    "recall_at_1",
    "recall_at_5",
    "recall_at_10",
    "reciprocal_rank"
]].agg(["mean", "median", "std"]).round(4))

print("\nSaved baseline:", DATA / "disease_ranking_retrieval_metrics.csv")


Profiles: 1829
Duplicate disease IDs: 0

Metric summary:
        recall_at_1  recall_at_5  recall_at_10  reciprocal_rank
mean         0.9333       0.9727        0.9787           0.9503
median       1.0000       1.0000        1.0000           1.0000
std          0.2496       0.1631        0.1445           0.1926

Saved baseline: /content/drive/MyDrive/Rare Disease ai/processed/disease_ranking_retrieval_metrics.csv


In [ ]:

from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
ROOT = Path("/content/drive/MyDrive/Rare Disease ai")

ANN = DATA / "clean_hpo_annotations.csv"
SPLIT = DATA / "disease_level_split.csv"
OBO = ROOT / "hp.obo"

ann = pd.read_csv(
    ANN, low_memory=False,
    dtype={"database_id": "string", "hpo_id": "string"}
).dropna(subset=["database_id", "hpo_id"])

split = pd.read_csv(SPLIT, dtype={"disease_id": "string"})
split["split"] = split["split"].str.lower().str.strip()

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()

train_ids = set(split.loc[split["split"] == "train", "disease_id"])
test_ids = set(split.loc[split["split"] == "test", "disease_id"])

profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()
train_profiles = {d: profiles[d] for d in train_ids if d in profiles}
test_profiles = {d: profiles[d] for d in test_ids if d in profiles}

# Parse HPO parent relationships.
parents = defaultdict(set)
current_id = None
in_term = False

with open(OBO, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line == "[Term]":
            current_id, in_term = None, True
        elif line.startswith("[") and line.endswith("]"):
            in_term = False
        elif in_term:
            if line.startswith("id: "):
                current_id = line[4:]
            elif line.startswith("is_a: ") and current_id:
                parents[current_id].add(line.split()[1])

ancestor_cache = {}

def ancestors(term):
    if term not in ancestor_cache:
        seen, stack = set(), list(parents.get(term, []))
        while stack:
            node = stack.pop()
            if node not in seen:
                seen.add(node)
                stack.extend(parents.get(node, []))
        seen.add(term)
        ancestor_cache[term] = seen
    return ancestor_cache[term]

# Train-only information content over propagated phenotype profiles.
propagated_train = {}
for disease, terms in train_profiles.items():
    expanded = set()
    for term in terms:
        expanded.update(ancestors(term))
    propagated_train[disease] = expanded

counts = defaultdict(int)
for terms in propagated_train.values():
    for term in terms:
        counts[term] += 1

n_train = max(len(propagated_train), 1)
ic = {t: -np.log(c / n_train) for t, c in counts.items() if c > 0}

# Candidate profiles use ontology propagation. Test disease annotations
# are used only as the target profile in this retrieval benchmark.
candidate_profiles = {}
for disease, terms in profiles.items():
    expanded = set()
    for term in terms:
        expanded.update(ancestors(term))
    candidate_profiles[disease] = expanded

rng = np.random.default_rng(42)
rows = []

for disease, raw_terms in sorted(test_profiles.items()):
    known = sorted(set(raw_terms) & set(ic))
    if len(known) < 2:
        continue

    hide_n = min(max(1, round(0.2 * len(known))), len(known) - 1)
    hidden_idx = set(rng.choice(len(known), size=hide_n, replace=False))
    observed = {t for i, t in enumerate(known) if i not in hidden_idx}

    observed_expanded = set()
    for term in observed:
        observed_expanded.update(ancestors(term))

    weights = {t: ic.get(t, 0.0) for t in observed}
    denom = sum(weights.values()) or 1.0

    scores = []
    for candidate, candidate_terms in candidate_profiles.items():
        # Direct and ancestor overlap, weighted by the specificity
        # of each observed term. Each observed term contributes at most once.
        score = sum(
            weight for term, weight in weights.items()
            if ancestors(term) & candidate_terms
        ) / denom
        scores.append((candidate, score))

    scores.sort(key=lambda x: (-x[1], x[0]))
    ranking = [candidate for candidate, _ in scores]
    rank = ranking.index(disease) + 1

    rows.append({
        "disease_id": disease,
        "rank": rank,
        "recall_at_1": int(rank <= 1),
        "recall_at_5": int(rank <= 5),
        "recall_at_10": int(rank <= 10),
        "reciprocal_rank": 1 / rank
    })

result = pd.DataFrame(rows)
output = DATA / "hierarchy_aware_retrieval_metrics.csv"
result.to_csv(output, index=False)

print("Profiles evaluated:", len(result))
print("Recall@1:", round(result["recall_at_1"].mean(), 4))
print("Recall@5:", round(result["recall_at_5"].mean(), 4))
print("Recall@10:", round(result["recall_at_10"].mean(), 4))
print("MRR:", round(result["reciprocal_rank"].mean(), 4))
print("Saved:", output)
print("Interpret as knowledge-base retrieval, not diagnostic accuracy.")


Profiles evaluated: 1829
Recall@1: 0.0
Recall@5: 0.0
Recall@10: 0.0
MRR: 0.0004
Saved: /content/drive/MyDrive/Rare Disease ai/processed/hierarchy_aware_retrieval_metrics.csv
Interpret as knowledge-base retrieval, not diagnostic accuracy.


In [ ]:

from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
ROOT = Path("/content/drive/MyDrive/Rare Disease ai")

ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False,
    dtype={"database_id": "string", "hpo_id": "string"}
).dropna(subset=["database_id", "hpo_id"])

split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype={"disease_id": "string"}
)
split["split"] = split["split"].str.lower().str.strip()

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()

test_ids = set(split.loc[split["split"] == "test", "disease_id"])
profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()

# Estimate information content from training diseases only.
train_ids = set(split.loc[split["split"] == "train", "disease_id"])
train_profiles = {d: profiles[d] for d in train_ids if d in profiles}

freq = defaultdict(int)
for terms in train_profiles.values():
    for term in terms:
        freq[term] += 1

n_train = max(len(train_profiles), 1)
ic = {t: -np.log(c / n_train) for t, c in freq.items() if c > 0}

# Parse HPO parent links.
obo = ROOT / "hp.obo"
parents = defaultdict(set)
current_id = None
in_term = False

with open(obo, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line == "[Term]":
            current_id, in_term = None, True
        elif line.startswith("[") and line.endswith("]"):
            in_term = False
        elif in_term:
            if line.startswith("id: "):
                current_id = line[4:]
            elif line.startswith("is_a: ") and current_id:
                parents[current_id].add(line.split()[1])

cache = {}

def ancestors(term):
    if term not in cache:
        seen = set()
        stack = list(parents.get(term, []))
        while stack:
            node = stack.pop()
            if node not in seen:
                seen.add(node)
                stack.extend(parents.get(node, []))
        seen.add(term)
        cache[term] = seen
    return cache[term]

# Candidate disease profiles are curated knowledge-base profiles.
candidates = {
    d: set(terms) for d, terms in profiles.items()
    if terms
}

rng = np.random.default_rng(42)
records = []

for disease in sorted(test_ids):
    full = sorted(set(profiles.get(disease, set())) & set(ic))
    if len(full) < 2:
        continue

    hide_n = min(max(1, round(0.2 * len(full))), len(full) - 1)
    hidden_idx = set(rng.choice(len(full), size=hide_n, replace=False))
    observed = {t for i, t in enumerate(full) if i not in hidden_idx}
    weights = {t: ic[t] for t in observed}
    denom = sum(weights.values()) or 1.0

    direct_scores = []
    hierarchy_scores = []

    for candidate, candidate_terms in candidates.items():
        # Method A: direct phenotype overlap.
        direct = sum(
            w for term, w in weights.items()
            if term in candidate_terms
        ) / denom

        # Method B: each input term is matched to a candidate term
        # only if the terms are identical or one is an ancestor of the other.
        # Similarity is discounted for broader, less specific matches.
        candidate_ancestors = set()
        for term in candidate_terms:
            candidate_ancestors.update(ancestors(term))

        hscore = 0.0
        for term, w in weights.items():
            if term in candidate_terms:
                hscore += w
            elif ancestors(term) & candidate_terms:
                # Match to a broader candidate phenotype: partial credit.
                hscore += 0.25 * w
            elif term in candidate_ancestors:
                # Candidate phenotype is more specific than the input term.
                hscore += 0.5 * w

        hierarchy = hscore / denom
        direct_scores.append((candidate, direct))
        hierarchy_scores.append((candidate, hierarchy))

    def get_rank(scored):
        # Deterministic tie handling; disease IDs break score ties.
        scored.sort(key=lambda x: (-x[1], x[0]))
        ranked_ids = [x[0] for x in scored]
        return ranked_ids.index(disease) + 1

    rd = get_rank(direct_scores)
    rh = get_rank(hierarchy_scores)

    records.append({
        "disease_id": disease,
        "direct_rank": rd,
        "direct_reciprocal_rank": 1 / rd,
        "direct_recall_at_1": int(rd <= 1),
        "direct_recall_at_5": int(rd <= 5),
        "direct_recall_at_10": int(rd <= 10),
        "hierarchy_rank": rh,
        "hierarchy_reciprocal_rank": 1 / rh,
        "hierarchy_recall_at_1": int(rh <= 1),
        "hierarchy_recall_at_5": int(rh <= 5),
        "hierarchy_recall_at_10": int(rh <= 10),
    })

result = pd.DataFrame(records)
out_path = DATA / "direct_vs_hierarchy_scoring_comparison.csv"
result.to_csv(out_path, index=False)

summary = pd.DataFrame([
    {
        "method": "Direct overlap",
        "profiles": len(result),
        "Recall@1": result["direct_recall_at_1"].mean(),
        "Recall@5": result["direct_recall_at_5"].mean(),
        "Recall@10": result["direct_recall_at_10"].mean(),
        "MRR": result["direct_reciprocal_rank"].mean(),
    },
    {
        "method": "HPO-aware",
        "profiles": len(result),
        "Recall@1": result["hierarchy_recall_at_1"].mean(),
        "Recall@5": result["hierarchy_recall_at_5"].mean(),
        "Recall@10": result["hierarchy_recall_at_10"].mean(),
        "MRR": result["hierarchy_reciprocal_rank"].mean(),
    },
])

print(summary.round(4).to_string(index=False))
print("\nSaved detailed comparison:", out_path)
print("Note: curated-profile retrieval only, not clinical diagnostic accuracy.")


        method  profiles  Recall@1  Recall@5  Recall@10    MRR
Direct overlap      1829    0.9306    0.9716     0.9787 0.9483
     HPO-aware      1829    0.9306    0.9716     0.9787 0.9483

Saved detailed comparison: /content/drive/MyDrive/Rare Disease ai/processed/direct_vs_hierarchy_scoring_comparison.csv
Note: curated-profile retrieval only, not clinical diagnostic accuracy.


In [ ]:

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

candidates = [
    PROJECT / "genes_to_disease.txt",
    DATA / "genes_to_disease.txt",
    PROJECT / "genes_to_phenotype.txt",
]

for path in candidates:
    print(("FOUND " if path.exists() else "MISSING "), path)


FOUND  /content/drive/MyDrive/Rare Disease ai/genes_to_disease.txt
MISSING  /content/drive/MyDrive/Rare Disease ai/processed/genes_to_disease.txt
FOUND  /content/drive/MyDrive/Rare Disease ai/genes_to_phenotype.txt


In [ ]:

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
gene_file = PROJECT / "genes_to_disease.txt"

with open(gene_file, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        print(line.rstrip())
        if i >= 7:
            break


ncbi_gene_id	gene_symbol	association_type	disease_id	source
NCBIGene:64170	CARD9	MENDELIAN	OMIM:212050	ftp://ftp.ncbi.nlm.nih.gov/gene/DATA/mim2gene_medgen
NCBIGene:51256	TBC1D7	MENDELIAN	OMIM:248000	ftp://ftp.ncbi.nlm.nih.gov/gene/DATA/mim2gene_medgen
NCBIGene:28981	IFT81	MENDELIAN	OMIM:617895	ftp://ftp.ncbi.nlm.nih.gov/gene/DATA/mim2gene_medgen
NCBIGene:8216	LZTR1	MENDELIAN	OMIM:616564	ftp://ftp.ncbi.nlm.nih.gov/gene/DATA/mim2gene_medgen
NCBIGene:6505	SLC1A1	POLYGENIC	OMIM:615232	ftp://ftp.ncbi.nlm.nih.gov/gene/DATA/mim2gene_medgen
NCBIGene:5949	RBP3	MENDELIAN	OMIM:615233	ftp://ftp.ncbi.nlm.nih.gov/gene/DATA/mim2gene_medgen
NCBIGene:583	BBS2	MENDELIAN	OMIM:616562	ftp://ftp.ncbi.nlm.nih.gov/gene/DATA/mim2gene_medgen


In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
GENE_FILE = PROJECT / "genes_to_disease.txt"

# Read the tab-separated gene-disease file.
gene_df = pd.read_csv(
    GENE_FILE,
    sep="\t",
    comment="#",
    dtype=str,
    low_memory=False
)

gene_df.columns = gene_df.columns.str.strip()
gene_df["disease_id"] = gene_df["disease_id"].str.strip()
gene_df["gene_symbol"] = gene_df["gene_symbol"].str.strip()

gene_df = gene_df.dropna(subset=["disease_id", "gene_symbol"])
gene_df = gene_df[
    (gene_df["disease_id"] != "") &
    (gene_df["gene_symbol"] != "")
].drop_duplicates(["disease_id", "gene_symbol"])

# Save a normalized mapping for later inference.
gene_map = (
    gene_df.groupby("disease_id")["gene_symbol"]
    .apply(lambda x: sorted(set(x)))
    .to_dict()
)

mapping = pd.DataFrame([
    {"disease_id": disease, "gene_symbols": ";".join(genes)}
    for disease, genes in gene_map.items()
])

mapping_path = DATA / "normalized_disease_gene_mapping.csv"
mapping.to_csv(mapping_path, index=False)

print("Gene-disease associations:", len(gene_df))
print("Diseases with gene associations:", len(gene_map))
print("Unique gene symbols:", gene_df["gene_symbol"].nunique())
print("Association types:")
print(gene_df["association_type"].value_counts(dropna=False).to_string())
print("Saved:", mapping_path)

print("\nImportant: these are disease-associated genes, not patient-specific genetic findings.")


Gene-disease associations: 16097
Diseases with gene associations: 11168
Unique gene symbols: 5550
Association types:
association_type
UNKNOWN      8423
MENDELIAN    7093
POLYGENIC     581
Saved: /content/drive/MyDrive/Rare Disease ai/processed/normalized_disease_gene_mapping.csv

Important: these are disease-associated genes, not patient-specific genetic findings.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

# Load disease phenotype profiles.
ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False,
    dtype={"database_id": "string", "hpo_id": "string"}
).dropna(subset=["database_id", "hpo_id"])

phenotype_map = (
    ann.groupby("database_id")["hpo_id"]
    .apply(set)
    .to_dict()
)

# Load normalized disease-gene mapping.
gene_df = pd.read_csv(
    DATA / "normalized_disease_gene_mapping.csv",
    dtype=str
).fillna("")

gene_map = {
    row["disease_id"]: set(
        g.strip().upper()
        for g in row["gene_symbols"].split(";")
        if g.strip()
    )
    for _, row in gene_df.iterrows()
}

# Estimate phenotype information content from training diseases only.
split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype={"disease_id": "string"}
)
train_ids = set(
    split.loc[split["split"].str.lower() == "train", "disease_id"]
)

term_frequency = {}
for disease in train_ids:
    for term in phenotype_map.get(disease, set()):
        term_frequency[term] = term_frequency.get(term, 0) + 1

n_train = max(len(train_ids), 1)
ic = {
    term: -np.log(count / n_train)
    for term, count in term_frequency.items()
    if count > 0
}

def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,
                     gene_weight=0.25):
    """
    Rank diseases from HPO phenotype IDs and optional patient candidate genes.

    Scores are ranking scores, NOT calibrated probabilities.
    Patient genes should be supported by actual genetic analysis.
    """
    observed = {
        str(term).strip()
        for term in patient_hpo_terms
        if str(term).strip() in ic
    }

    if not observed:
        raise ValueError(
            "No recognized HPO terms. Supply valid IDs such as HP:0001250."
        )

    weights = {term: ic[term] for term in observed}
    denom = sum(weights.values()) or 1.0

    genes = {
        str(gene).strip().upper()
        for gene in (patient_genes or [])
        if str(gene).strip()
    }

    scored = []
    for disease, disease_terms in phenotype_map.items():
        overlap = observed & disease_terms
        phenotype_score = sum(weights[t] for t in overlap) / denom

        disease_genes = gene_map.get(disease, set())
        gene_score = (
            len(genes & disease_genes) / len(genes)
            if genes else 0.0
        )

        # Use phenotype-only scoring when patient genes are unavailable.
        final_score = (
            (1 - gene_weight) * phenotype_score + gene_weight * gene_score
            if genes else phenotype_score
        )

        scored.append({
            "disease_id": disease,
            "phenotype_score": phenotype_score,
            "gene_score": gene_score if genes else np.nan,
            "combined_score": final_score,
            "matching_phenotypes": len(overlap),
            "matching_genes": (
                ";".join(sorted(genes & disease_genes)) if genes else ""
            )
        })

    result = pd.DataFrame(scored).sort_values(
        ["combined_score", "phenotype_score", "disease_id"],
        ascending=[False, False, True]
    ).head(top_k).reset_index(drop=True)

    result.insert(0, "rank", range(1, len(result) + 1))
    return result

print("Prediction function ready.")
print("Phenotype profiles:", len(phenotype_map))
print("Diseases with gene mappings:", len(gene_map))
print("\nExample call:")
print("predict_diseases(['HP:0001250', 'HP:0001263'], top_k=10)")


Prediction function ready.
Phenotype profiles: 12879
Diseases with gene mappings: 11168

Example call:
predict_diseases(['HP:0001250', 'HP:0001263'], top_k=10)


In [ ]:
predict_diseases(
    patient_hpo_terms=["HP:0001250", "HP:0001263"],
    top_k=10
)

,rank,disease_id,phenotype_score,gene_score,combined_score,matching_phenotypes,matching_genes
0,1,DECIPHER:45,1.0,NaN,1.0,2,
1,2,OMIM:100300,1.0,NaN,1.0,2,
2,3,OMIM:103050,1.0,NaN,1.0,2,
3,4,OMIM:105830,1.0,NaN,1.0,2,
4,5,OMIM:114290,1.0,NaN,1.0,2,
5,6,OMIM:117360,1.0,NaN,1.0,2,
6,7,OMIM:117550,1.0,NaN,1.0,2,
7,8,OMIM:122470,1.0,NaN,1.0,2,
8,9,OMIM:124000,1.0,NaN,1.0,2,
9,10,OMIM:135900,1.0,NaN,1.0,2,


In [ ]:

def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,
                     gene_weight=0.25):
    observed = {
        str(t).strip()
        for t in patient_hpo_terms
        if str(t).strip() in ic
    }

    if not observed:
        raise ValueError("No recognized HPO terms. Use valid HPO IDs.")

    weights = {t: ic[t] for t in observed}
    input_weight = sum(weights.values()) or 1.0

    genes = {
        str(g).strip().upper()
        for g in (patient_genes or [])
        if str(g).strip()
    }

    results = []

    for disease, disease_terms in phenotype_map.items():
        overlap = observed & disease_terms

        # How much of the patient's weighted phenotype input is explained?
        precision = sum(weights[t] for t in overlap) / input_weight

        # How much of the candidate disease profile is explained?
        recall = len(overlap) / max(len(disease_terms), 1)

        # Penalize candidates that match only a tiny part of their profile.
        phenotype_score = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0 else 0.0
        )

        disease_genes = gene_map.get(disease, set())
        matching_genes = genes & disease_genes

        gene_score = (
            len(matching_genes) / len(genes) if genes else np.nan
        )

        combined_score = (
            (1 - gene_weight) * phenotype_score + gene_weight * gene_score
            if genes else phenotype_score
        )

        results.append({
            "disease_id": disease,
            "phenotype_score": phenotype_score,
            "gene_score": gene_score,
            "combined_score": combined_score,
            "matching_phenotypes": len(overlap),
            "matching_genes": ";".join(sorted(matching_genes))
        })

    result = pd.DataFrame(results).sort_values(
        ["combined_score", "matching_phenotypes", "disease_id"],
        ascending=[False, False, True]
    ).head(top_k).reset_index(drop=True)

    result.insert(0, "rank", range(1, len(result) + 1))
    return result


In [ ]:
predict_diseases(
    patient_hpo_terms=["HP:0001250", "HP:0001263"],
    top_k=10
)

,rank,disease_id,phenotype_score,gene_score,combined_score,matching_phenotypes,matching_genes
0,1,ORPHA:2798,0.666667,NaN,0.666667,2,
1,2,OMIM:300852,0.571429,NaN,0.571429,2,
2,3,OMIM:611718,0.571429,NaN,0.571429,2,
3,4,OMIM:605375,0.505061,NaN,0.505061,1,
4,5,ORPHA:46348,0.505061,NaN,0.505061,1,
5,6,OMIM:616887,0.500000,NaN,0.500000,2,
6,7,ORPHA:85336,0.500000,NaN,0.500000,2,
7,8,OMIM:613657,0.444444,NaN,0.444444,2,
8,9,OMIM:619615,0.444444,NaN,0.444444,2,
9,10,DECIPHER:74,0.403232,NaN,0.403232,1,


In [ ]:

test_genes = ["SCN1A", "KCNQ2", "MECP2"]

gene_test = predict_diseases(
    patient_hpo_terms=["HP:0001250", "HP:0001263"],
    patient_genes=test_genes,
    top_k=10
)

print(gene_test.to_string(index=False))


 rank  disease_id  phenotype_score  gene_score  combined_score  matching_phenotypes matching_genes
    1  ORPHA:2798         0.666667         0.0        0.500000                    2               
    2 OMIM:300852         0.571429         0.0        0.428571                    2               
    3 OMIM:611718         0.571429         0.0        0.428571                    2               
    4 OMIM:605375         0.505061         0.0        0.378795                    1               
    5 ORPHA:46348         0.505061         0.0        0.378795                    1               
    6 OMIM:616887         0.500000         0.0        0.375000                    2               
    7 ORPHA:85336         0.500000         0.0        0.375000                    2               
    8 OMIM:613657         0.444444         0.0        0.333333                    2               
    9 OMIM:619615         0.444444         0.0        0.333333                    2               
   10 DECI

In [ ]:

def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,
                     gene_weight=0.25):
    observed = {
        str(t).strip()
        for t in patient_hpo_terms
        if str(t).strip() in ic
    }

    if not observed:
        raise ValueError("No recognized HPO terms. Use valid HPO IDs.")

    weights = {t: ic[t] for t in observed}
    denominator = sum(weights.values()) or 1.0

    genes = {
        str(g).strip().upper()
        for g in (patient_genes or [])
        if str(g).strip()
    }

    results = []

    for disease, disease_terms in phenotype_map.items():
        overlap = observed & disease_terms

        precision = sum(weights[t] for t in overlap) / denominator
        recall = len(overlap) / max(len(disease_terms), 1)

        phenotype_score = (
            2 * precision * recall / (precision + recall)
            if precision + recall > 0 else 0.0
        )

        disease_genes = gene_map.get(disease, set())
        matching_genes = genes & disease_genes
        gene_score = len(matching_genes) / len(genes) if genes else np.nan

        if genes and matching_genes:
            combined_score = (
                (1 - gene_weight) * phenotype_score
                + gene_weight * gene_score
            )
        else:
            combined_score = phenotype_score

        results.append({
            "disease_id": disease,
            "phenotype_score": phenotype_score,
            "gene_score": gene_score,
            "combined_score": combined_score,
            "matching_phenotypes": len(overlap),
            "matching_genes": ";".join(sorted(matching_genes))
        })

    result = pd.DataFrame(results).sort_values(
        ["combined_score", "matching_phenotypes", "disease_id"],
        ascending=[False, False, True]
    ).head(top_k).reset_index(drop=True)

    result.insert(0, "rank", range(1, len(result) + 1))
    return result


print("Updated prediction function successfully.")


Updated prediction function successfully.


In [ ]:

predict_diseases(
    patient_hpo_terms=["HP:0001250", "HP:0001263"],
    patient_genes=["SCN1A", "KCNQ2", "MECP2"],
    top_k=10
)


,rank,disease_id,phenotype_score,gene_score,combined_score,matching_phenotypes,matching_genes
0,1,ORPHA:2798,0.666667,0.0,0.666667,2,
1,2,OMIM:300852,0.571429,0.0,0.571429,2,
2,3,OMIM:611718,0.571429,0.0,0.571429,2,
3,4,OMIM:605375,0.505061,0.0,0.505061,1,
4,5,ORPHA:46348,0.505061,0.0,0.505061,1,
5,6,OMIM:616887,0.500000,0.0,0.500000,2,
6,7,ORPHA:85336,0.500000,0.0,0.500000,2,
7,8,OMIM:613657,0.444444,0.0,0.444444,2,
8,9,OMIM:619615,0.444444,0.0,0.444444,2,
9,10,DECIPHER:74,0.403232,0.0,0.403232,1,


In [ ]:

# Build a disease-ID-to-name mapping from the existing annotations.
disease_names = (
    ann.dropna(subset=["database_id", "disease_name"])
    .assign(
        database_id=lambda df: df["database_id"].str.strip(),
        disease_name=lambda df: df["disease_name"].astype(str).str.strip()
    )
    .groupby("database_id")["disease_name"]
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)

def explain_predictions(patient_hpo_terms, patient_genes=None, top_k=10):
    results = predict_diseases(
        patient_hpo_terms=patient_hpo_terms,
        patient_genes=patient_genes,
        top_k=top_k
    ).copy()

    results["disease_name"] = results["disease_id"].map(disease_names)
    results["disease_name"] = results["disease_name"].fillna(
        "Disease name unavailable in annotations"
    )

    # Display matching phenotype IDs for transparency.
    observed = set(patient_hpo_terms)
    results["matching_phenotype_ids"] = results["disease_id"].apply(
        lambda d: ";".join(sorted(observed & phenotype_map.get(d, set())))
    )

    columns = [
        "rank", "disease_id", "disease_name", "combined_score",
        "matching_phenotypes", "matching_phenotype_ids",
        "matching_genes"
    ]
    return results[columns]

print("Explanation function ready.")


Explanation function ready.


In [ ]:

explain_predictions(
    patient_hpo_terms=["HP:0001250", "HP:0001263"],
    top_k=10
)



,rank,disease_id,disease_name,combined_score,matching_phenotypes,matching_phenotype_ids,matching_genes
0,1,ORPHA:2798,Pachygyria-intellectual disability-epilepsy sy...,0.666667,2,HP:0001250;HP:0001263,
1,2,OMIM:300852,"Intellectual developmental disorder, X-linked 88",0.571429,2,HP:0001250;HP:0001263,
2,3,OMIM:611718,"Hypomagnesemia 4, renal",0.571429,2,HP:0001250;HP:0001263,
3,4,OMIM:605375,"Epilepsy, nocturnal frontal lobe, 3",0.505061,1,HP:0001250,
4,5,ORPHA:46348,Paroxysmal extreme pain disorder,0.505061,1,HP:0001250,
5,6,OMIM:616887,"Intellectual developmental disorder, autosomal...",0.500000,2,HP:0001250;HP:0001263,
6,7,ORPHA:85336,"X-linked neurodegenerative syndrome, Hamel type",0.500000,2,HP:0001250;HP:0001263,
7,8,OMIM:613657,D-2-hydroxyglutaric aciduria 2,0.444444,2,HP:0001250;HP:0001263,
8,9,OMIM:619615,"Deafness, autosomal recessive 119",0.444444,2,HP:0001250;HP:0001263,
9,10,DECIPHER:74,15q13.3 microdeletion syndrome,0.403232,1,HP:0001250,


In [ ]:

from pathlib import Path
import json

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
MODEL_DIR = Path("/content/drive/MyDrive/Rare Disease ai/model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save the model configuration and artifact locations.
config = {
    "model_name": "Rare Disease Phenotype Ranking Prototype",
    "version": "1.0",
    "phenotype_scoring": "information_content_weighted_f1",
    "gene_weight": 0.25,
    "phenotype_annotations": str(DATA / "clean_hpo_annotations.csv"),
    "gene_mapping": str(DATA / "normalized_disease_gene_mapping.csv"),
    "disease_split": str(DATA / "disease_level_split.csv"),
    "clinical_validation": "not_yet_performed",
    "scores_are_probabilities": False
}

config_path = MODEL_DIR / "prediction_config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Configuration saved:", config_path)
print("Prediction function is available in the current notebook session.")
print("Clinical validation is still required before real-world use.")


Configuration saved: /content/drive/MyDrive/Rare Disease ai/model/prediction_config.json
Prediction function is available in the current notebook session.
Clinical validation is still required before real-world use.


In [ ]:

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Current functions available:")
print("predict_diseases:", callable(globals().get("predict_diseases")))
print("explain_predictions:", callable(globals().get("explain_predictions")))

if not callable(globals().get("predict_diseases")):
    raise RuntimeError("Run the cell defining predict_diseases first.")

if not callable(globals().get("explain_predictions")):
    raise RuntimeError("Run the cell defining explain_predictions first.")

print("\nBoth functions are available in this session.")
print("Next we will export them into a reusable Python module.")


Current functions available:
predict_diseases: True
explain_predictions: True

Both functions are available in this session.
Next we will export them into a reusable Python module.


In [ ]:

import inspect
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

predictor = globals().get("predict_diseases")
explainer = globals().get("explain_predictions")

if not callable(predictor) or not callable(explainer):
    raise RuntimeError(
        "Run the cells defining both prediction functions first."
    )

# Export the function source code.
predictor_source = inspect.getsource(predictor)
explainer_source = inspect.getsource(explainer)

module_source = '''"""Rare Disease Prediction AI — research prototype.

This module contains ranking functions exported from the Colab notebook.
It is not a clinically validated diagnostic system.
"""

import pandas as pd
import numpy as np

''' + predictor_source + "\n\n" + explainer_source + "\n"

module_path = MODEL_DIR / "rare_disease_predictor.py"
module_path.write_text(module_source, encoding="utf-8")

print("Exported:", module_path)
print("Bytes written:", module_path.stat().st_size)
print("Note: the module also requires its data mappings and IC values to be loaded.")


Exported: /content/drive/MyDrive/Rare Disease ai/model/rare_disease_predictor.py
Bytes written: 2993
Note: the module also requires its data mappings and IC values to be loaded.


In [ ]:

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

loader_code = r'''
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

def load_prediction_data():
    ann = pd.read_csv(
        DATA / "clean_hpo_annotations.csv",
        low_memory=False,
        dtype={"database_id": "string", "hpo_id": "string"}
    ).dropna(subset=["database_id", "hpo_id"])

    ann["database_id"] = ann["database_id"].str.strip()
    ann["hpo_id"] = ann["hpo_id"].str.strip()

    phenotype_map = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()

    gene_df = pd.read_csv(
        DATA / "normalized_disease_gene_mapping.csv",
        dtype=str
    ).fillna("")

    gene_map = {
        row["disease_id"]: {
            g.strip().upper()
            for g in row["gene_symbols"].split(";")
            if g.strip()
        }
        for _, row in gene_df.iterrows()
    }

    split = pd.read_csv(
        DATA / "disease_level_split.csv",
        dtype={"disease_id": "string"}
    )
    train_ids = set(
        split.loc[split["split"].str.lower() == "train", "disease_id"]
    )

    counts = defaultdict(int)
    for disease in train_ids:
        for term in phenotype_map.get(disease, set()):
            counts[term] += 1

    n_train = max(len(train_ids), 1)
    ic = {
        term: -np.log(count / n_train)
        for term, count in counts.items() if count > 0
    }

    disease_names = (
        ann.dropna(subset=["disease_name"])
        .groupby("database_id")["disease_name"]
        .agg(lambda x: x.value_counts().index[0])
        .to_dict()
    )

    return {
        "annotations": ann,
        "phenotype_map": phenotype_map,
        "gene_map": gene_map,
        "ic": ic,
        "disease_names": disease_names
    }
'''

loader_path = MODEL_DIR / "rare_disease_data_loader.py"
loader_path.write_text(loader_code, encoding="utf-8")

print("Saved reusable data loader:", loader_path)
print("It loads annotations, gene mappings, training-only IC weights, and disease names.")


Saved reusable data loader: /content/drive/MyDrive/Rare Disease ai/model/rare_disease_data_loader.py
It loads annotations, gene mappings, training-only IC weights, and disease names.


In [ ]:

import sys
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"

if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

from rare_disease_data_loader import load_prediction_data

model_data = load_prediction_data()

ann = model_data["annotations"]
phenotype_map = model_data["phenotype_map"]
gene_map = model_data["gene_map"]
ic = model_data["ic"]
disease_names = model_data["disease_names"]

print("DATA LOADER TEST PASSED")
print("Disease phenotype profiles:", len(phenotype_map))
print("Diseases with gene mappings:", len(gene_map))
print("Phenotype information weights:", len(ic))
print("Disease names:", len(disease_names))


DATA LOADER TEST PASSED
Disease phenotype profiles: 12879
Diseases with gene mappings: 11168
Phenotype information weights: 9982
Disease names: 12879


In [ ]:

import sys
from pathlib import Path
import importlib

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"

if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

from rare_disease_data_loader import load_prediction_data

# Load all required data into a single dictionary.
model_data = load_prediction_data()

# Check that every dependency required by the predictor is present.
required = [
    "annotations",
    "phenotype_map",
    "gene_map",
    "ic",
    "disease_names",
]

missing = [key for key in required if key not in model_data]
if missing:
    raise RuntimeError(f"Missing predictor dependencies: {missing}")

print("Prediction engine dependencies verified.")
print("Ready to build the standalone predictor module.")


Prediction engine dependencies verified.
Ready to build the standalone predictor module.


In [ ]:

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

module_code = r'''
from rare_disease_data_loader import load_prediction_data

_DATA = load_prediction_data()
PHENOTYPE_MAP = _DATA["phenotype_map"]
GENE_MAP = _DATA["gene_map"]
IC = _DATA["ic"]
DISEASE_NAMES = _DATA["disease_names"]

def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,
                     gene_weight=0.25):
    observed = {
        str(t).strip() for t in patient_hpo_terms
        if str(t).strip() in IC
    }
    if not observed:
        raise ValueError("No recognized HPO terms were supplied.")

    weights = {t: IC[t] for t in observed}
    denominator = sum(weights.values()) or 1.0
    genes = {
        str(g).strip().upper() for g in (patient_genes or [])
        if str(g).strip()
    }

    results = []
    for disease, disease_terms in PHENOTYPE_MAP.items():
        overlap = observed & disease_terms
        precision = sum(weights[t] for t in overlap) / denominator
        recall = len(overlap) / max(len(disease_terms), 1)
        phenotype_score = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        matching_genes = genes & GENE_MAP.get(disease, set())
        gene_score = len(matching_genes) / len(genes) if genes else None

        # Do not penalize candidates when no patient gene matches.
        if genes and matching_genes:
            combined_score = (
                (1 - gene_weight) * phenotype_score
                + gene_weight * gene_score
            )
        else:
            combined_score = phenotype_score

        results.append({
            "disease_id": disease,
            "disease_name": DISEASE_NAMES.get(
                disease, "Disease name unavailable"
            ),
            "phenotype_score": phenotype_score,
            "gene_score": gene_score,
            "combined_score": combined_score,
            "matching_phenotypes": len(overlap),
            "matching_phenotype_ids": ";".join(sorted(overlap)),
            "matching_genes": ";".join(sorted(matching_genes))
        })

    import pandas as pd
    result = pd.DataFrame(results).sort_values(
        ["combined_score", "matching_phenotypes", "disease_id"],
        ascending=[False, False, True]
    ).head(top_k).reset_index(drop=True)

    result.insert(0, "rank", range(1, len(result) + 1))
    return result
'''

module_path = MODEL_DIR / "rare_disease_ai.py"
module_path.write_text(module_code, encoding="utf-8")

print("Standalone predictor saved:", module_path)
print("Next: import it and test it.")


Standalone predictor saved: /content/drive/MyDrive/Rare Disease ai/model/rare_disease_ai.py
Next: import it and test it.


In [ ]:

import sys
import importlib

if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

import rare_disease_ai
importlib.reload(rare_disease_ai)

test_result = rare_disease_ai.predict_diseases(
    patient_hpo_terms=["HP:0001250", "HP:0001263"],
    top_k=10
)

print(test_result.to_string(index=False))


 rank  disease_id                                                disease_name  phenotype_score gene_score  combined_score  matching_phenotypes matching_phenotype_ids matching_genes
    1  ORPHA:2798        Pachygyria-intellectual disability-epilepsy syndrome         0.666667       None        0.666667                    2  HP:0001250;HP:0001263               
    2 OMIM:300852            Intellectual developmental disorder, X-linked 88         0.571429       None        0.571429                    2  HP:0001250;HP:0001263               
    3 OMIM:611718                                     Hypomagnesemia 4, renal         0.571429       None        0.571429                    2  HP:0001250;HP:0001263               
    4 OMIM:605375                         Epilepsy, nocturnal frontal lobe, 3         0.505061       None        0.505061                    1             HP:0001250               
    5 ORPHA:46348                            Paroxysmal extreme pain disorder         0.505061 

In [ ]:

from pathlib import Path
import sys

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"

if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

import rare_disease_ai

def run_rare_disease_ai(hpo_terms, candidate_genes=None, top_k=10):
    """Return ranked disease candidates for supplied HPO terms."""
    results = rare_disease_ai.predict_diseases(
        patient_hpo_terms=hpo_terms,
        patient_genes=candidate_genes,
        top_k=top_k
    )

    display_columns = [
        "rank",
        "disease_id",
        "disease_name",
        "combined_score",
        "matching_phenotypes",
        "matching_phenotype_ids",
        "matching_genes"
    ]

    return results[display_columns]

print("Prediction interface ready.")
print("Call run_rare_disease_ai([...]) with valid HPO identifiers.")


Prediction interface ready.
Call run_rare_disease_ai([...]) with valid HPO identifiers.


In [ ]:

run_rare_disease_ai(
    hpo_terms=["HP:0001250", "HP:0001263"],
    top_k=10
)


,rank,disease_id,disease_name,combined_score,matching_phenotypes,matching_phenotype_ids,matching_genes
0,1,ORPHA:2798,Pachygyria-intellectual disability-epilepsy sy...,0.666667,2,HP:0001250;HP:0001263,
1,2,OMIM:300852,"Intellectual developmental disorder, X-linked 88",0.571429,2,HP:0001250;HP:0001263,
2,3,OMIM:611718,"Hypomagnesemia 4, renal",0.571429,2,HP:0001250;HP:0001263,
3,4,OMIM:605375,"Epilepsy, nocturnal frontal lobe, 3",0.505061,1,HP:0001250,
4,5,ORPHA:46348,Paroxysmal extreme pain disorder,0.505061,1,HP:0001250,
5,6,OMIM:616887,"Intellectual developmental disorder, autosomal...",0.500000,2,HP:0001250;HP:0001263,
6,7,ORPHA:85336,"X-linked neurodegenerative syndrome, Hamel type",0.500000,2,HP:0001250;HP:0001263,
7,8,OMIM:613657,D-2-hydroxyglutaric aciduria 2,0.444444,2,HP:0001250;HP:0001263,
8,9,OMIM:619615,"Deafness, autosomal recessive 119",0.444444,2,HP:0001250;HP:0001263,
9,10,DECIPHER:74,15q13.3 microdeletion syndrome,0.403232,1,HP:0001250,


In [ ]:

import pandas as pd

# Build a searchable phenotype-name lookup from the existing annotations.
phenotype_lookup = (
    ann.dropna(subset=["hpo_id", "disease_name"])
    [["hpo_id", "disease_name"]]
    .drop_duplicates()
    .rename(columns={"disease_name": "associated_disease_name"})
)

def search_phenotypes(query, limit=20):
    """Search annotation text for a phenotype-related disease description."""
    query = str(query).strip().lower()
    if not query:
        raise ValueError("Enter a search term.")

    matches = phenotype_lookup[
        phenotype_lookup["associated_disease_name"]
        .str.lower()
        .str.contains(query, regex=False, na=False)
    ].head(limit)

    return matches.reset_index(drop=True)

print("Phenotype search helper ready.")


Phenotype search helper ready.


In [ ]:

from pathlib import Path
import pandas as pd

OBO = Path("/content/drive/MyDrive/Rare Disease ai/hp.obo")

def load_hpo_terms(obo_path):
    terms = []
    current = None

    def save_term(term):
        if (
            term
            and term.get("id", "").startswith("HP:")
            and term.get("name")
            and not term.get("obsolete", False)
        ):
            terms.append({
                "id": term["id"],
                "name": term["name"]
            })

    with open(obo_path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line == "[Term]":
                save_term(current)
                current = {}
            elif line.startswith("[") and line.endswith("]"):
                save_term(current)
                current = None
            elif current is not None:
                if line.startswith("id: "):
                    current["id"] = line[4:].strip()
                elif line.startswith("name: "):
                    current["name"] = line[6:].strip()
                elif line == "is_obsolete: true":
                    current["obsolete"] = True

    save_term(current)

    return pd.DataFrame(terms).drop_duplicates("id").reset_index(drop=True)

hpo_terms = load_hpo_terms(OBO)

def search_hpo(query, limit=20):
    query = str(query).strip().lower()
    if not query:
        raise ValueError("Enter a phenotype name.")

    return (
        hpo_terms[
            hpo_terms["name"].str.lower().str.contains(query, regex=False)
        ]
        .head(limit)
        .reset_index(drop=True)
    )

print("Official HPO terms loaded:", len(hpo_terms))
print("\nSeizure search test:")
print(search_hpo("seizure").to_string(index=False))


Official HPO terms loaded: 19894

Seizure search test:
        id                                                          name
HP:0001250                                                       Seizure
HP:0001327                              Photosensitive myoclonic seizure
HP:0002069                                Bilateral tonic-clonic seizure
HP:0002121                       Generalized non-motor (absence) seizure
HP:0002123                                 Generalized myoclonic seizure
HP:0002173                                         Hypoglycemic seizures
HP:0002197                                     Generalized-onset seizure
HP:0002199                                         Hypocalcemic seizures
HP:0002266                                          Focal clonic seizure
HP:0002349                                           Focal aware seizure
HP:0002373 Febrile seizure (within the age range of 3 months to 6 years)
HP:0002384                              Focal impaired awareness seiz

In [ ]:

import ipywidgets as widgets
from IPython.display import display, clear_output

# Search and selection controls
search_box = widgets.Text(
    description="Find:",
    placeholder="e.g. seizure, hypotonia",
    layout=widgets.Layout(width="500px")
)

search_button = widgets.Button(
    description="Search HPO",
    button_style="primary"
)

results_dropdown = widgets.SelectMultiple(
    options=[],
    description="Matches:",
    rows=8,
    layout=widgets.Layout(width="700px")
)

selected_terms = widgets.SelectMultiple(
    options=[],
    description="Selected:",
    rows=6,
    layout=widgets.Layout(width="700px")
)

add_button = widgets.Button(description="Add selected")
remove_button = widgets.Button(description="Remove selected")
predict_button = widgets.Button(
    description="Rank diseases",
    button_style="success"
)

output = widgets.Output()
selected_map = {}

def do_search(_):
    matches = search_hpo(search_box.value, limit=100)
    results_dropdown.options = [
        (f"{row['name']} ({row['id']})", row["id"])
        for _, row in matches.iterrows()
    ]

def add_terms(_):
    for term in results_dropdown.value:
        selected_map[term] = hpo_terms.loc[
            hpo_terms["id"] == term, "name"
        ].iloc[0]
    selected_terms.options = [
        (f"{name} ({term})", term)
        for term, name in selected_map.items()
    ]

def remove_terms(_):
    for term in selected_terms.value:
        selected_map.pop(term, None)
    selected_terms.options = [
        (f"{name} ({term})", term)
        for term, name in selected_map.items()
    ]

def run_prediction(_):
    with output:
        clear_output()
        if not selected_map:
            print("Search and select at least one phenotype first.")
            return

        display(
            run_rare_disease_ai(
                hpo_terms=list(selected_map.keys()),
                top_k=10
            )
        )

search_button.on_click(do_search)
add_button.on_click(add_terms)
remove_button.on_click(remove_terms)
predict_button.on_click(run_prediction)

display(
    widgets.VBox([
        widgets.HTML("<h3>Rare Disease Prediction AI — Research Prototype</h3>"),
        search_box,
        search_button,
        results_dropdown,
        add_button,
        selected_terms,
        remove_button,
        predict_button,
        output
    ])
)


In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    low_memory=False,
    dtype={"database_id": "string", "hpo_id": "string"}
).dropna(subset=["database_id", "hpo_id"])

split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype={"disease_id": "string"}
)
split["split"] = split["split"].str.lower().str.strip()

profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()

train_ids = set(split.loc[split["split"] == "train", "disease_id"])
test_ids = set(split.loc[split["split"] == "test", "disease_id"])

# Information content is estimated from training diseases only.
freq = {}
for disease in train_ids:
    for term in profiles.get(disease, set()):
        freq[term] = freq.get(term, 0) + 1

n_train = max(len(train_ids), 1)
ic = {term: -np.log(count / n_train) for term, count in freq.items()}

rng = np.random.default_rng(42)
rows = []

for disease in sorted(test_ids):
    full_profile = set(profiles.get(disease, set()))
    known = sorted(full_profile & set(ic))

    if len(known) < 2:
        continue

    # Hold out 20% of the disease's known phenotypes.
    n_hide = min(max(1, round(0.2 * len(known))), len(known) - 1)
    hidden_idx = set(rng.choice(len(known), size=n_hide, replace=False))

    hidden = {term for i, term in enumerate(known) if i in hidden_idx}
    observed = {term for i, term in enumerate(known) if i not in hidden_idx}

    # Remove hidden terms from the target candidate profile.
    # Other diseases retain their known profiles.
    candidate_profiles = {
        candidate: set(terms)
        for candidate, terms in profiles.items()
        if terms
    }
    candidate_profiles[disease] = full_profile - hidden

    weights = {term: ic[term] for term in observed}
    denominator = sum(weights.values()) or 1.0

    scores = []
    for candidate, candidate_terms in candidate_profiles.items():
        overlap = observed & candidate_terms
        score = sum(weights[t] for t in overlap) / denominator
        scores.append((candidate, score))

    scores.sort(key=lambda x: (-x[1], x[0]))
    ranking = [candidate for candidate, _ in scores]

    # Rank the correct held-out disease against all candidates.
    rank = ranking.index(disease) + 1

    rows.append({
        "disease_id": disease,
        "observed_phenotypes": len(observed),
        "hidden_phenotypes": len(hidden),
        "correct_disease_rank": rank,
        "recall_at_1": int(rank <= 1),
        "recall_at_5": int(rank <= 5),
        "recall_at_10": int(rank <= 10),
        "reciprocal_rank": 1.0 / rank
    })

result = pd.DataFrame(rows)
output = DATA / "leakage_reduced_retrieval_evaluation.csv"
result.to_csv(output, index=False)

print("Profiles evaluated:", len(result))
if len(result):
    print("Recall@1:", round(result["recall_at_1"].mean(), 4))
    print("Recall@5:", round(result["recall_at_5"].mean(), 4))
    print("Recall@10:", round(result["recall_at_10"].mean(), 4))
    print("MRR:", round(result["reciprocal_rank"].mean(), 4))
else:
    print("No eligible profiles.")
print("Saved:", output)
print("Research retrieval evaluation only; not clinical diagnostic accuracy.")


Profiles evaluated: 1829
Recall@1: 0.9295
Recall@5: 0.9716
Recall@10: 0.9787
MRR: 0.9475
Saved: /content/drive/MyDrive/Rare Disease ai/processed/leakage_reduced_retrieval_evaluation.csv
Research retrieval evaluation only; not clinical diagnostic accuracy.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

print("Project folder exists:", PROJECT.exists())

if PROJECT.exists():
    print("\nProject folder contents:")
    for p in sorted(PROJECT.iterdir()):
        print(("FOLDER " if p.is_dir() else "FILE   ") + p.name)

    processed = PROJECT / "processed"
    if processed.exists():
        print("\nProcessed folder contents:")
        for p in sorted(processed.iterdir()):
            print(("FOLDER " if p.is_dir() else "FILE   ") + p.name)
else:
    print("Project folder not found. Check the folder name in Google Drive.")

Mounted at /content/drive
Project folder exists: True

Project folder contents:
FOLDER all_phenopackets
FILE   genes_to_disease.txt
FILE   genes_to_phenotype.txt
FILE   hp.obo
FOLDER model
FILE   phenotype.hpoa
FOLDER processed

Processed folder contents:
FILE   baseline_coverage_comparison_audit.csv
FILE   baseline_model_comparison.csv
FILE   candidate_coverage_audit.csv
FILE   clean_hpo_annotations.csv
FILE   direct_vs_hierarchy_scoring_comparison.csv
FILE   disease_level_split.csv
FILE   disease_ranking_retrieval_metrics.csv
FILE   evaluation_eligibility_audit.csv
FILE   evaluation_protocol_checks.csv
FILE   final_artifact_inventory.csv
FILE   final_consolidated_results.csv
FILE   final_project_handover_inventory.csv
FILE   gene_informed_phenotype_recovery_metrics.csv
FILE   gene_informed_phenotype_recovery_top10.csv
FILE   gnn_checkpoint_compatibility.csv
FILE   gnn_evaluation_results.csv
FILE   gnn_phenotype_recovery_metrics.csv
FILE   gnn_training_annotations.csv
FILE   gnn_zero_

In [ ]:
from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

# Verify saved artifacts
required = {
    "annotations": DATA / "clean_hpo_annotations.csv",
    "training_graph": DATA / "rare_disease_training_graph.graphml",
    "training_checkpoint": DATA / "rare_disease_gnn_training_only.pt",
    "disease_split": DATA / "disease_level_split.csv",
    "results": DATA / "final_consolidated_results.csv",
}

print("Artifact verification:")
for name, path in required.items():
    print(f"{name}: {'OK' if path.exists() else 'MISSING'}")

# Reload lightweight evaluation data
annotations = pd.read_csv(required["annotations"])
disease_split = pd.read_csv(required["disease_split"])
previous_results = pd.read_csv(required["results"])

print("\nAnnotations:", annotations.shape)
print("Disease split:", disease_split.shape)
print("Previous results:", previous_results.shape)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Artifact verification:
annotations: OK
training_graph: OK
training_checkpoint: OK
disease_split: OK
results: OK


/tmp/ipykernel_302/827847694.py:22: DtypeWarning: Columns (2,6,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  annotations = pd.read_csv(required["annotations"])



Annotations: (286047, 12)
Disease split: (9176, 2)
Previous results: (11, 6)
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
from pathlib import Path
import torch

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
checkpoint_path = DATA / "rare_disease_gnn_training_only.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys()))

    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"\n{key}: dictionary with {len(value)} entries")
            print("Sample keys:", list(value.keys())[:8])
        elif isinstance(value, torch.Tensor):
            print(f"{key}: tensor shape {tuple(value.shape)}")
        elif isinstance(value, (str, int, float, bool, list, tuple)):
            print(f"{key}: {str(value)[:200]}")
        else:
            print(f"{key}: {type(value).__name__}")
else:
    print("Checkpoint contents require architecture inspection.")



Checkpoint type: dict
Checkpoint keys: ['model_state_dict', 'num_nodes', 'hidden_dim', 'disease_ids', 'phenotype_ids', 'disease_to_idx', 'phenotype_to_idx', 'training_disease_count', 'training_association_count', 'validation_loss', 'seed']

model_state_dict: dictionary with 5 entries
Sample keys: ['embedding.weight', 'conv1.bias', 'conv1.lin.weight', 'conv2.bias', 'conv2.lin.weight']
num_nodes: 17322
hidden_dim: 64
disease_ids: ['OMIM:100300', 'OMIM:100800', 'OMIM:101000', 'OMIM:101200', 'OMIM:101400', 'OMIM:101600', 'OMIM:101800', 'OMIM:102200', 'OMIM:102370', 'OMIM:102500', 'OMIM:102700', 'OMIM:103050', 'OMIM:103285', 'OMI
phenotype_ids: ['HP:0000002', 'HP:0000003', 'HP:0000006', 'HP:0000007', 'HP:0000008', 'HP:0000009', 'HP:0000010', 'HP:0000011', 'HP:0000012', 'HP:0000013', 'HP:0000014', 'HP:0000015', 'HP:0000016', 'HP:0000017', 'HP

disease_to_idx: dictionary with 7340 entries
Sample keys: ['OMIM:100300', 'OMIM:100800', 'OMIM:101000', 'OMIM:101200', 'OMIM:101400', 'OMIM:101600', '

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA version:", torch.version.cuda)

%pip install torch-geometric

import torch
import torch.nn as nn
from torch_geometric.nn import GCNConv
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
checkpoint_path = DATA / "rare_disease_gnn_training_only.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

class RareDiseaseGNN(nn.Module):
    def __init__(self, num_nodes, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(num_nodes, hidden_dim)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

    def forward(self, edge_index):
        x = self.embedding.weight
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

model = RareDiseaseGNN(
    num_nodes=checkpoint["num_nodes"],
    hidden_dim=checkpoint["hidden_dim"]
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("GNN restored successfully.")
print("Device:", device)
print("Training diseases:", checkpoint["training_disease_count"])
print("Training associations:", checkpoint["training_association_count"])
print("Validation loss:", checkpoint["validation_loss"])
print("Disease index entries:", len(checkpoint["disease_to_idx"]))
print("Phenotype index entries:", len(checkpoint["phenotype_to_idx"]))

PyTorch version: 2.11.0+cu128
CUDA version: 12.8
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 63.4 MB/s eta 0:00:00
GNN restored successfully.
Device: cuda
Training diseases: 7340
Training associations: 179653
Validation loss: 0.57393878698349
Disease index entries: 7340
Phenotype index entries: 9982


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = [
    "three_model_comparison_summary.csv",
    "baseline_model_comparison.csv",
    "hierarchy_aware_retrieval_metrics.csv",
    "ic_weighted_heldout_ranking_results.csv",
    "leakage_reduced_retrieval_evaluation.csv",
]

for name in files:
    path = DATA / name
    print(f"\n{'=' * 65}\n{name}")

    if path.exists():
        df = pd.read_csv(path, low_memory=False)
        print("Shape:", df.shape)
        print(df.head(8).to_string(index=False))
    else:
        print("File missing")



three_model_comparison_summary.csv
Shape: (3, 4)
                   Model  Profiles  Mean Recall@10  Mean MRR
          Phenotype-only      1827        0.213532  0.520367
           Gene-informed      1827        0.133764  0.379349
GNN phenotype embeddings      1827        0.086528  0.278151

baseline_model_comparison.csv
Shape: (2, 3)
                  Model  Mean Recall@10  Mean Reciprocal Rank
Phenotype co-occurrence        0.212949              0.518946
  Gene-informed scoring        0.198043              0.474340

hierarchy_aware_retrieval_metrics.csv
Shape: (1829, 6)
 disease_id  rank  recall_at_1  recall_at_5  recall_at_10  reciprocal_rank
OMIM:100100    49            0            0             0         0.020408
OMIM:101900    65            0            0             0         0.015385
OMIM:102530    75            0            0             0         0.013333
OMIM:102900    79            0            0             0         0.012658
OMIM:103500    86            0            0 

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

for name in [
    "evaluation_protocol_checks.csv",
    "self_match_sanity_check.csv",
    "phenotype_disease_ranking_evaluation.csv",
    "disease_ranking_retrieval_metrics.csv",
]:
    path = DATA / name
    print(f"\n--- {name} ---")

    if not path.exists():
        print("Not found")
        continue

    df = pd.read_csv(path, low_memory=False)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(5).to_string(index=False))

print("\n--- Disease split counts ---")
split = pd.read_csv(DATA / "disease_level_split.csv", dtype=str)
print("Columns:", split.columns.tolist())
print(split.nunique().to_string())
print(split.head(5).to_string(index=False))


--- evaluation_protocol_checks.csv ---
Shape: (6, 2)
Columns: ['Check', 'Value']
                              Check  Value
               Input phenotype rows  21647
       Hidden target phenotype rows  22562
          Input/target pair overlap      0
             Training diseases used   7340
Held-out diseases in training index      0

--- self_match_sanity_check.csv ---
Shape: (5, 3)
Columns: ['Metric', 'Value', 'Evaluation']
                    Metric       Value              Evaluation
         Queries evaluated 1832.000000 Self-match sanity check
  Mean rank of own profile    1.111354 Self-match sanity check
Median rank of own profile    1.000000 Self-match sanity check
     Top-1 self-match rate    0.979258 Self-match sanity check
 Recall@10 self-match rate    0.996179 Self-match sanity check

--- phenotype_disease_ranking_evaluation.csv ---
Shape: (1832, 8)
Columns: ['disease_id', 'input_phenotype_count', 'hidden_phenotype', 'correct_disease_rank', 'recall_at_1', 'recall_at_5'

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

# Check split integrity
split = pd.read_csv(DATA / "disease_level_split.csv", dtype=str)
split["split"] = split["split"].str.lower().str.strip()

print("DISEASE SPLIT")
print(split["split"].value_counts(dropna=False))
print("Duplicate disease IDs:", split["disease_id"].duplicated().sum())

train_ids = set(split.loc[split["split"] == "train", "disease_id"])
test_ids = set(split.loc[split["split"] == "test", "disease_id"])
print("Train/test ID overlap:", len(train_ids & test_ids))

# Compare held-out IDs with the candidate coverage audit
coverage_path = DATA / "candidate_coverage_audit.csv"
if coverage_path.exists():
    coverage = pd.read_csv(coverage_path, low_memory=False)
    print("\nCANDIDATE COVERAGE AUDIT")
    print("Columns:", coverage.columns.tolist())
    print(coverage.head(8).to_string(index=False))

# Check saved ranking metrics
ranking_path = DATA / "disease_ranking_retrieval_metrics.csv"
ranking = pd.read_csv(ranking_path, low_memory=False)

print("\nRANKING METRICS")
print("Rows:", len(ranking))
print("Rank summary:")
print(ranking["rank"].describe())
print("Ranks <= 1:", int((ranking["rank"] <= 1).sum()))
print("Ranks <= 10:", int((ranking["rank"] <= 10).sum()))

if "disease_id" in ranking.columns:
    ranked_ids = set(ranking["disease_id"].dropna().astype(str))
    print("Ranking IDs overlapping test split:", len(ranked_ids & test_ids))
    print("Ranking IDs overlapping train split:", len(ranked_ids & train_ids))

DISEASE SPLIT
split
train    7340
test     1836
Name: count, dtype: int64
Duplicate disease IDs: 0
Train/test ID overlap: 0

CANDIDATE COVERAGE AUDIT
Columns: ['disease_id', 'input_count', 'covered_input_count', 'candidate_count']
 disease_id  input_count  covered_input_count  candidate_count
OMIM:100100            8                    8             7375
OMIM:101900            4                    4             4511
OMIM:102530            2                    2             6539
OMIM:102900            1                    0                0
OMIM:103500            4                    4             1021
OMIM:103780            1                    1              157
OMIM:104290            5                    5             6335
OMIM:105150            3                    3             5797

RANKING METRICS
Rows: 1829
Rank summary:
count    1829.000000
mean       11.486058
std       122.665070
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max      2534

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = [
    "gnn_training_annotations.csv",
    "heldout_input_phenotypes.csv",
    "heldout_target_phenotypes.csv",
    "phenotype_baseline_per_profile.csv",
    "evaluation_eligibility_audit.csv",
    "evaluation_protocol_checks.csv",
]

for name in files:
    path = DATA / name
    print(f"\n{'=' * 60}\n{name}")

    if not path.exists():
        print("MISSING")
        continue

    df = pd.read_csv(path, low_memory=False)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string(index=False))


gnn_training_annotations.csv
Shape: (179653, 12)
Columns: ['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']
database_id                                  disease_name qualifier     hpo_id     reference evidence onset frequency sex modifier aspect               biocuration
OMIM:619340 Developmental and epileptic encephalopathy 96       NaN HP:0011097 PMID:31675180      PCS   NaN       1/2 NaN      NaN      P HPO:probinson[2021-06-21]
OMIM:619340 Developmental and epileptic encephalopathy 96       NaN HP:0002187 PMID:31675180      PCS   NaN       1/1 NaN      NaN      P HPO:probinson[2021-06-21]
OMIM:619340 Developmental and epileptic encephalopathy 96       NaN HP:0001518 PMID:31675180      PCS   NaN       1/2 NaN      NaN      P HPO:probinson[2021-06-21]

heldout_input_phenotypes.csv
Shape: (21647, 2)
Columns: ['disease_id', 'hpo_id']
 disease_id     hpo_id
OMIM:100100 HP:0030211
OMIM:100100

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

for folder in [PROJECT, PROJECT / "model"]:
    print(f"\n--- Python files in {folder} ---")
    if folder.exists():
        for path in sorted(folder.glob("*.py")):
            print(path.name)


--- Python files in /content/drive/MyDrive/Rare Disease ai ---

--- Python files in /content/drive/MyDrive/Rare Disease ai/model ---
rare_disease_ai.py
rare_disease_data_loader.py
rare_disease_predictor.py


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL_DIR = PROJECT / "model"

keywords = (
    "predict", "rank", "score", "similarity",
    "candidate", "phenotype", "recall", "reciprocal"
)

for filename in [
    "rare_disease_ai.py",
    "rare_disease_predictor.py",
    "rare_disease_data_loader.py",
]:
    path = MODEL_DIR / filename
    print(f"\n{'=' * 70}\nFILE: {filename}")

    if not path.exists():
        print("File not found")
        continue

    lines = path.read_text(errors="replace").splitlines()
    matches = [
        i for i, line in enumerate(lines)
        if any(word in line.lower() for word in keywords)
    ]

    shown = set()
    for i in matches:
        start, end = max(0, i - 3), min(len(lines), i + 5)
        if any(j in shown for j in range(start, end)):
            continue
        print(f"\n--- lines {start + 1}-{end} ---")
        for j in range(start, end):
            print(f"{j + 1}: {lines[j]}")
            shown.add(j)


FILE: rare_disease_ai.py

--- lines 1-6 ---
1: 
2: from rare_disease_data_loader import load_prediction_data
3: 
4: _DATA = load_prediction_data()
5: PHENOTYPE_MAP = _DATA["phenotype_map"]
6: GENE_MAP = _DATA["gene_map"]

--- lines 7-14 ---
7: IC = _DATA["ic"]
8: DISEASE_NAMES = _DATA["disease_names"]
9: 
10: def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,
11:                      gene_weight=0.25):
12:     observed = {
13:         str(t).strip() for t in patient_hpo_terms
14:         if str(t).strip() in IC

--- lines 24-31 ---
24:     }
25: 
26:     results = []
27:     for disease, disease_terms in PHENOTYPE_MAP.items():
28:         overlap = observed & disease_terms
29:         precision = sum(weights[t] for t in overlap) / denominator
30:         recall = len(overlap) / max(len(disease_terms), 1)
31:         phenotype_score = (

--- lines 34-41 ---
34:         )
35: 
36:         matching_genes = genes & GENE_MAP.get(disease, set())
37:         gene_score = l

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

annotations = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    dtype={"database_id": str, "hpo_id": str},
    low_memory=False
)
split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype=str
)

train_ids = set(split.loc[split["split"] == "train", "disease_id"].str.strip())
test_ids = set(split.loc[split["split"] == "test", "disease_id"].str.strip())
all_candidate_ids = set(annotations["database_id"].dropna().str.strip())

print("Training diseases:", len(train_ids))
print("Held-out diseases:", len(test_ids))
print("All diseases in phenotype map:", len(all_candidate_ids))
print("Held-out diseases present in all-disease map:",
      len(test_ids & all_candidate_ids))
print("Training diseases present in all-disease map:",
      len(train_ids & all_candidate_ids))

train_annotations = annotations[
    annotations["database_id"].str.strip().isin(train_ids)
]
train_phenotype_map = (
    train_annotations.groupby("database_id")["hpo_id"]
    .apply(lambda s: set(s.dropna().str.strip()))
    .to_dict()
)

print("Training-only phenotype profiles:", len(train_phenotype_map))
print("Held-out IDs in training-only map:",
      len(set(train_phenotype_map) & test_ids))

Training diseases: 7340
Held-out diseases: 1836
All diseases in phenotype map: 12879
Held-out diseases present in all-disease map: 1836
Training diseases present in all-disease map: 7340
Training-only phenotype profiles: 7340
Held-out IDs in training-only map: 0


In [ ]:
  from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    dtype={"database_id": str, "hpo_id": str},
    low_memory=False
)
split = pd.read_csv(DATA / "disease_level_split.csv", dtype=str)
inputs = pd.read_csv(DATA / "heldout_input_phenotypes.csv", dtype=str)
targets = pd.read_csv(DATA / "heldout_target_phenotypes.csv", dtype=str)

train_ids = set(split.loc[split["split"] == "train", "disease_id"].str.strip())
test_ids = set(split.loc[split["split"] == "test", "disease_id"].str.strip())

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()
ann = ann[ann["database_id"].isin(train_ids)].dropna(subset=["hpo_id"])

profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()
input_map = inputs.groupby("disease_id")["hpo_id"].apply(set).to_dict()
target_map = targets.groupby("disease_id")["hpo_id"].apply(set).to_dict()

# Training-only phenotype information content
term_counts = defaultdict(int)
for terms in profiles.values():
    for term in terms:
        term_counts[term] += 1

n_train = max(len(profiles), 1)
ic = {term: -np.log(count / n_train) for term, count in term_counts.items()}

rows = []
for disease_id in sorted(test_ids & set(input_map) & set(target_map)):
    observed = input_map[disease_id]
    hidden = target_map[disease_id]
    observed_known = observed & set(ic)

    if not hidden or not observed_known:
        continue

    weights_total = sum(ic[t] for t in observed_known)
    scored = []

    for candidate, terms in profiles.items():
        overlap = observed_known & terms
        weighted_overlap = sum(ic[t] for t in overlap)
        precision = weighted_overlap / weights_total if weights_total else 0.0
        recall = len(overlap) / max(len(terms), 1)
        score = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )
        scored.append((candidate, score))

    scored.sort(key=lambda x: (-x[1], x[0]))
    # Score how well the top training-disease candidate predicts hidden phenotypes.
    top_candidate, _ = scored[0]
    recovered = len(hidden & profiles[top_candidate])
    rows.append({
        "heldout_disease_id": disease_id,
        "top_training_candidate": top_candidate,
        "hidden_phenotype_count": len(hidden),
        "recovered_hidden_phenotypes": recovered,
        "hidden_phenotype_recall": recovered / len(hidden),
        "candidate_count": len(scored),
    })

results = pd.DataFrame(rows)
out_path = DATA / "training_only_phenotype_recovery_audit.csv"
results.to_csv(out_path, index=False)

print("Evaluated profiles:", len(results))
if len(results):
    print("Mean hidden-phenotype recall:",
          round(results["hidden_phenotype_recall"].mean(), 4))
    print("Median hidden-phenotype recall:",
          round(results["hidden_phenotype_recall"].median(), 4))
    print("Mean candidate count:", round(results["candidate_count"].mean(), 1))
print("Saved:", out_path)

Evaluated profiles: 1828
Mean hidden-phenotype recall: 0.2055
Median hidden-phenotype recall: 0.1429
Mean candidate count: 7340.0
Saved: /content/drive/MyDrive/Rare Disease ai/processed/training_only_phenotype_recovery_audit.csv


In [ ]:
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ann = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    dtype={"database_id": str, "hpo_id": str},
    low_memory=False
)
split = pd.read_csv(DATA / "disease_level_split.csv", dtype=str)
inputs = pd.read_csv(DATA / "heldout_input_phenotypes.csv", dtype=str)
targets = pd.read_csv(DATA / "heldout_target_phenotypes.csv", dtype=str)

train_ids = set(split.loc[split["split"] == "train", "disease_id"].str.strip())
test_ids = set(split.loc[split["split"] == "test", "disease_id"].str.strip())

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()
ann = ann[ann["database_id"].isin(train_ids)].dropna(subset=["hpo_id"])

profiles = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()
input_map = inputs.groupby("disease_id")["hpo_id"].apply(set).to_dict()
target_map = targets.groupby("disease_id")["hpo_id"].apply(set).to_dict()

term_counts = defaultdict(int)
for terms in profiles.values():
    for term in terms:
        term_counts[term] += 1

n_train = max(len(profiles), 1)
ic = {t: -np.log(c / n_train) for t, c in term_counts.items()}

methods = ["weighted_f1", "jaccard", "ic_overlap"]
rows = []

for disease_id in sorted(test_ids & set(input_map) & set(target_map)):
    observed = input_map[disease_id] & set(ic)
    hidden = target_map[disease_id]

    if not observed or not hidden:
        continue

    observed_ic = sum(ic[t] for t in observed)
    scored = {method: [] for method in methods}

    for candidate, terms in profiles.items():
        overlap = observed & terms
        union = observed | terms

        # Weighted F1 using training-only information content
        overlap_ic = sum(ic[t] for t in overlap)
        precision = overlap_ic / observed_ic if observed_ic else 0.0
        recall = overlap_ic / sum(ic[t] for t in terms if t in ic) \
            if any(t in ic for t in terms) else 0.0
        weighted_f1 = (
            2 * precision * recall / (precision + recall)
            if precision + recall else 0.0
        )

        # Standard Jaccard similarity
        jaccard = len(overlap) / len(union) if union else 0.0

        # Fraction of observed IC explained by candidate
        ic_overlap = overlap_ic / observed_ic if observed_ic else 0.0

        scored["weighted_f1"].append((candidate, weighted_f1))
        scored["jaccard"].append((candidate, jaccard))
        scored["ic_overlap"].append((candidate, ic_overlap))

    for method, candidates in scored.items():
        candidates.sort(key=lambda x: (-x[1], x[0]))
        top_candidate = candidates[0][0]
        candidate_terms = profiles[top_candidate]
        recovered = len(hidden & candidate_terms)

        rows.append({
            "disease_id": disease_id,
            "method": method,
            "top_training_candidate": top_candidate,
            "hidden_phenotype_count": len(hidden),
            "recovered_hidden_phenotypes": recovered,
            "hidden_phenotype_recall": recovered / len(hidden),
        })

results = pd.DataFrame(rows)
results.to_csv(DATA / "ranking_method_comparison.csv", index=False)

summary = (
    results.groupby("method")
    .agg(
        profiles=("disease_id", "nunique"),
        mean_hidden_recall=("hidden_phenotype_recall", "mean"),
        median_hidden_recall=("hidden_phenotype_recall", "median"),
    )
    .reset_index()
)

print(summary.to_string(index=False))
print("\nSaved:", DATA / "ranking_method_comparison.csv")

     method  profiles  mean_hidden_recall  median_hidden_recall
 ic_overlap      1828            0.341602              0.333333
    jaccard      1828            0.173206              0.100000
weighted_f1      1828            0.209105              0.142857

Saved: /content/drive/MyDrive/Rare Disease ai/processed/ranking_method_comparison.csv


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

results = pd.read_csv(DATA / "ranking_method_comparison.csv")
annotations = pd.read_csv(
    DATA / "clean_hpo_annotations.csv",
    dtype={"database_id": str, "hpo_id": str},
    low_memory=False
)
split = pd.read_csv(DATA / "disease_level_split.csv", dtype=str)

train_ids = set(
    split.loc[split["split"] == "train", "disease_id"].str.strip()
)

annotations["database_id"] = annotations["database_id"].str.strip()
annotations["hpo_id"] = annotations["hpo_id"].str.strip()

train = annotations[
    annotations["database_id"].isin(train_ids)
].dropna(subset=["hpo_id"])

profiles = train.groupby("database_id")["hpo_id"].apply(set).to_dict()

term_counts = {}
for terms in profiles.values():
    for term in terms:
        term_counts[term] = term_counts.get(term, 0) + 1

n_train = len(profiles)
ic = {
    term: -np.log(count / n_train)
    for term, count in term_counts.items()
}

inputs = pd.read_csv(DATA / "heldout_input_phenotypes.csv", dtype=str)
inputs["disease_id"] = inputs["disease_id"].str.strip()
inputs["hpo_id"] = inputs["hpo_id"].str.strip()
input_map = inputs.groupby("disease_id")["hpo_id"].apply(set).to_dict()

tie_count = 0
zero_score_count = 0
top_scores = []

for disease_id, observed in input_map.items():
    observed = observed & set(ic)
    if not observed:
        continue

    denominator = sum(ic[t] for t in observed)
    scores = []
    for disease, terms in profiles.items():
        score = (
            sum(ic[t] for t in observed & terms) / denominator
            if denominator else 0.0
        )
        scores.append(score)

    scores.sort(reverse=True)
    top_scores.append(scores[0])

    if scores[0] == 0:
        zero_score_count += 1
    if sum(np.isclose(s, scores[0]) for s in scores) > 1:
        tie_count += 1

print("Training candidate profiles:", len(profiles))
print("Queries with usable phenotypes:", len(top_scores))
print("Queries whose best IC-overlap score is zero:", zero_score_count)
print("Queries with tied top scores:", tie_count)
print("Mean best IC-overlap score:",
      round(float(np.mean(top_scores)), 4) if top_scores else "N/A")
print("\nSaved method comparison:")
print(
    results.groupby("method")["hidden_phenotype_recall"]
    .agg(["mean", "median", "count"])
    .round(4)
)

Training candidate profiles: 7340
Queries with usable phenotypes: 1828
Queries whose best IC-overlap score is zero: 0
Queries with tied top scores: 410
Mean best IC-overlap score: 0.6277

Saved method comparison:
               mean  median  count
method                            
ic_overlap   0.3416  0.3333   1828
jaccard      0.1732  0.1000   1828
weighted_f1  0.2091  0.1429   1828


In [ ]:
from pathlib import Path
import importlib.util

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

print("HPO ontology:", (PROJECT / "hp.obo").exists())
print("Existing hierarchy results:",
      (DATA / "hierarchy_aware_retrieval_metrics.csv").exists())

for package in ["obonet", "networkx"]:
    print(f"{package} installed:",
          importlib.util.find_spec(package) is not None)

HPO ontology: True
Existing hierarchy results: True
obonet installed: False
networkx installed: True


In [ ]:
%pip install obonet

from pathlib import Path
import pandas as pd
import obonet
import networkx as nx

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

# Load HPO ontology
with open(PROJECT / "hp.obo", "r", encoding="utf-8") as f:
    ontology = obonet.read_obo(f)

print("HPO ontology terms:", len(ontology))
print("Ontology edges:", ontology.number_of_edges())

# Inspect existing hierarchy-aware results
path = DATA / "hierarchy_aware_retrieval_metrics.csv"
df = pd.read_csv(path)

print("\nExisting hierarchy evaluation:", df.shape)
print(df.head().to_string(index=False))

metric_cols = [
    c for c in ["recall_at_1", "recall_at_5", "recall_at_10",
                "reciprocal_rank"] if c in df.columns
]
print("\nExisting metric averages:")
print(df[metric_cols].mean().round(4))

HPO ontology terms: 19894
Ontology edges: 24436

Existing hierarchy evaluation: (1829, 6)
 disease_id  rank  recall_at_1  recall_at_5  recall_at_10  reciprocal_rank
OMIM:100100    49            0            0             0         0.020408
OMIM:101900    65            0            0             0         0.015385
OMIM:102530    75            0            0             0         0.013333
OMIM:102900    79            0            0             0         0.012658
OMIM:103500    86            0            0             0         0.011628

Existing metric averages:
recall_at_1        0.0000
recall_at_5        0.0000
recall_at_10       0.0000
reciprocal_rank    0.0004
dtype: float64


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

hier = pd.read_csv(DATA / "hierarchy_aware_retrieval_metrics.csv")
coverage = pd.read_csv(DATA / "candidate_coverage_audit.csv")

print("HIERARCHY RANK DISTRIBUTION")
print(hier["rank"].describe(percentiles=[.25, .5, .75, .9, .99]))
print("Rank <= 10:", int((hier["rank"] <= 10).sum()))
print("Rank <= 100:", int((hier["rank"] <= 100).sum()))
print("Missing ranks:", int(hier["rank"].isna().sum()))

print("\nCANDIDATE COVERAGE")
print("Profiles:", len(coverage))
print("Zero candidates:", int((coverage["candidate_count"] == 0).sum()))
print("Candidate count summary:")
print(coverage["candidate_count"].describe(
    percentiles=[.25, .5, .75, .9]
))

print("\nSAMPLE LOW-COVERAGE PROFILES")
print(
    coverage.sort_values("candidate_count")
    .head(10).to_string(index=False)
)

HIERARCHY RANK DISTRIBUTION
count     1829.000000
mean      6405.753417
std       3472.250644
min         49.000000
25%       3658.000000
50%       6388.000000
75%       8936.000000
90%      11495.600000
99%      12741.600000
max      12876.000000
Name: rank, dtype: float64
Rank <= 10: 0
Rank <= 100: 7
Missing ranks: 0

CANDIDATE COVERAGE
Profiles: 1832
Zero candidates: 4
Candidate count summary:
count    1832.000000
mean     5906.044760
std      2218.459707
min         0.000000
25%      4692.750000
50%      6528.500000
75%      7556.500000
90%      8258.400000
max      9305.000000
Name: candidate_count, dtype: float64

SAMPLE LOW-COVERAGE PROFILES
 disease_id  input_count  covered_input_count  candidate_count
OMIM:619319            1                    0                0
OMIM:303900            1                    0                0
OMIM:102900            1                    0                0
OMIM:300813            1                    0                0
OMIM:136800            3    

In [ ]:
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

for p in sorted(DATA.glob("*")):
    if any(word in p.name.lower() for word in [
        "hierarchy", "ontology", "ranking", "baseline", "evaluation"
    ]):
        print(p.name)

baseline_coverage_comparison_audit.csv
baseline_model_comparison.csv
direct_vs_hierarchy_scoring_comparison.csv
disease_ranking_retrieval_metrics.csv
evaluation_eligibility_audit.csv
evaluation_protocol_checks.csv
gnn_evaluation_results.csv
hierarchy_aware_retrieval_metrics.csv
ic_weighted_heldout_ranking_results.csv
jaccard_baseline_top10.csv
leakage_reduced_retrieval_evaluation.csv
missing_evaluation_profiles.csv
ontology_fallback_candidates.csv
ontology_fallback_coverage_summary.csv
ontology_fallback_evaluation.csv
ontology_fallback_ranked.csv
ontology_fallback_target_comparison.csv
phenotype_baseline_per_profile.csv
phenotype_disease_ranking_evaluation.csv
phenotype_overlap_baseline_top10.csv
phenotype_recovery_baseline_metrics.csv
ranking_method_comparison.csv
zero_candidate_ontology_ancestors.csv
zero_candidate_ontology_check.csv
zero_candidate_ontology_parents.csv


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = [
    "direct_vs_hierarchy_scoring_comparison.csv",
    "ontology_fallback_coverage_summary.csv",
    "ontology_fallback_evaluation.csv",
    "phenotype_recovery_baseline_metrics.csv",
    "ranking_method_comparison.csv",
]

for name in files:
    path = DATA / name
    print(f"\n{'=' * 65}\n{name}")

    if not path.exists():
        print("Missing")
        continue

    df = pd.read_csv(path, low_memory=False)
    print("Shape:", df.shape)
    print(df.to_string(index=False, max_rows=12))


direct_vs_hierarchy_scoring_comparison.csv
Shape: (1829, 11)
 disease_id  direct_rank  direct_reciprocal_rank  direct_recall_at_1  direct_recall_at_5  direct_recall_at_10  hierarchy_rank  hierarchy_reciprocal_rank  hierarchy_recall_at_1  hierarchy_recall_at_5  hierarchy_recall_at_10
OMIM:100100            1                     1.0                   1                   1                    1               1                        1.0                      1                      1                       1
OMIM:101900            1                     1.0                   1                   1                    1               1                        1.0                      1                      1                       1
OMIM:102530            1                     1.0                   1                   1                    1               1                        1.0                      1                      1                       1
OMIM:102900            1                     1

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

direct = pd.read_csv(DATA / "direct_vs_hierarchy_scoring_comparison.csv")
methods = pd.read_csv(DATA / "ranking_method_comparison.csv")

summary = []

for label, df, prefix in [
    ("Direct scoring", direct, "direct"),
    ("Hierarchy scoring", direct, "hierarchy"),
]:
    summary.append({
        "Method": label,
        "Profiles": len(df),
        "Recall@1": df[f"{prefix}_recall_at_1"].mean(),
        "Recall@5": df[f"{prefix}_recall_at_5"].mean(),
        "Recall@10": df[f"{prefix}_recall_at_10"].mean(),
        "MRR": df[f"{prefix}_reciprocal_rank"].mean(),
    })

for method, group in methods.groupby("method"):
    summary.append({
        "Method": f"{method} (phenotype recovery)",
        "Profiles": group["disease_id"].nunique(),
        "Mean hidden-phenotype recall":
            group["hidden_phenotype_recall"].mean(),
        "Median hidden-phenotype recall":
            group["hidden_phenotype_recall"].median(),
    })

summary_df = pd.DataFrame(summary)
out = DATA / "ranking_method_consolidated_summary.csv"
summary_df.to_csv(out, index=False)

print(summary_df.round(4).to_string(index=False))
print("\nSaved:", out)

                          Method  Profiles  Recall@1  Recall@5  Recall@10    MRR  Mean hidden-phenotype recall  Median hidden-phenotype recall
                  Direct scoring      1829    0.9306    0.9716     0.9787 0.9483                           NaN                             NaN
               Hierarchy scoring      1829    0.9306    0.9716     0.9787 0.9483                           NaN                             NaN
 ic_overlap (phenotype recovery)      1828       NaN       NaN        NaN    NaN                        0.3416                          0.3333
    jaccard (phenotype recovery)      1828       NaN       NaN        NaN    NaN                        0.1732                          0.1000
weighted_f1 (phenotype recovery)      1828       NaN       NaN        NaN    NaN                        0.2091                          0.1429

Saved: /content/drive/MyDrive/Rare Disease ai/processed/ranking_method_consolidated_summary.csv


In [ ]:
from pathlib import Path
from collections import Counter

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
PHENO = PROJECT / "all_phenopackets"

print("Folder exists:", PHENO.exists())

if PHENO.exists():
    files = [p for p in PHENO.rglob("*") if p.is_file()]
    print("Total files:", len(files))

    extensions = Counter(p.suffix.lower() or "[no extension]" for p in files)
    print("\nFile types:")
    for ext, count in extensions.most_common(15):
        print(f"{ext}: {count}")

    print("\nExample paths:")
    for p in files[:25]:
        print(p.relative_to(PHENO))
else:
    print("Check whether the extracted folder is nested elsewhere in the project.")

Folder exists: True
Total files: 10428

File types:
.json: 10427
.tsv: 1

Example paths:
0.1.27/phenopacket_store.summary.tsv
0.1.27/ADAMTS10/PMID_19836009_Family_E_II_6.json
0.1.27/ADAMTS10/PMID_19836009_Family_E_II_3.json
0.1.27/ADAMTS10/PMID_19836009_Family_B_II_4.json
0.1.27/ADAMTS10/PMID_19836009_Family_B_II_2.json
0.1.27/ADAMTS10/PMID_25469541_patient.json
0.1.27/AAGAB/PMID_24573067_CASE_REPORT.json
0.1.27/AAGAB/PMID_28239884_Family_1_proband.json
0.1.27/AAGAB/PMID_28239884_Family_2_proband.json
0.1.27/AAGAB/PMID_28239884_Family_3_proband.json
0.1.27/ADA/PMID_28823388_3_month_old_girl.json
0.1.27/ADA/PMID_1680289_patient.json
0.1.27/ACTB/PMID_30315159_Patient_1.json
0.1.27/ACTB/PMID_30733661_4_year_old_girl.json
0.1.27/ACTB/PMID_30315159_Patient_4_Mother_of_Patient_3.json
0.1.27/ACTB/PMID_30315159_Patient_2_Father_of_Patient_1.json
0.1.27/ACTB/PMID_30315159_Patient_6.json
0.1.27/ACTB/PMID_30315159_Patient_N.json
0.1.27/ACTB/PMID_30315159_Patient_5.json
0.1.27/ACTB/PMID_30315159_P

In [ ]:
from pathlib import Path
import json
from collections import Counter

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
PHENO = PROJECT / "all_phenopackets"

json_files = list(PHENO.rglob("*.json"))
print("JSON files found:", len(json_files))

# Inspect up to 3 cases without printing potentially identifying case details
for path in json_files[:3]:
    print("\nFile:", path.name)

    with open(path, "r", encoding="utf-8") as f:
        case = json.load(f)

    print("Top-level keys:", list(case.keys()))

    subject = case.get("subject", {})
    print("Subject keys:", list(subject.keys()))

    phenotypes = case.get("phenotypicFeatures", [])
    print("Phenotypic features:", len(phenotypes))
    if phenotypes:
        print("First phenotype keys:", list(phenotypes[0].keys()))
        print("First phenotype type:",
              phenotypes[0].get("type", {}).get("id", "not found"))

    diseases = case.get("diseases", [])
    print("Recorded diagnoses:", len(diseases))
    if diseases:
        print("First diagnosis keys:", list(diseases[0].keys()))
        print("First diagnosis term:",
              diseases[0].get("term", {}).get("id", "not found"))

JSON files found: 10427

File: PMID_19836009_Family_E_II_6.json
Top-level keys: ['id', 'subject', 'phenotypicFeatures', 'interpretations', 'diseases', 'metaData']
Subject keys: ['id', 'timeAtLastEncounter', 'sex']
Phenotypic features: 9
First phenotype keys: ['type', 'excluded']
First phenotype type: HP:0007833
Recorded diagnoses: 1
First diagnosis keys: ['term']
First diagnosis term: OMIM:277600

File: PMID_19836009_Family_E_II_3.json
Top-level keys: ['id', 'subject', 'phenotypicFeatures', 'interpretations', 'diseases', 'metaData']
Subject keys: ['id', 'timeAtLastEncounter', 'sex']
Phenotypic features: 9
First phenotype keys: ['type', 'excluded']
First phenotype type: HP:0007833
Recorded diagnoses: 1
First diagnosis keys: ['term']
First diagnosis term: OMIM:277600

File: PMID_19836009_Family_B_II_4.json
Top-level keys: ['id', 'subject', 'phenotypicFeatures', 'interpretations', 'diseases', 'metaData']
Subject keys: ['id', 'timeAtLastEncounter', 'sex']
Phenotypic features: 9
First pheno

In [ ]:
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
out = DATA / "phenopacket_patient_cases.csv"

print("Output file exists:", out.exists())
if out.exists():
    print("Output size (MB):", round(out.stat().st_size / (1024 * 1024), 2))

Output file exists: False


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
PHENO = PROJECT / "all_phenopackets"
DATA = PROJECT / "processed"
DATA.mkdir(parents=True, exist_ok=True)

checkpoint = DATA / "phenopacket_parse_checkpoint.csv"
output = DATA / "phenopacket_patient_cases.csv"

# Resume from previously saved progress, if available
rows = []
done = set()

if checkpoint.exists():
    old = pd.read_csv(checkpoint, dtype=str).fillna("")
    rows = old.to_dict("records")
    done = set(old["source_file"])
    print("Resuming:", len(done), "cases already processed")

files = [p for p in PHENO.rglob("*.json")
         if str(p.relative_to(PHENO)) not in done]
print("Cases remaining:", len(files))

errors = 0
for i, path in enumerate(files, 1):
    try:
        with open(path, "r", encoding="utf-8") as f:
            case = json.load(f)

        observed, excluded = set(), set()
        for feature in case.get("phenotypicFeatures", []):
            term = feature.get("type", {}).get("id")
            if not term or not term.startswith("HP:"):
                continue
            (excluded if feature.get("excluded") is True else observed).add(term)

        observed -= excluded
        diagnoses = {
            d.get("term", {}).get("id")
            for d in case.get("diseases", [])
            if d.get("term", {}).get("id")
        }

        rows.append({
            "case_id": case.get("id") or path.stem,
            "source_file": str(path.relative_to(PHENO)),
            "hpo_terms": ";".join(sorted(observed)),
            "phenotype_count": len(observed),
            "diagnosis_ids": ";".join(sorted(diagnoses)),
            "diagnosis_count": len(diagnoses),
        })
    except Exception:
        errors += 1

    if i % 500 == 0:
        pd.DataFrame(rows).to_csv(checkpoint, index=False)
        print(f"Processed {i}/{len(files)} remaining files")

pd.DataFrame(rows).to_csv(output, index=False)
if checkpoint.exists():
    checkpoint.unlink()

print("\nCompleted.")
print("Cases parsed:", len(rows))
print("Errors in this run:", errors)
print("Saved:", output)

Resuming: 7000 cases already processed
Cases remaining: 3427
Processed 500/3427 remaining files
Processed 1000/3427 remaining files
Processed 1500/3427 remaining files
Processed 2000/3427 remaining files
Processed 2500/3427 remaining files
Processed 3000/3427 remaining files

Completed.
Cases parsed: 10427
Errors in this run: 0
Saved: /content/drive/MyDrive/Rare Disease ai/processed/phenopacket_patient_cases.csv


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

print("Project exists:", PROJECT.exists())
print("Processed folder exists:", DATA.exists())
print("Model folder exists:", MODEL.exists())

print("\nProcessed files:")
for f in sorted(DATA.iterdir()):
    print(f.name)

Project exists: True
Processed folder exists: True
Model folder exists: True

Processed files:
baseline_coverage_comparison_audit.csv
baseline_model_comparison.csv
candidate_coverage_audit.csv
clean_hpo_annotations.csv
direct_vs_hierarchy_scoring_comparison.csv
disease_level_split.csv
disease_ranking_retrieval_metrics.csv
evaluation_eligibility_audit.csv
evaluation_protocol_checks.csv
final_artifact_inventory.csv
final_consolidated_results.csv
final_project_handover_inventory.csv
gene_informed_phenotype_recovery_metrics.csv
gene_informed_phenotype_recovery_top10.csv
gnn_checkpoint_compatibility.csv
gnn_evaluation_results.csv
gnn_phenotype_recovery_metrics.csv
gnn_training_annotations.csv
gnn_zero_shot_hidden_targets.csv
gnn_zero_shot_input_profiles.csv
heldout_gene_coverage.csv
heldout_input_phenotypes.csv
heldout_profile_nearest_training_disease.csv
heldout_target_phenotypes.csv
hierarchy_aware_retrieval_metrics.csv
ic_weighted_heldout_ranking_results.csv
jaccard_baseline_top10.csv
le

In [ ]:

from pathlib import Path
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

required_files = [
    DATA / "clean_hpo_annotations.csv",
    DATA / "phenopacket_patient_cases.csv",
    DATA / "disease_level_split.csv",
    DATA / "gnn_training_annotations.csv",
    DATA / "rare_disease_gnn_training_only.pt",
    DATA / "rare_disease_final_research_report.md",
]

print("PREPROCESSING AND ASSET CHECK")
print("-" * 45)

for file in required_files:
    status = "OK" if file.exists() else "MISSING"
    size = f"{file.stat().st_size / (1024 * 1024):.2f} MB" if file.exists() else "-"
    print(f"{status:8} | {size:>10} | {file.name}")

if all(file.exists() for file in required_files):
    print("\nAll required files are present.")
else:
    print("\nSome files are missing; we will investigate before proceeding.")

PREPROCESSING AND ASSET CHECK
---------------------------------------------
OK       |   34.24 MB | clean_hpo_annotations.csv
OK       |    1.87 MB | phenopacket_patient_cases.csv
OK       |    0.16 MB | disease_level_split.csv
OK       |   21.95 MB | gnn_training_annotations.csv
OK       |    4.73 MB | rare_disease_gnn_training_only.pt
OK       |    0.01 MB | rare_disease_final_research_report.md

All required files are present.


In [ ]:
import pandas as pd

cases_file = DATA / "phenopacket_patient_cases.csv"
annotations_file = DATA / "clean_hpo_annotations.csv"
split_file = DATA / "disease_level_split.csv"
training_file = DATA / "gnn_training_annotations.csv"

cases = pd.read_csv(cases_file, low_memory=False)
annotations = pd.read_csv(annotations_file, low_memory=False)
split = pd.read_csv(split_file, low_memory=False)
training = pd.read_csv(training_file, low_memory=False)

print("DATASET READINESS REPORT")
print("=" * 45)

print("\n1. PHENOPACKET PATIENT CASES")
print("Total cases:", len(cases))
print("Cases with phenotypes:",
      int((cases["phenotype_count"] > 0).sum()))
print("Cases with diagnoses:",
      int((cases["diagnosis_count"] > 0).sum()))
print("Cases with both:",
      int(((cases["phenotype_count"] > 0) &
           (cases["diagnosis_count"] > 0)).sum()))

print("\n2. DISEASE–PHENOTYPE KNOWLEDGE BASE")
print("Total association rows:", len(annotations))
print("Unique disease IDs:", annotations["database_id"].nunique())

print("\n3. DISEASE-LEVEL SPLIT")
print("Split columns:", split.columns.tolist())
print("Split rows:", len(split))

print("\n4. TRAINING ANNOTATIONS")
print("Training association rows:", len(training))
print("Training columns:", training.columns.tolist())

DATASET READINESS REPORT

1. PHENOPACKET PATIENT CASES
Total cases: 10427
Cases with phenotypes: 10424
Cases with diagnoses: 10427
Cases with both: 10424

2. DISEASE–PHENOTYPE KNOWLEDGE BASE
Total association rows: 286047
Unique disease IDs: 12879

3. DISEASE-LEVEL SPLIT
Split columns: ['disease_id', 'split']
Split rows: 9176

4. TRAINING ANNOTATIONS
Training association rows: 179653
Training columns: ['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']


In [ ]:
def split_ids(value):
    if pd.isna(value) or not str(value).strip():
        return []
    return [
        x.strip()
        for x in str(value).split(";")
        if x.strip()
    ]

# Disease IDs present in the knowledge base
known_ids = set(
    annotations["database_id"].dropna().astype(str).str.strip()
)

# Extract the diagnosis IDs associated with each patient case
cases["diagnosis_list"] = cases["diagnosis_ids"].apply(split_ids)

# Check for exact diagnosis-ID matches
cases["has_matching_diagnosis"] = cases["diagnosis_list"].apply(
    lambda ids: any(d in known_ids for d in ids)
)

# Initial eligible cases: phenotype data, diagnosis, and a matching ID
eligible = cases[
    (cases["phenotype_count"] > 0)
    & (cases["diagnosis_count"] > 0)
    & cases["has_matching_diagnosis"]
].copy()

print("PATIENT DIAGNOSIS COVERAGE")
print("=" * 40)
print("Total patient cases:", len(cases))
print("Cases with a matching diagnosis ID:",
      int(cases["has_matching_diagnosis"].sum()))
print("Cases eligible for initial evaluation:", len(eligible))

matched_ids = {
    disease_id
    for ids in eligible["diagnosis_list"]
    for disease_id in ids
    if disease_id in known_ids
}

print("Unique matched diagnosis IDs:", len(matched_ids))

print("\nExamples of diagnosis IDs:")
print(cases.loc[cases["has_matching_diagnosis"],
                ["case_id", "diagnosis_ids"]].head(5).to_string(index=False))

PATIENT DIAGNOSIS COVERAGE
Total patient cases: 10427
Cases with a matching diagnosis ID: 10398
Cases eligible for initial evaluation: 10395
Unique matched diagnosis IDs: 778

Examples of diagnosis IDs:
                    case_id diagnosis_ids
PMID_19836009_Family_E_II_6   OMIM:277600
PMID_19836009_Family_E_II_3   OMIM:277600
PMID_19836009_Family_B_II_4   OMIM:277600
PMID_19836009_Family_B_II_2   OMIM:277600
      PMID_25469541_patient   OMIM:277600


In [ ]:
print("CASE STRUCTURE AUDIT")
print("=" * 45)

print("\nAvailable columns:")
print(cases.columns.tolist())

print("\nExample case IDs:")
print(cases["case_id"].head(10).to_list())

print("\nRepeated case IDs:")
print("Duplicate case IDs:", int(cases["case_id"].duplicated().sum()))

print("\nDiagnosis frequency:")
print(
    cases.loc[cases["has_matching_diagnosis"], "diagnosis_list"]
    .explode()
    .value_counts()
    .head(10)
    .to_string()
)

print("\nSource-file examples:")
print(cases["source_file"].head(10).to_string(index=False))

CASE STRUCTURE AUDIT

Available columns:
['case_id', 'source_file', 'hpo_terms', 'phenotype_count', 'diagnosis_ids', 'diagnosis_count', 'diagnosis_list', 'has_matching_diagnosis']

Example case IDs:
['PMID_19836009_Family_E_II_6', 'PMID_19836009_Family_E_II_3', 'PMID_19836009_Family_B_II_4', 'PMID_19836009_Family_B_II_2', 'PMID_25469541_patient', 'PMID_24573067_CASE_REPORT', 'PMID_28239884_Family_1_proband', 'PMID_28239884_Family_2_proband', 'PMID_28239884_Family_3_proband', 'PMID_28823388_3_month_old_girl']

Repeated case IDs:
Duplicate case IDs: 50

Diagnosis frequency:
diagnosis_list
OMIM:612164    477
OMIM:162200    421
OMIM:148050    334
OMIM:162000    199
OMIM:612313    157
OMIM:142900    150
OMIM:151660    127
OMIM:610253    125
OMIM:616364    117
OMIM:117360    104

Source-file examples:
0.1.27/ADAMTS10/PMID_19836009_Family_E_II_6.json
0.1.27/ADAMTS10/PMID_19836009_Family_E_II_3.json
0.1.27/ADAMTS10/PMID_19836009_Family_B_II_4.json
0.1.27/ADAMTS10/PMID_19836009_Family_B_II_2.js

In [ ]:
# Identify duplicate records
duplicate_cases = cases[cases["case_id"].duplicated(keep=False)].copy()

# Extract publication ID from case ID, e.g. PMID_19836009_...
cases["publication_id"] = cases["case_id"].str.extract(
    r"(PMID_\d+)", expand=False
)

print("DUPLICATE AND GROUPING AUDIT")
print("=" * 45)

print("Total records:", len(cases))
print("Duplicate case-ID rows:",
      int(cases["case_id"].duplicated().sum()))
print("Rows involved in duplicate IDs:",
      len(duplicate_cases))
print("Unique publication IDs:",
      cases["publication_id"].nunique())
print("Rows without a publication ID:",
      int(cases["publication_id"].isna().sum()))

print("\nPublications contributing the most cases:")
print(
    cases["publication_id"]
    .value_counts()
    .head(10)
    .to_string()
)

print("\nDuplicate case-ID examples:")
print(
    duplicate_cases[
        ["case_id", "source_file", "diagnosis_ids"]
    ].head(10).to_string(index=False)
)

DUPLICATE AND GROUPING AUDIT
Total records: 10427
Duplicate case-ID rows: 50
Rows involved in duplicate IDs: 100
Unique publication IDs: 1733
Rows without a publication ID: 0

Publications contributing the most cases:
publication_id
PMID_35190816    477
PMID_36446582    334
PMID_29290338    157
PMID_31021519    157
PMID_26178382    136
PMID_39013458    125
PMID_37843397    115
PMID_25802881    103
PMID_33674768     80
PMID_37761890     76

Duplicate case-ID examples:
                                    case_id                                                         source_file diagnosis_ids
PMID_36446582_Kutkowska_Kazmierczak2021_P23 0.1.27/ANKRD11/PMID_36446582_Kutkowska_Kazmierczak2021_P23 (1).json   OMIM:148050
PMID_36446582_Kutkowska_Kazmierczak2021_P23     0.1.27/ANKRD11/PMID_36446582_Kutkowska_Kazmierczak2021_P23.json   OMIM:148050
 PMID_20513137_individual_GUI_GSM_GSM492741      0.1.27/NF1/PMID_20513137_individual_GUI_GSM_GSM492741 (1).json   OMIM:162200
 PMID_20513137_individua

In [ ]:
from collections import Counter

def normalize_terms(value):
    if pd.isna(value) or not str(value).strip():
        return set()
    return {
        term.strip()
        for term in str(value).split(";")
        if term.strip()
    }

duplicate_groups = cases[
    cases["case_id"].duplicated(keep=False)
].groupby("case_id", sort=False)

identical_duplicates = 0
different_duplicates = 0

for case_id, group in duplicate_groups:
    phenotype_sets = group["hpo_terms"].apply(normalize_terms)
    diagnosis_sets = group["diagnosis_list"].apply(
        lambda x: set(x) if isinstance(x, list) else set()
    )

    identical = (
        phenotype_sets.apply(lambda x: frozenset(x)).nunique() == 1
        and diagnosis_sets.apply(lambda x: frozenset(x)).nunique() == 1
    )

    if identical:
        identical_duplicates += 1
    else:
        different_duplicates += 1

print("DUPLICATE CONTENT CHECK")
print("=" * 40)
print("Duplicate case-ID groups:", len(duplicate_groups))
print("Groups with identical phenotype and diagnosis sets:",
      identical_duplicates)
print("Groups with differing contents:", different_duplicates)

DUPLICATE CONTENT CHECK
Duplicate case-ID groups: 50
Groups with identical phenotype and diagnosis sets: 50
Groups with differing contents: 0


In [ ]:
# Preserve the original `cases` DataFrame.
cases_eval = cases.drop_duplicates(
    subset="case_id",
    keep="first"
).copy()

print("CLEAN EVALUATION DATASET")
print("=" * 40)
print("Original records:", len(cases))
print("Records after deduplication:", len(cases_eval))
print("Duplicate records removed:", len(cases) - len(cases_eval))
print("Remaining duplicate case IDs:",
      int(cases_eval["case_id"].duplicated().sum()))

print("\nEligible cases after deduplication:",
      int((
          (cases_eval["phenotype_count"] > 0)
          & (cases_eval["diagnosis_count"] > 0)
          & cases_eval["has_matching_diagnosis"]
      ).sum()))

CLEAN EVALUATION DATASET
Original records: 10427
Records after deduplication: 10377
Duplicate records removed: 50
Remaining duplicate case IDs: 0

Eligible cases after deduplication: 10345


In [ ]:
import pandas as pd

files_to_check = {
    "protocol_checks": DATA / "evaluation_protocol_checks.csv",
    "eligibility_audit": DATA / "evaluation_eligibility_audit.csv",
    "final_results": DATA / "final_consolidated_results.csv",
    "training_annotations": DATA / "gnn_training_annotations.csv",
    "disease_split": DATA / "disease_level_split.csv",
}

for name, path in files_to_check.items():
    print(f"\n{'=' * 15} {name.upper()} {'=' * 15}")

    if path.exists():
        df = pd.read_csv(path, low_memory=False)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(5).to_string(index=False))
    else:
        print("FILE MISSING:", path.name)


=============== PROTOCOL_CHECKS ===============
Shape: (6, 2)
Columns: ['Check', 'Value']
                              Check  Value
               Input phenotype rows  21647
       Hidden target phenotype rows  22562
          Input/target pair overlap      0
             Training diseases used   7340
Held-out diseases in training index      0

=============== ELIGIBILITY_AUDIT ===============
Shape: (1836, 5)
Columns: ['disease_id', 'input_phenotype_count', 'target_phenotype_count', 'in_metrics', 'status']
 disease_id  input_phenotype_count  target_phenotype_count  in_metrics    status
OMIM:100100                      8                       8        True evaluated
OMIM:101900                      4                       5        True evaluated
OMIM:102530                      2                       2        True evaluated
OMIM:102900                      1                       2        True evaluated
OMIM:103500                      4                       5        True evaluate

In [ ]:
 # Build disease-to-phenotype sets from the saved knowledge base.
disease_col = "database_id"
phenotype_col = "hpo_id"

kb = annotations[[disease_col, phenotype_col]].dropna().copy()

kb[disease_col] = kb[disease_col].astype(str).str.strip()
kb[phenotype_col] = kb[phenotype_col].astype(str).str.strip()

disease_phenotypes = (
    kb.groupby(disease_col)[phenotype_col]
    .apply(set)
    .to_dict()
)

# Convert patient phenotype strings to sets.
cases_eval["phenotype_set"] = cases_eval["hpo_terms"].apply(
    normalize_terms
)

# Count exact phenotype-set matches against each recorded diagnosis.
def patient_diagnosis_overlap(row):
    patient_terms = row["phenotype_set"]
    matched_diagnoses = [
        d for d in row["diagnosis_list"]
        if d in disease_phenotypes
    ]

    if not patient_terms or not matched_diagnoses:
        return None

    return {
        d: len(patient_terms & disease_phenotypes[d])
        / len(patient_terms)
        for d in matched_diagnoses
    }

overlaps = cases_eval.apply(patient_diagnosis_overlap, axis=1)

print("PATIENT–KNOWLEDGE-BASE OVERLAP CHECK")
print("=" * 45)
print("Unique disease profiles in knowledge base:",
      len(disease_phenotypes))
print("Unique patient cases:", len(cases_eval))
print("Cases with phenotype sets:",
      int(cases_eval["phenotype_set"].map(bool).sum()))
print("Cases with diagnosis/knowledge-base comparison:",
      int(overlaps.notna().sum()))

PATIENT–KNOWLEDGE-BASE OVERLAP CHECK
Unique disease profiles in knowledge base: 12879
Unique patient cases: 10377
Cases with phenotype sets: 10374
Cases with diagnosis/knowledge-base comparison: 10345


In [ ]:
import re

def extract_pmid(value):
    if pd.isna(value):
        return None
    match = re.search(r"PMID[_:]?(\d+)", str(value), re.IGNORECASE)
    return match.group(1) if match else None

# Publication IDs from patient source files
cases_eval["pmid"] = cases_eval["source_file"].apply(extract_pmid)

# Publication IDs from disease–phenotype annotation references
annotations["pmid"] = annotations["reference"].apply(extract_pmid)

patient_pmids = set(cases_eval["pmid"].dropna())
annotation_pmids = set(annotations["pmid"].dropna())

shared_pmids = patient_pmids & annotation_pmids

cases_eval["publication_in_knowledge_base"] = (
    cases_eval["pmid"].isin(annotation_pmids)
)

print("PUBLICATION OVERLAP AUDIT")
print("=" * 45)
print("Unique patient publications:", len(patient_pmids))
print("Unique annotation publications:", len(annotation_pmids))
print("Publications appearing in both:", len(shared_pmids))
print("Patient cases from shared publications:",
      int(cases_eval["publication_in_knowledge_base"].sum()))
print("Patient cases without a matched PMID:",
      int(cases_eval["pmid"].isna().sum()))

print("\nExamples of shared publications:")
print(sorted(shared_pmids)[:15])

PUBLICATION OVERLAP AUDIT
Unique patient publications: 1733
Unique annotation publications: 9314
Publications appearing in both: 1240
Patient cases from shared publications: 7454
Patient cases without a matched PMID: 0

Examples of shared publications:
['10077612', '10198255', '10487826', '10498624', '10534505', '10560675', '10580070', '10805329', '10842287', '10851256', '10939567', '11050011', '11093277', '11118249', '11179005']


In [ ]:
audit_path = DATA / "patient_publication_overlap_audit.csv"

publication_audit = pd.DataFrame([{
    "unique_patient_publications": len(patient_pmids),
    "unique_annotation_publications": len(annotation_pmids),
    "shared_publications": len(shared_pmids),
    "patient_cases": len(cases_eval),
    "cases_from_shared_publications": int(
        cases_eval["publication_in_knowledge_base"].sum()
    ),
    "cases_without_matched_pmid": int(
        cases_eval["pmid"].isna().sum()
    ),
    "interpretation": (
        "Potential information overlap; shared PMIDs alone "
        "do not establish leakage."
    )
}])

publication_audit.to_csv(audit_path, index=False)

print(publication_audit.to_string(index=False))
print("\nSaved to:", audit_path)

 unique_patient_publications  unique_annotation_publications  shared_publications  patient_cases  cases_from_shared_publications  cases_without_matched_pmid                                                              interpretation
                        1733                            9314                 1240          10377                            7454                           0 Potential information overlap; shared PMIDs alone do not establish leakage.

Saved to: /content/drive/MyDrive/Rare Disease ai/processed/patient_publication_overlap_audit.csv


In [ ]:
import numpy as np
import pandas as pd

# Use only unique cases with phenotypes and a matching diagnosis.
eligible_cases = cases_eval[
    (cases_eval["phenotype_count"] > 0)
    & (cases_eval["diagnosis_count"] > 0)
    & cases_eval["has_matching_diagnosis"]
].copy()

# Every eligible case must have a publication ID.
assert eligible_cases["pmid"].notna().all(), (
    "Some eligible cases have no publication ID."
)

# Split publications, not individual patients.
unique_pmids = np.array(sorted(eligible_cases["pmid"].unique()))
rng = np.random.default_rng(42)
rng.shuffle(unique_pmids)

n_test = max(1, round(0.20 * len(unique_pmids)))
test_pmids = set(unique_pmids[:n_test])

eligible_cases["split"] = np.where(
    eligible_cases["pmid"].isin(test_pmids),
    "test",
    "development"
)

# Save the split manifest; preserve all original files.
split_path = DATA / "patient_publication_split.csv"

eligible_cases[
    ["case_id", "pmid", "split", "diagnosis_ids"]
].to_csv(split_path, index=False)

# Verify that no publication appears in both groups.
dev_pmids = set(
    eligible_cases.loc[
        eligible_cases["split"] == "development", "pmid"
    ]
)
test_pmids_actual = set(
    eligible_cases.loc[
        eligible_cases["split"] == "test", "pmid"
    ]
)

print("PUBLICATION-GROUPED SPLIT")
print("=" * 42)
print("Eligible cases:", len(eligible_cases))
print("Development cases:",
      int((eligible_cases["split"] == "development").sum()))
print("Test cases:",
      int((eligible_cases["split"] == "test").sum()))
print("Development publications:", len(dev_pmids))
print("Test publications:", len(test_pmids_actual))
print("Publication overlap:", len(dev_pmids & test_pmids_actual))
print("Saved to:", split_path)

PUBLICATION-GROUPED SPLIT
Eligible cases: 10345
Development cases: 8386
Test cases: 1959
Development publications: 1383
Test publications: 346
Publication overlap: 0
Saved to: /content/drive/MyDrive/Rare Disease ai/processed/patient_publication_split.csv


In [ ]:
# Use the publication-grouped split saved in Step 13.
patient_split = pd.read_csv(
    DATA / "patient_publication_split.csv",
    dtype=str
)

test_pmids = set(
    patient_split.loc[
        patient_split["split"] == "test", "pmid"
    ].dropna()
)

# Extract publication IDs from annotation references.
annotation_pmids = annotations["reference"].apply(extract_pmid)

annotations_audit = annotations.copy()
annotations_audit["pmid"] = annotation_pmids

test_reference_mask = annotations_audit["pmid"].isin(test_pmids)

print("TEST-PUBLICATION ANNOTATION AUDIT")
print("=" * 45)
print("Held-out test publications:", len(test_pmids))
print("Total annotation rows:", len(annotations_audit))
print("Annotation rows citing test publications:",
      int(test_reference_mask.sum()))
print("Unique test PMIDs found in annotations:",
      annotations_audit.loc[test_reference_mask, "pmid"].nunique())
print("Annotation rows without matched PMID:",
      int(annotations_audit["pmid"].isna().sum()))

print("\nExample overlapping annotation rows:")
print(
    annotations_audit.loc[
        test_reference_mask,
        ["database_id", "hpo_id", "reference"]
    ].head(10).to_string(index=False)
)

TEST-PUBLICATION ANNOTATION AUDIT
Held-out test publications: 346
Total annotation rows: 286047
Annotation rows citing test publications: 5314
Unique test PMIDs found in annotations: 248
Annotation rows without matched PMID: 167539

Example overlapping annotation rows:
database_id     hpo_id     reference
OMIM:620502 HP:0000750 PMID:36103875
OMIM:620502 HP:0002650 PMID:36103875
OMIM:620502 HP:0000343 PMID:36103875
OMIM:620502 HP:0000431 PMID:36103875
OMIM:620502 HP:0001250 PMID:36103875
OMIM:620502 HP:0001252 PMID:36103875
OMIM:620502 HP:0002360 PMID:36103875
OMIM:620502 HP:0003593 PMID:36103875
OMIM:620502 HP:0040183 PMID:36103875
OMIM:620502 HP:0012427 PMID:36103875


In [ ]:
# Exclude annotations citing held-out test publications.
safe_annotations = annotations_audit.loc[
    ~test_reference_mask
].copy()

# Save a separate, auditable dataset.
safe_path = DATA / "patient_test_leakage_reduced_annotations.csv"
safe_annotations.drop(columns=["pmid"]).to_csv(
    safe_path,
    index=False
)

# Verify the result.
remaining_test_pmids = (
    set(safe_annotations["pmid"].dropna()) & test_pmids
)

print("LEAKAGE-REDUCED ANNOTATIONS")
print("=" * 45)
print("Original annotation rows:", len(annotations_audit))
print("Excluded rows:", int(test_reference_mask.sum()))
print("Remaining annotation rows:", len(safe_annotations))
print("Remaining rows citing test publications:",
      len(remaining_test_pmids))
print("Saved to:", safe_path)

LEAKAGE-REDUCED ANNOTATIONS
Original annotation rows: 286047
Excluded rows: 5314
Remaining annotation rows: 280733
Remaining rows citing test publications: 0
Saved to: /content/drive/MyDrive/Rare Disease ai/processed/patient_test_leakage_reduced_annotations.csv


In [ ]:
# Load the leakage-reduced annotation file.
safe_annotations = pd.read_csv(
    safe_path,
    dtype=str,
    low_memory=False
)

# Build disease-to-phenotype profiles from the reduced annotations.
safe_profiles = (
    safe_annotations.dropna(subset=["database_id", "hpo_id"])
    .assign(
        database_id=lambda df: df["database_id"].str.strip(),
        hpo_id=lambda df: df["hpo_id"].str.strip()
    )
    .groupby("database_id")["hpo_id"]
    .apply(set)
    .to_dict()
)

# Identify eligible test cases.
test_cases = eligible_cases[
    eligible_cases["split"] == "test"
].copy()

test_cases["diagnosis_list"] = test_cases["diagnosis_ids"].apply(
    split_ids
)

test_cases["has_profile_after_filter"] = test_cases[
    "diagnosis_list"
].apply(
    lambda ids: any(d in safe_profiles for d in ids)
)

print("POST-FILTER DISEASE COVERAGE")
print("=" * 45)
print("Held-out patient cases:", len(test_cases))
print("Cases with at least one diagnosis profile:",
      int(test_cases["has_profile_after_filter"].sum()))
print("Cases without a diagnosis profile:",
      int((~test_cases["has_profile_after_filter"]).sum()))
print("Diseases with profiles after filtering:", len(safe_profiles))

# Save the coverage summary.
coverage_path = DATA / "patient_test_disease_coverage.csv"

pd.DataFrame([{
    "test_cases": len(test_cases),
    "cases_with_diagnosis_profile": int(
        test_cases["has_profile_after_filter"].sum()
    ),
    "cases_without_diagnosis_profile": int(
        (~test_cases["has_profile_after_filter"]).sum()
    ),
    "diseases_with_profiles": len(safe_profiles),
    "annotation_rows_after_filter": len(safe_annotations)
}]).to_csv(coverage_path, index=False)

print("\nCoverage report saved to:", coverage_path)

POST-FILTER DISEASE COVERAGE
Held-out patient cases: 1959
Cases with at least one diagnosis profile: 1551
Cases without a diagnosis profile: 408
Diseases with profiles after filtering: 12827

Coverage report saved to: /content/drive/MyDrive/Rare Disease ai/processed/patient_test_disease_coverage.csv


In [ ]:
print("PHENOTYPE BASELINE PREPARATION")
print("=" * 45)

print("Annotation columns:", safe_annotations.columns.tolist())
print("Patient phenotype examples:")

print(
    cases_eval.loc[
        cases_eval["phenotype_count"] > 0,
        ["case_id", "hpo_terms", "diagnosis_ids"]
    ].head(3).to_string(index=False)
)

# Check whether the HPO ontology file is available.
hpo_file = PROJECT / "hp.obo"

print("\nHPO ontology file exists:", hpo_file.exists())

# Count unique phenotype terms in the reduced annotations.
print(
    "Unique annotated HPO terms:",
    safe_annotations["hpo_id"].dropna().nunique()
)

PHENOTYPE BASELINE PREPARATION
Annotation columns: ['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']
Patient phenotype examples:
                    case_id                                                         hpo_terms diagnosis_ids
PMID_19836009_Family_E_II_6 HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0007906;HP:0011003   OMIM:277600
PMID_19836009_Family_E_II_3 HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0007906;HP:0011003   OMIM:277600
PMID_19836009_Family_B_II_4            HP:0000594;HP:0001156;HP:0001387;HP:0004322;HP:0011003   OMIM:277600

HPO ontology file exists: True
Unique annotated HPO terms: 11611


In [ ]:
import math
import pandas as pd

# Build disease phenotype profiles from the filtered annotations.
baseline_annotations = safe_annotations.dropna(
    subset=["database_id", "hpo_id"]
).copy()

baseline_annotations["database_id"] = (
    baseline_annotations["database_id"].astype(str).str.strip()
)
baseline_annotations["hpo_id"] = (
    baseline_annotations["hpo_id"].astype(str).str.strip()
)

disease_profiles = (
    baseline_annotations.groupby("database_id")["hpo_id"]
    .apply(set)
    .to_dict()
)

# Information content: rarer disease-associated terms receive greater weight.
term_disease_counts = {}

for terms in disease_profiles.values():
    for term in terms:
        term_disease_counts[term] = term_disease_counts.get(term, 0) + 1

n_diseases = len(disease_profiles)

term_ic = {
    term: -math.log(count / n_diseases)
    for term, count in term_disease_counts.items()
}

print("BASELINE PROFILE SUMMARY")
print("=" * 40)
print("Disease profiles:", len(disease_profiles))
print("Unique weighted HPO terms:", len(term_ic))
print("Total disease–phenotype pairs:",
      sum(len(terms) for terms in disease_profiles.values()))
print("Example information-content weights:")
print(sorted(term_ic.items(), key=lambda x: x[1], reverse=True)[:10])

BASELINE PROFILE SUMMARY
Disease profiles: 12827
Unique weighted HPO terms: 11611
Total disease–phenotype pairs: 280733
Example information-content weights:
[('HP:0007590', 9.45930760329985), ('HP:6000946', 9.45930760329985), ('HP:6000920', 9.45930760329985), ('HP:0005733', 9.45930760329985), ('HP:0025096', 9.45930760329985), ('HP:0009590', 9.45930760329985), ('HP:0500089', 9.45930760329985), ('HP:0007935', 9.45930760329985), ('HP:6000934', 9.45930760329985), ('HP:0007343', 9.45930760329985)]


In [ ]:
import numpy as np
import pandas as pd

# Use the existing held-out patient cases.
test_cases = eligible_cases[
    eligible_cases["split"] == "test"
].copy()

# Keep only cases with at least one matching diagnosis profile.
test_cases["diagnosis_list"] = test_cases["diagnosis_ids"].apply(split_ids)

def weighted_overlap(query_terms, disease_terms):
    shared = query_terms & disease_terms
    denominator = sum(term_ic.get(t, 0.0) for t in query_terms)
    if denominator <= 0:
        return 0.0
    return sum(term_ic.get(t, 0.0) for t in shared) / denominator

results = []

for _, row in test_cases.iterrows():
    query_terms = normalize_terms(row["hpo_terms"])
    if not query_terms:
        continue

    # Rank all diseases using patient phenotypes only.
    ranked = sorted(
        (
            (disease_id, weighted_overlap(query_terms, terms))
            for disease_id, terms in disease_profiles.items()
        ),
        key=lambda x: x[1],
        reverse=True
    )

    ranked_ids = [d for d, _ in ranked]
    true_diagnoses = set(row["diagnosis_list"])
    ranks = [
        ranked_ids.index(d) + 1
        for d in true_diagnoses
        if d in ranked_ids
    ]

    results.append({
        "case_id": row["case_id"],
        "pmid": row["pmid"],
        "diagnosis_ids": row["diagnosis_ids"],
        "rank": min(ranks) if ranks else np.nan,
        "top1": int(bool(ranks) and min(ranks) == 1),
        "top5": int(bool(ranks) and min(ranks) <= 5),
        "top10": int(bool(ranks) and min(ranks) <= 10),
        "candidate_covered": int(bool(ranks))
    })

baseline_results = pd.DataFrame(results)

print("PHENOTYPE-ONLY BASELINE")
print("=" * 40)
print("Evaluated cases:", len(baseline_results))
print("Candidate coverage:",
      round(baseline_results["candidate_covered"].mean(), 4))
print("Top-1 accuracy:",
      round(baseline_results["top1"].mean(), 4))
print("Top-5 accuracy:",
      round(baseline_results["top5"].mean(), 4))
print("Top-10 accuracy:",
      round(baseline_results["top10"].mean(), 4))
print("Mean reciprocal rank:",
      round((1 / baseline_results["rank"].dropna()).mean(), 4))

out = DATA / "patient_phenotype_only_baseline.csv"
baseline_results.to_csv(out, index=False)
print("\nSaved results to:", out)

PHENOTYPE-ONLY BASELINE
Evaluated cases: 1959
Candidate coverage: 0.7917
Top-1 accuracy: 0.1138
Top-5 accuracy: 0.196
Top-10 accuracy: 0.221
Mean reciprocal rank: 0.1928

Saved results to: /content/drive/MyDrive/Rare Disease ai/processed/patient_phenotype_only_baseline.csv


In [ ]:
import pandas as pd
import numpy as np

results_file = DATA / "patient_phenotype_only_baseline.csv"
baseline_results = pd.read_csv(results_file)

covered = baseline_results[
    baseline_results["candidate_covered"] == 1
].copy()

print("BASELINE METRICS AUDIT")
print("=" * 45)

print("All test cases:", len(baseline_results))
print("Cases with diagnosis in candidate set:", len(covered))
print("Cases without diagnosis in candidate set:",
      len(baseline_results) - len(covered))

print("\nCOVERAGE-ADJUSTED METRICS (all cases)")
print("Top-1:", round(baseline_results["top1"].mean(), 4))
print("Top-5:", round(baseline_results["top5"].mean(), 4))
print("Top-10:", round(baseline_results["top10"].mean(), 4))

print("\nRANKING METRICS (covered cases only)")
if len(covered):
    print("Top-1:", round(covered["top1"].mean(), 4))
    print("Top-5:", round(covered["top5"].mean(), 4))
    print("Top-10:", round(covered["top10"].mean(), 4))
    print(
        "MRR:",
        round((1 / covered["rank"].dropna()).mean(), 4)
    )

print("\nRank summary:")
print(covered["rank"].describe().to_string())

audit_path = DATA / "patient_phenotype_baseline_metrics_audit.csv"

pd.DataFrame([
    {
        "metric_group": "all_cases",
        "cases": len(baseline_results),
        "coverage": baseline_results["candidate_covered"].mean(),
        "top1": baseline_results["top1"].mean(),
        "top5": baseline_results["top5"].mean(),
        "top10": baseline_results["top10"].mean(),
        "mrr": (1 / baseline_results["rank"].dropna()).sum()
              / len(baseline_results)
    },
    {
        "metric_group": "covered_cases",
        "cases": len(covered),
        "coverage": 1.0 if len(covered) else 0.0,
        "top1": covered["top1"].mean() if len(covered) else np.nan,
        "top5": covered["top5"].mean() if len(covered) else np.nan,
        "top10": covered["top10"].mean() if len(covered) else np.nan,
        "mrr": (1 / covered["rank"].dropna()).mean()
              if len(covered) else np.nan
    }
]).to_csv(audit_path, index=False)

print("\nSaved metrics audit to:", audit_path)

BASELINE METRICS AUDIT
All test cases: 1959
Cases with diagnosis in candidate set: 1551
Cases without diagnosis in candidate set: 408

COVERAGE-ADJUSTED METRICS (all cases)
Top-1: 0.1138
Top-5: 0.196
Top-10: 0.221

RANKING METRICS (covered cases only)
Top-1: 0.1438
Top-5: 0.2476
Top-10: 0.2792
MRR: 0.1928

Rank summary:
count     1551.000000
mean      2230.521599
std       3368.590345
min          1.000000
25%          6.000000
50%        258.000000
75%       3766.000000
max      10724.000000

Saved metrics audit to: /content/drive/MyDrive/Rare Disease ai/processed/patient_phenotype_baseline_metrics_audit.csv


In [ ]:
from pathlib import Path

hpo_file = PROJECT / "hp.obo"

print("ONTOLOGY CHECK")
print("=" * 40)
print("HPO file exists:", hpo_file.exists())
if hpo_file.exists():
    print("HPO file size (MB):",
          round(hpo_file.stat().st_size / (1024 * 1024), 2))

print("\nDISEASE PROFILE EXAMPLE")
example_disease = next(iter(disease_profiles))
print("Disease ID:", example_disease)
print("Number of HPO terms:",
      len(disease_profiles[example_disease]))
print("Example terms:",
      sorted(disease_profiles[example_disease])[:10])

print("\nPATIENT PROFILE EXAMPLE")
example_case = test_cases.iloc[0]
print("Case ID:", example_case["case_id"])
print("Phenotype count:", example_case["phenotype_count"])
print("Diagnosis IDs:", example_case["diagnosis_ids"])

ONTOLOGY CHECK
HPO file exists: True
HPO file size (MB): 10.36

DISEASE PROFILE EXAMPLE
Disease ID: DECIPHER:1
Number of HPO terms: 5
Example terms: ['HP:0000252', 'HP:0001249', 'HP:0001250', 'HP:0001252', 'HP:0001518']

PATIENT PROFILE EXAMPLE
Case ID: PMID_28239884_Family_1_proband
Phenotype count: 1
Diagnosis IDs: OMIM:148600


In [ ]:
try:
    import obonet
except ImportError:
    %pip -q install obonet
    import obonet

import networkx as nx

hpo_graph = obonet.read_obo(str(PROJECT / "hp.obo"))

print("HPO ONTOLOGY SUMMARY")
print("=" * 40)
print("Ontology terms:", len(hpo_graph.nodes))
print("Ontology edges:", len(hpo_graph.edges))

# HPO terms should normally be represented as HP: identifiers.
hpo_nodes = [n for n in hpo_graph.nodes if str(n).startswith("HP:")]
print("HP-prefixed nodes:", len(hpo_nodes))

# Inspect parent relationships for an example term.
example_term = "HP:0001250"
print("\nExample term:", example_term)
print("Term exists:", example_term in hpo_graph)

if example_term in hpo_graph:
    print("Direct parents:", list(hpo_graph.successors(example_term))[:10])

HPO ONTOLOGY SUMMARY
Ontology terms: 19894
Ontology edges: 24436
HP-prefixed nodes: 19894

Example term: HP:0001250
Term exists: True
Direct parents: ['HP:0012638']


In [ ]:
# Build a lookup of each HPO term and its ancestors.
# In this graph, successors are parents and ancestors are more general terms.

hpo_term_set = set(hpo_graph.nodes)

def get_hpo_ancestors(term):
    if term not in hpo_graph:
        return set()
    return nx.descendants(hpo_graph, term) | set(
        hpo_graph.successors(term)
    )

# Check term coverage before computing similarities.
patient_terms_all = {
    term
    for terms in test_cases["hpo_terms"].dropna()
    for term in normalize_terms(terms)
}

disease_terms_all = {
    term
    for terms in disease_profiles.values()
    for term in terms
}

print("ONTOLOGY TERM COVERAGE")
print("=" * 40)
print("Unique patient HPO terms:", len(patient_terms_all))
print("Patient terms in ontology:",
      len(patient_terms_all & hpo_term_set))
print("Unique disease HPO terms:", len(disease_terms_all))
print("Disease terms in ontology:",
      len(disease_terms_all & hpo_term_set))
print("Patient terms missing from ontology:",
      len(patient_terms_all - hpo_term_set))
print("Disease terms missing from ontology:",
      len(disease_terms_all - hpo_term_set))

ONTOLOGY TERM COVERAGE
Unique patient HPO terms: 2004
Patient terms in ontology: 2003
Unique disease HPO terms: 11611
Disease terms in ontology: 11611
Patient terms missing from ontology: 1
Disease terms missing from ontology: 0


In [ ]:
from functools import lru_cache

@lru_cache(maxsize=None)
def hpo_ancestors(term):
    if term not in hpo_graph:
        return frozenset()

    return frozenset(
        nx.descendants(hpo_graph, term)
        | set(hpo_graph.successors(term))
    )

# Cache ontology ancestors for terms used in the test cases
# and the disease profiles.
terms_to_cache = patient_terms_all | disease_terms_all

for term in terms_to_cache:
    hpo_ancestors(term)

print("ONTOLOGY CACHE SUMMARY")
print("=" * 40)
print("Terms prepared:", len(terms_to_cache))
print("Example term:", "HP:0001250")
print("Example ancestors:", sorted(hpo_ancestors("HP:0001250")))

ONTOLOGY CACHE SUMMARY
Terms prepared: 11650
Example term: HP:0001250
Example ancestors: ['HP:0000001', 'HP:0000118', 'HP:0000707', 'HP:0012638']


In [ ]:
print("Baseline saved:",
      (DATA / "patient_phenotype_only_baseline.csv").exists())
print("Filtered annotations saved:",
      (DATA / "patient_test_leakage_reduced_annotations.csv").exists())
print("Publication split saved:",
      (DATA / "patient_publication_split.csv").exists())

Baseline saved: True
Filtered annotations saved: True
Publication split saved: True


In [ ]:
import torch
from pathlib import Path

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Training-only GNN checkpoint:",
      (DATA / "rare_disease_gnn_training_only.pt").exists())
print("Existing all-disease GNN checkpoint:",
      (DATA / "rare_disease_gnn.pt").exists())
print("Saved phenotype baseline:",
      (DATA / "patient_phenotype_only_baseline.csv").exists())

CUDA available: True
GPU: Tesla T4


NameError: name 'DATA' is not defined

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not active yet. Do not start GNN training.")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Project exists:", PROJECT.exists())
print("Processed folder exists:", DATA.exists())
print("Training checkpoint exists:",
      (DATA / "rare_disease_gnn_training_only.pt").exists())
print("Publication split exists:",
      (DATA / "patient_publication_split.csv").exists())

Mounted at /content/drive
GPU available: True
GPU: Tesla T4
Project exists: True
Processed folder exists: True
Training checkpoint exists: True
Publication split exists: True


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

files_to_check = [
    DATA / "rare_disease_gnn_training_only.pt",
    DATA / "patient_publication_split.csv",
    DATA / "patient_test_leakage_reduced_annotations.csv",
    DATA / "patient_test_disease_coverage.csv",
    DATA / "patient_phenotype_only_baseline.csv",
    DATA / "patient_phenotype_baseline_metrics_audit.csv",
    DATA / "gnn_training_annotations.csv",
    DATA / "disease_level_split.csv",
    PROJECT / "hp.obo",
]

for path in files_to_check:
    if path.exists():
        print(f"FOUND | {path.name} | {path.stat().st_size / (1024**2):.2f} MB")
    else:
        print(f"MISSING | {path.name}")

print("\nModel-folder Python files:")
if MODEL.exists():
    for path in sorted(MODEL.glob("*.py")):
        print(f"FOUND | {path.name}")

FOUND | rare_disease_gnn_training_only.pt | 4.73 MB
FOUND | patient_publication_split.csv | 0.59 MB
FOUND | patient_test_leakage_reduced_annotations.csv | 33.52 MB
FOUND | patient_test_disease_coverage.csv | 0.00 MB
FOUND | patient_phenotype_only_baseline.csv | 0.12 MB
FOUND | patient_phenotype_baseline_metrics_audit.csv | 0.00 MB
FOUND | gnn_training_annotations.csv | 21.95 MB
FOUND | disease_level_split.csv | 0.16 MB
FOUND | hp.obo | 10.36 MB

Model-folder Python files:
FOUND | rare_disease_ai.py
FOUND | rare_disease_data_loader.py
FOUND | rare_disease_predictor.py


In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = [
    "patient_publication_split.csv",
    "patient_test_disease_coverage.csv",
    "patient_phenotype_only_baseline.csv",
    "patient_phenotype_baseline_metrics_audit.csv",
]

for name in files:
    path = DATA / name
    print(f"\n{'=' * 65}\n{name}")

    if not path.exists():
        print("FILE MISSING")
        continue

    try:
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(5).to_string(index=False))
    except Exception as e:
        print("Read error:", str(e))

print("\n" + "=" * 65)
print("Checkpoint file size:",
      f"{(DATA / 'rare_disease_gnn_training_only.pt').stat().st_size / (1024**2):.2f} MB")|

SyntaxError: invalid syntax (2139282329.py, line 31)

In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

files = [
    "patient_publication_split.csv",
    "patient_test_disease_coverage.csv",
    "patient_phenotype_only_baseline.csv",
    "patient_phenotype_baseline_metrics_audit.csv",
]

for name in files:
    path = DATA / name
    print(f"\n{'=' * 60}")
    print(name)

    if not path.exists():
        print("FILE MISSING")
        continue

    try:
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(5).to_string(index=False))
    except Exception as e:
        print("Read error:", str(e))

print("\n" + "=" * 60)

checkpoint = DATA / "rare_disease_gnn_training_only.pt"

if checkpoint.exists():
    size_mb = checkpoint.stat().st_size / (1024 ** 2)
    print(f"Checkpoint file size: {size_mb:.2f} MB")
else:
    print("Checkpoint file missing")


patient_publication_split.csv
Shape: (10345, 4)
Columns: ['case_id', 'pmid', 'split', 'diagnosis_ids']
                    case_id     pmid       split diagnosis_ids
PMID_19836009_Family_E_II_6 19836009 development   OMIM:277600
PMID_19836009_Family_E_II_3 19836009 development   OMIM:277600
PMID_19836009_Family_B_II_4 19836009 development   OMIM:277600
PMID_19836009_Family_B_II_2 19836009 development   OMIM:277600
      PMID_25469541_patient 25469541 development   OMIM:277600

patient_test_disease_coverage.csv
Shape: (1, 5)
Columns: ['test_cases', 'cases_with_diagnosis_profile', 'cases_without_diagnosis_profile', 'diseases_with_profiles', 'annotation_rows_after_filter']
 test_cases  cases_with_diagnosis_profile  cases_without_diagnosis_profile  diseases_with_profiles  annotation_rows_after_filter
       1959                          1551                              408                   12827                        280733

patient_phenotype_only_baseline.csv
Shape: (1959, 8)
Columns:

In [ ]:
from pathlib import Path
import torch

from google.colab import drive
drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

print("Project exists:", PROJECT.exists())
print("Processed folder exists:", DATA.exists())
print("Model folder exists:", MODEL.exists())

for name in [
    "patient_publication_split.csv",
    "patient_test_disease_coverage.csv",
    "patient_phenotype_only_baseline.csv",
    "patient_phenotype_baseline_metrics_audit.csv",
    "patient_test_leakage_reduced_annotations.csv",
    "rare_disease_gnn_training_only.pt",
]:
    path = DATA / name
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")

print("\nPyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project exists: True
Processed folder exists: True
Model folder exists: True
patient_publication_split.csv: FOUND
patient_test_disease_coverage.csv: FOUND
patient_phenotype_only_baseline.csv: FOUND
patient_phenotype_baseline_metrics_audit.csv: FOUND
patient_test_leakage_reduced_annotations.csv: FOUND
rare_disease_gnn_training_only.pt: FOUND

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

print("=== Saved model files ===")
for folder in [MODEL, DATA]:
    print(f"\n{folder}")
    for p in sorted(folder.glob("*")):
        if p.is_file() and (
            p.suffix in {".pt", ".pth", ".py", ".json", ".yaml", ".yml"}
        ):
            print(f"{p.name:45} {p.stat().st_size / (1024**2):8.2f} MB")

print("\n=== Training annotations ===")
for name in [
    "gnn_training_annotations.csv",
    "clean_hpo_annotations.csv",
    "disease_level_split.csv",
]:
    p = DATA / name
    if p.exists():
        df = pd.read_csv(p, nrows=5)
        print(f"\n{name}")
        print("Columns:", df.columns.tolist())
        print(df.head(2).to_string(index=False))

print("\n=== Runtime ===")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GB):", round(
    torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
))

=== Saved model files ===

/content/drive/MyDrive/Rare Disease ai/model
prediction_config.json                            0.00 MB
rare_disease_ai.py                                0.00 MB
rare_disease_data_loader.py                       0.00 MB
rare_disease_predictor.py                         0.00 MB

/content/drive/MyDrive/Rare Disease ai/processed
rare_disease_gnn.pt                               6.69 MB
rare_disease_gnn_training_only.pt                 4.73 MB

=== Training annotations ===

gnn_training_annotations.csv
Columns: ['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']
database_id                                  disease_name  qualifier     hpo_id     reference evidence  onset frequency  sex  modifier aspect               biocuration
OMIM:619340 Developmental and epileptic encephalopathy 96        NaN HP:0011097 PMID:31675180      PCS    NaN       1/2  NaN       NaN      P HPO:p

In [ ]:
from pathlib import Path
import torch

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
checkpoint_path = DATA / "rare_disease_gnn_training_only.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("Checkpoint type:", type(checkpoint).__name__)

if isinstance(checkpoint, dict):
    print("Top-level keys:", list(checkpoint.keys()))

    for key, value in checkpoint.items():
        if isinstance(value, dict):
            print(f"\n{key}: dictionary with {len(value)} entries")
            for name, tensor in list(value.items())[:12]:
                if torch.is_tensor(tensor):
                    print(f"  {name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}")
                else:
                    print(f"  {name}: {type(tensor).__name__}")
        elif torch.is_tensor(value):
            print(f"{key}: tensor shape={tuple(value.shape)}")
        else:
            print(f"{key}: {type(value).__name__} = {str(value)[:150]}")
else:
    print("Unexpected checkpoint format.")

Checkpoint type: dict
Top-level keys: ['model_state_dict', 'num_nodes', 'hidden_dim', 'disease_ids', 'phenotype_ids', 'disease_to_idx', 'phenotype_to_idx', 'training_disease_count', 'training_association_count', 'validation_loss', 'seed']

model_state_dict: dictionary with 5 entries
  embedding.weight: shape=(17322, 64), dtype=torch.float32
  conv1.bias: shape=(64,), dtype=torch.float32
  conv1.lin.weight: shape=(64, 64), dtype=torch.float32
  conv2.bias: shape=(64,), dtype=torch.float32
  conv2.lin.weight: shape=(64, 64), dtype=torch.float32
num_nodes: int = 17322
hidden_dim: int = 64
disease_ids: list = ['OMIM:100300', 'OMIM:100800', 'OMIM:101000', 'OMIM:101200', 'OMIM:101400', 'OMIM:101600', 'OMIM:101800', 'OMIM:102200', 'OMIM:102370', 'OMIM:102500',
phenotype_ids: list = ['HP:0000002', 'HP:0000003', 'HP:0000006', 'HP:0000007', 'HP:0000008', 'HP:0000009', 'HP:0000010', 'HP:0000011', 'HP:0000012', 'HP:0000013', 'HP:00000

disease_to_idx: dictionary with 7340 entries
  OMIM:100300: in

In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")

for folder in [PROJECT / "model", PROJECT]:
    print(f"\n{'=' * 65}")
    print("Folder:", folder)

    for path in sorted(folder.glob("*.py")):
        print(f"\n--- {path.relative_to(PROJECT)} ---")
        lines = path.read_text(encoding="utf-8").splitlines()

        # Show model architecture, training, graph construction,
        # and prediction code without dumping unrelated files.
        keywords = (
            "class ", "def ", "GCNConv", "GATConv", "forward(",
            "optimizer", "loss", "train", "edge_index",
            "model_state_dict", "predict", "validation"
        )
        selected = [
            (i, line) for i, line in enumerate(lines)
            if any(word.lower() in line.lower() for word in keywords)
        ]

        if not selected:
            print("(No matching model-related lines)")
        else:
            for i, line in selected[:100]:
                print(f"{i + 1:4}: {line[:180]}")


Folder: /content/drive/MyDrive/Rare Disease ai/model

--- model/rare_disease_ai.py ---
   2: from rare_disease_data_loader import load_prediction_data
   4: _DATA = load_prediction_data()
  10: def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,

--- model/rare_disease_data_loader.py ---
  10: def load_prediction_data():
  40:     train_ids = set(
  41:         split.loc[split["split"].str.lower() == "train", "disease_id"]
  45:     for disease in train_ids:
  49:     n_train = max(len(train_ids), 1)
  51:         term: -np.log(count / n_train)

--- model/rare_disease_predictor.py ---
   1: """Rare Disease Prediction AI — research prototype.
  10: def predict_diseases(patient_hpo_terms, patient_genes=None, top_k=10,
  73: def explain_predictions(patient_hpo_terms, patient_genes=None, top_k=10):
  74:     results = predict_diseases(

Folder: /content/drive/MyDrive/Rare Disease ai


In [ ]:
from pathlib import Path

MODEL = Path("/content/drive/MyDrive/Rare Disease ai/model")

for filename in [
    "rare_disease_data_loader.py",
    "rare_disease_predictor.py",
    "rare_disease_ai.py",
]:
    path = MODEL / filename

    print("\n" + "=" * 70)
    print(filename)
    print("=" * 70)

    if path.exists():
        print(path.read_text(encoding="utf-8"))
    else:
        print("FILE NOT FOUND")


rare_disease_data_loader.py

from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

def load_prediction_data():
    ann = pd.read_csv(
        DATA / "clean_hpo_annotations.csv",
        low_memory=False,
        dtype={"database_id": "string", "hpo_id": "string"}
    ).dropna(subset=["database_id", "hpo_id"])

    ann["database_id"] = ann["database_id"].str.strip()
    ann["hpo_id"] = ann["hpo_id"].str.strip()

    phenotype_map = ann.groupby("database_id")["hpo_id"].apply(set).to_dict()

    gene_df = pd.read_csv(
        DATA / "normalized_disease_gene_mapping.csv",
        dtype=str
    ).fillna("")

    gene_map = {
        row["disease_id"]: {
            g.strip().upper()
            for g in row["gene_symbols"].split(";")
            if g.strip()
        }
        for _, row in gene_df.iterrows()
    }

    split = pd.read_csv(
        DATA / "di

In [ ]:
from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
MODEL = PROJECT / "model"

# Load saved checkpoint on CPU
ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

# Compare checkpoint disease IDs with the saved split
split = pd.read_csv(DATA / "disease_level_split.csv", dtype=str)
train_ids = set(
    split.loc[split["split"].str.lower() == "train", "disease_id"].dropna()
)
checkpoint_ids = set(ckpt["disease_ids"])

print("=== Checkpoint split verification ===")
print("Diseases in saved training split:", len(train_ids))
print("Diseases in checkpoint index:", len(checkpoint_ids))
print("Exact match:", train_ids == checkpoint_ids)
print("Missing from checkpoint:", len(train_ids - checkpoint_ids))
print("Unexpected in checkpoint:", len(checkpoint_ids - train_ids))

# Compare annotation files
print("\n=== Annotation comparison ===")
for filename in ["clean_hpo_annotations.csv", "gnn_training_annotations.csv"]:
    path = DATA / filename
    df = pd.read_csv(path, usecols=["database_id", "hpo_id"], dtype=str)
    pairs = set(
        zip(
            df["database_id"].dropna().str.strip(),
            df["hpo_id"].dropna().str.strip()
        )
    )
    print(filename, "| rows:", len(df), "| unique disease-term pairs:", len(pairs))
    if filename == "clean_hpo_annotations.csv":
        clean_pairs = pairs
    else:
        training_pairs = pairs

print("Training pairs also in clean annotations:", len(training_pairs & clean_pairs))
print("Training pairs absent from clean annotations:", len(training_pairs - clean_pairs))

# Locate scripts that may contain the actual GNN training loop
print("\n=== Possible training scripts ===")
for path in PROJECT.rglob("*.py"):
    if any(word in path.name.lower() for word in ["train", "gnn", "graph", "model"]):
        print(path.relative_to(PROJECT))

=== Checkpoint split verification ===
Diseases in saved training split: 7340
Diseases in checkpoint index: 7340
Exact match: True
Missing from checkpoint: 0
Unexpected in checkpoint: 0

=== Annotation comparison ===
clean_hpo_annotations.csv | rows: 286047 | unique disease-term pairs: 286047
gnn_training_annotations.csv | rows: 179653 | unique disease-term pairs: 179653
Training pairs also in clean annotations: 179653
Training pairs absent from clean annotations: 0

=== Possible training scripts ===


In [ ]:
from pathlib import Path
import pandas as pd
import torch

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

graph_files = [
    "rare_disease_training_graph.graphml",
    "rare_disease_knowledge_graph.graphml",
    "rare_disease_enriched_graph.graphml",
]

for name in graph_files:
    path = DATA / name
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}",
          f"({path.stat().st_size / (1024**2):.2f} MB)" if path.exists() else "")

print("\n=== Training checkpoint metadata ===")
ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

print("Nodes:", ckpt["num_nodes"])
print("Hidden dimension:", ckpt["hidden_dim"])
print("Training diseases:", ckpt["training_disease_count"])
print("Training associations:", ckpt["training_association_count"])
print("Validation loss:", ckpt["validation_loss"])
print("Seed:", ckpt["seed"])

print("\n=== Training annotation index coverage ===")
ann = pd.read_csv(
    DATA / "gnn_training_annotations.csv",
    usecols=["database_id", "hpo_id"],
    dtype=str
).dropna()

disease_ids = set(ann["database_id"].str.strip())
phenotype_ids = set(ann["hpo_id"].str.strip())

print("Annotation diseases:", len(disease_ids))
print("Annotation phenotypes:", len(phenotype_ids))
print("Diseases outside checkpoint index:",
      len(disease_ids - set(ckpt["disease_ids"])))
print("Phenotypes outside checkpoint index:",
      len(phenotype_ids - set(ckpt["phenotype_ids"])))

rare_disease_training_graph.graphml: FOUND (20.36 MB)
rare_disease_knowledge_graph.graphml: FOUND (29.03 MB)
rare_disease_enriched_graph.graphml: FOUND (31.06 MB)

=== Training checkpoint metadata ===
Nodes: 17322
Hidden dimension: 64
Training diseases: 7340
Training associations: 179653
Validation loss: 0.57393878698349
Seed: 42

=== Training annotation index coverage ===
Annotation diseases: 7340
Annotation phenotypes: 9982
Diseases outside checkpoint index: 0
Phenotypes outside checkpoint index: 0


In [ ]:
from pathlib import Path
import networkx as nx
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

# Load only the training graph
graph_path = DATA / "rare_disease_training_graph.graphml"
G = nx.read_graphml(graph_path)

print("Graph nodes:", G.number_of_nodes())
print("Graph edges:", G.number_of_edges())

# Inspect node and edge attributes
print("\nSample nodes:")
for node, attrs in list(G.nodes(data=True))[:5]:
    print(node, attrs)

print("\nSample edges:")
for u, v, attrs in list(G.edges(data=True))[:5]:
    print(u, "->", v, attrs)

# Compare expected training associations with graph edges
ann = pd.read_csv(
    DATA / "gnn_training_annotations.csv",
    usecols=["database_id", "hpo_id"],
    dtype=str
).dropna()

expected = set(zip(
    ann["database_id"].str.strip(),
    ann["hpo_id"].str.strip()
))

actual = set()
for u, v in G.edges():
    u, v = str(u), str(v)
    if u.startswith(("OMIM:", "ORPHA:", "DECIPHER:")) and v.startswith("HP:"):
        actual.add((u, v))
    elif v.startswith(("OMIM:", "ORPHA:", "DECIPHER:")) and u.startswith("HP:"):
        actual.add((v, u))

print("\n=== Association audit ===")
print("Expected training pairs:", len(expected))
print("Pairs found in graph:", len(actual))
print("Expected pairs missing from graph:", len(expected - actual))
print("Graph pairs absent from training annotations:", len(actual - expected))

Graph nodes: 24279
Graph edges: 189775

Sample nodes:
OMIM:619340 {'node_type': 'disease', 'name': 'Developmental and epileptic encephalopathy 96'}
OMIM:614102 {'node_type': 'disease', 'name': 'Immunoglobulin kappa light chain deficiency'}
OMIM:619426 {'node_type': 'disease', 'name': 'White-Kernohan syndrome'}
OMIM:610370 {'node_type': 'disease', 'name': 'Diarrhea 4, malabsorptive, congenital'}
OMIM:609621 {'node_type': 'disease', 'name': 'Short QT syndrome 2'}

Sample edges:
OMIM:619340 -> HP:0011097 {'relation': 'associated_with'}
OMIM:619340 -> HP:0002187 {'relation': 'associated_with'}
OMIM:619340 -> HP:0001518 {'relation': 'associated_with'}
OMIM:619340 -> HP:0032792 {'relation': 'associated_with'}
OMIM:619340 -> HP:0011451 {'relation': 'associated_with'}

=== Association audit ===
Expected training pairs: 179653
Pairs found in graph: 179653
Expected pairs missing from graph: 0
Graph pairs absent from training annotations: 0


In [ ]:
from pathlib import Path
from collections import Counter
import networkx as nx
import pandas as pd

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

G = nx.read_graphml(DATA / "rare_disease_training_graph.graphml")

print("=== Node types ===")
node_types = Counter(
    attrs.get("node_type", "UNSPECIFIED")
    for _, attrs in G.nodes(data=True)
)
for node_type, count in node_types.most_common():
    print(f"{node_type}: {count}")

print("\n=== Edge relations ===")
relations = Counter(
    attrs.get("relation", "UNSPECIFIED")
    for _, _, attrs in G.edges(data=True)
)
for relation, count in relations.most_common():
    print(f"{relation}: {count}")

print("\n=== Edge-type audit ===")
edge_types = Counter()
for u, v, attrs in G.edges(data=True):
    type_u = G.nodes[u].get("node_type", "UNSPECIFIED")
    type_v = G.nodes[v].get("node_type", "UNSPECIFIED")
    relation = attrs.get("relation", "UNSPECIFIED")
    edge_types[(type_u, relation, type_v)] += 1

for (type_u, relation, type_v), count in edge_types.most_common(25):
    print(f"{type_u} --{relation}--> {type_v}: {count}")

print("\n=== Example extra nodes ===")
for node, attrs in G.nodes(data=True):
    if attrs.get("node_type", "UNSPECIFIED") not in {"disease", "phenotype"}:
        print(node, attrs)
        if sum(
            1 for _, a in G.nodes(data=True)
            if a.get("node_type", "UNSPECIFIED") not in {"disease", "phenotype"}
        ) >= 10:
            break

=== Node types ===
phenotype: 11658
disease: 7340
gene: 5281

=== Edge relations ===
associated_with: 179653
disease_gene: 10122

=== Edge-type audit ===
disease --associated_with--> phenotype: 179653
disease --disease_gene--> gene: 10122

=== Example extra nodes ===
NCBIGene:64170 {'node_type': 'gene', 'gene_symbol': 'CARD9'}


In [ ]:
from pathlib import Path
import pandas as pd
import networkx as nx

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

G = nx.read_graphml(DATA / "rare_disease_training_graph.graphml")

split = pd.read_csv(DATA / "disease_level_split.csv", dtype=str)
train_ids = set(
    split.loc[split["split"].str.lower() == "train", "disease_id"].dropna()
)
test_ids = set(
    split.loc[split["split"].str.lower() == "test", "disease_id"].dropna()
)

gene_edges = []
for u, v, attrs in G.edges(data=True):
    if attrs.get("relation") == "disease_gene":
        disease = u if G.nodes[u].get("node_type") == "disease" else v
        gene = v if disease == u else u
        gene_edges.append((disease, gene))

edge_diseases = {d for d, _ in gene_edges}

print("Disease-gene edges:", len(gene_edges))
print("Unique diseases with gene edges:", len(edge_diseases))
print("Edges involving training diseases:",
      sum(d in train_ids for d, _ in gene_edges))
print("Edges involving held-out diseases:",
      sum(d in test_ids for d, _ in gene_edges))
print("Diseases outside saved split:", len(edge_diseases - train_ids - test_ids))

print("\n=== Candidate gene mapping files ===")
for p in sorted(DATA.glob("*gene*mapping*.csv")):
    print(p.name, f"({p.stat().st_size / (1024**2):.2f} MB)")
    df = pd.read_csv(p, nrows=3, dtype=str)
    print("Columns:", df.columns.tolist())
    print(df.to_string(index=False))

Disease-gene edges: 10122
Unique diseases with gene edges: 7340
Edges involving training diseases: 10122
Edges involving held-out diseases: 0
Diseases outside saved split: 0

=== Candidate gene mapping files ===
normalized_disease_gene_mapping.csv (0.22 MB)
Columns: ['disease_id', 'gene_symbols']
 disease_id gene_symbols
OMIM:100100        CHRM3
OMIM:100300     ARHGAP31
OMIM:100800        FGFR3


In [ ]:
import importlib.util
import json
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
MODEL = PROJECT / "model"

print("=== Graph libraries ===")
for package in ["torch", "torch_geometric", "networkx", "sklearn"]:
    spec = importlib.util.find_spec(package)
    print(f"{package}: {'installed' if spec else 'not installed'}")

print("\n=== Prediction configuration ===")
config_path = MODEL / "prediction_config.json"

if config_path.exists():
    print(config_path.read_text(encoding="utf-8"))
else:
    print("Configuration file not found")

print("\n=== Project Python files ===")
for path in sorted(PROJECT.rglob("*.py")):
    print(path.relative_to(PROJECT))

=== Graph libraries ===
torch: installed
torch_geometric: not installed
networkx: installed
sklearn: installed

=== Prediction configuration ===
{
  "model_name": "Rare Disease Phenotype Ranking Prototype",
  "version": "1.0",
  "phenotype_scoring": "information_content_weighted_f1",
  "gene_weight": 0.25,
  "phenotype_annotations": "/content/drive/MyDrive/Rare Disease ai/processed/clean_hpo_annotations.csv",
  "gene_mapping": "/content/drive/MyDrive/Rare Disease ai/processed/normalized_disease_gene_mapping.csv",
  "disease_split": "/content/drive/MyDrive/Rare Disease ai/processed/disease_level_split.csv",
  "clinical_validation": "not_yet_performed",
  "scores_are_probabilities": false
}

=== Project Python files ===
model/rare_disease_ai.py
model/rare_disease_data_loader.py
model/rare_disease_predictor.py


In [ ]:
%pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 43.8 MB/s eta 0:00:00


In [ ]:
import torch
import torch_geometric
from torch_geometric.nn import GCNConv

print("PyTorch:", torch.__version__)
print("PyTorch Geometric:", torch_geometric.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

layer = GCNConv(64, 64)
print("GCNConv import and initialization: OK")

PyTorch: 2.11.0+cu128
PyTorch Geometric: 2.8.1
CUDA available: True
GPU: Tesla T4
GCNConv import and initialization: OK


In [ ]:
from pathlib import Path
import torch
from torch_geometric.nn import GCNConv

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

class RareDiseaseGCN(torch.nn.Module):
    def __init__(self, num_nodes, hidden_dim=64):
        super().__init__()
        self.embedding = torch.nn.Embedding(num_nodes, hidden_dim)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

    def forward(self, edge_index):
        x = self.embedding.weight
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x

model = RareDiseaseGCN(
    num_nodes=ckpt["num_nodes"],
    hidden_dim=ckpt["hidden_dim"]
)

model.load_state_dict(ckpt["model_state_dict"], strict=True)
model = model.to(device).eval()

print("Checkpoint loaded successfully.")
print("Parameter count:", sum(p.numel() for p in model.parameters()))
print("Device:", device)
print("Disease index:", len(ckpt["disease_to_idx"]))
print("Phenotype index:", len(ckpt["phenotype_to_idx"]))
print("Architecture:", model)

Checkpoint loaded successfully.
Parameter count: 1116928
Device: cuda
Disease index: 7340
Phenotype index: 9982
Architecture: RareDiseaseGCN(
  (embedding): Embedding(17322, 64)
  (conv1): GCNConv(64, 64)
  (conv2): GCNConv(64, 64)
)


In [ ]:
from pathlib import Path
import pandas as pd
import torch

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

disease_to_idx = ckpt["disease_to_idx"]
phenotype_to_idx = ckpt["phenotype_to_idx"]

# Verify the two index spaces are contiguous and non-overlapping.
disease_indices = sorted(disease_to_idx.values())
phenotype_indices = sorted(phenotype_to_idx.values())

print("Disease index range:", min(disease_indices), max(disease_indices))
print("Phenotype index range:", min(phenotype_indices), max(phenotype_indices))
print("Disease indexes contiguous:",
      disease_indices == list(range(len(disease_to_idx))))
print("Phenotype indexes contiguous:",
      phenotype_indices == list(range(len(disease_to_idx),
                                      len(disease_to_idx) + len(phenotype_to_idx))))

# Load only the training associations.
ann = pd.read_csv(
    DATA / "gnn_training_annotations.csv",
    usecols=["database_id", "hpo_id"],
    dtype=str
).dropna()

ann["database_id"] = ann["database_id"].str.strip()
ann["hpo_id"] = ann["hpo_id"].str.strip()
ann = ann.drop_duplicates(["database_id", "hpo_id"])

# Map associations using the saved checkpoint's exact indexes.
pairs = [
    (disease_to_idx[d], phenotype_to_idx[h])
    for d, h in zip(ann["database_id"], ann["hpo_id"])
    if d in disease_to_idx and h in phenotype_to_idx
]

edge_pairs = torch.tensor(pairs, dtype=torch.long).T.contiguous()

# GCN message passing over both directions of each association.
edge_index = torch.cat([edge_pairs, edge_pairs.flip(0)], dim=1).to(device)

print("\n=== Reconstructed graph ===")
print("Unique training associations mapped:", len(pairs))
print("Expected checkpoint associations:", ckpt["training_association_count"])
print("Edge index shape:", tuple(edge_index.shape))
print("Bidirectional edges:", edge_index.size(1))
print("All indexes in range:",
      int(edge_index.min()) >= 0 and
      int(edge_index.max()) < ckpt["num_nodes"])

Disease index range: 0 7339
Phenotype index range: 7340 17321
Disease indexes contiguous: True
Phenotype indexes contiguous: True

=== Reconstructed graph ===
Unique training associations mapped: 179653
Expected checkpoint associations: 179653
Edge index shape: (2, 359306)
Bidirectional edges: 359306
All indexes in range: True


In [ ]:
import torch

model.eval()

with torch.no_grad():
    embeddings = model(edge_index)

print("Embedding shape:", tuple(embeddings.shape))
print("Expected shape:", (ckpt["num_nodes"], ckpt["hidden_dim"]))
print("All embeddings finite:", bool(torch.isfinite(embeddings).all()))
print("Embedding device:", embeddings.device)

disease_embeddings = embeddings[:len(disease_to_idx)]
phenotype_embeddings = embeddings[
    len(disease_to_idx):
    len(disease_to_idx) + len(phenotype_to_idx)
]

print("Disease embeddings:", tuple(disease_embeddings.shape))
print("Phenotype embeddings:", tuple(phenotype_embeddings.shape))

Embedding shape: (17322, 64)
Expected shape: (17322, 64)
All embeddings finite: True
Embedding device: cuda:0
Disease embeddings: (7340, 64)
Phenotype embeddings: (9982, 64)


In [ ]:
from pathlib import Path
import pandas as pd
import json

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

cases_path = PROJECT / "all_phenopackets"
split_path = DATA / "patient_publication_split.csv"

print("Test split exists:", split_path.exists())
print("Phenopacket directory exists:", cases_path.exists())

split = pd.read_csv(split_path, dtype=str)
test_cases = split.loc[split["split"] == "test"].copy()

print("Publication-held-out test cases:", len(test_cases))
print("Unique test publications:", test_cases["pmid"].nunique())
print("Checkpoint phenotype vocabulary:", len(phenotype_to_idx))
print("\nTest split columns:", test_cases.columns.tolist())
print(test_cases.head(3).to_string(index=False))

Test split exists: True
Phenopacket directory exists: True
Publication-held-out test cases: 1959
Unique test publications: 346
Checkpoint phenotype vocabulary: 9982

Test split columns: ['case_id', 'pmid', 'split', 'diagnosis_ids']
                       case_id     pmid split diagnosis_ids
PMID_28239884_Family_1_proband 28239884  test   OMIM:148600
PMID_28239884_Family_2_proband 28239884  test   OMIM:148600
PMID_28239884_Family_3_proband 28239884  test   OMIM:148600


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
PACKETS = PROJECT / "all_phenopackets"

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
test_cases = split.loc[split["split"] == "test"].copy()

# Index JSON files by filename for direct lookup.
json_files = {p.name: p for p in PACKETS.glob("*.json")}
print("JSON files indexed:", len(json_files))

def extract_hpo_terms(obj):
    terms = set()

    def walk(x):
        if isinstance(x, dict):
            term = x.get("id")
            if isinstance(term, str) and term.startswith("HP:"):
                terms.add(term)
            for value in x.values():
                walk(value)
        elif isinstance(x, list):
            for value in x:
                walk(value)

    walk(obj)
    return terms

case_terms = {}
missing_files = []

for case_id in test_cases["case_id"]:
    path = json_files.get(f"{case_id}.json")
    if path is None:
        missing_files.append(case_id)
        continue

    with path.open(encoding="utf-8") as f:
        case_terms[case_id] = extract_hpo_terms(json.load(f))

known_terms = set(phenotype_to_idx)

coverage_rows = []
for row in test_cases.itertuples(index=False):
    terms = case_terms.get(row.case_id, set())
    recognized = terms & known_terms
    coverage_rows.append({
        "case_id": row.case_id,
        "pmid": row.pmid,
        "diagnosis_ids": row.diagnosis_ids,
        "n_patient_terms": len(terms),
        "n_recognized_terms": len(recognized),
        "term_coverage": len(recognized) / len(terms) if terms else 0.0,
    })

coverage = pd.DataFrame(coverage_rows)

print("\nTest cases:", len(test_cases))
print("Cases with missing JSON files:", len(missing_files))
print("Cases with at least one phenotype:", int((coverage["n_patient_terms"] > 0).sum()))
print("Cases with at least one recognized term:", int((coverage["n_recognized_terms"] > 0).sum()))
print("Mean per-case HPO vocabulary coverage:",
      round(coverage.loc[coverage["n_patient_terms"] > 0, "term_coverage"].mean(), 4))
print("Median per-case HPO vocabulary coverage:",
      round(coverage.loc[coverage["n_patient_terms"] > 0, "term_coverage"].median(), 4))

print("\nCoverage summary:")
print(coverage[["n_patient_terms", "n_recognized_terms", "term_coverage"]].describe().round(4))

JSON files indexed: 0

Test cases: 1959
Cases with missing JSON files: 1959
Cases with at least one phenotype: 0
Cases with at least one recognized term: 0
Mean per-case HPO vocabulary coverage: nan
Median per-case HPO vocabulary coverage: nan

Coverage summary:
       n_patient_terms  n_recognized_terms  term_coverage
count           1959.0              1959.0         1959.0
mean               0.0                 0.0            0.0
std                0.0                 0.0            0.0
min                0.0                 0.0            0.0
25%                0.0                 0.0            0.0
50%                0.0                 0.0            0.0
75%                0.0                 0.0            0.0
max                0.0                 0.0            0.0


In [ ]:
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
PACKETS = PROJECT / "all_phenopackets"

print("Phenopacket directory exists:", PACKETS.exists())
print("Directory path:", PACKETS)

if PACKETS.exists():
    print("\nImmediate contents:")
    for p in list(PACKETS.iterdir())[:20]:
        print("DIR " if p.is_dir() else "FILE", p.name)

    print("\nFile counts:")
    print("JSON files directly inside:", sum(1 for _ in PACKETS.glob("*.json")))
    print("JSON files recursively:", sum(1 for _ in PACKETS.rglob("*.json")))
    print("TSV files recursively:", sum(1 for _ in PACKETS.rglob("*.tsv")))

Phenopacket directory exists: True
Directory path: /content/drive/MyDrive/Rare Disease ai/all_phenopackets

Immediate contents:
DIR  0.1.27

File counts:
JSON files directly inside: 0
JSON files recursively: 10427
TSV files recursively: 1


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
PACKETS = PROJECT / "all_phenopackets"

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
test_cases = split.loc[split["split"] == "test"].copy()

# Search recursively inside version subfolders.
json_files = {p.stem: p for p in PACKETS.rglob("*.json")}
print("JSON files indexed:", len(json_files))

def extract_hpo_terms(obj):
    terms = set()

    def walk(x):
        if isinstance(x, dict):
            term = x.get("id")
            if isinstance(term, str) and term.startswith("HP:"):
                terms.add(term)
            for value in x.values():
                walk(value)
        elif isinstance(x, list):
            for value in x:
                walk(value)

    walk(obj)
    return terms

known_terms = set(phenotype_to_idx)
coverage_rows = []
missing_files = []

for row in test_cases.itertuples(index=False):
    path = json_files.get(row.case_id)

    if path is None:
        missing_files.append(row.case_id)
        terms = set()
    else:
        with path.open(encoding="utf-8") as f:
            terms = extract_hpo_terms(json.load(f))

    recognized = terms & known_terms
    coverage_rows.append({
        "case_id": row.case_id,
        "pmid": row.pmid,
        "diagnosis_ids": row.diagnosis_ids,
        "n_patient_terms": len(terms),
        "n_recognized_terms": len(recognized),
        "term_coverage": len(recognized) / len(terms) if terms else 0.0,
    })

coverage = pd.DataFrame(coverage_rows)

print("\nTest cases:", len(test_cases))
print("Missing JSON files:", len(missing_files))
print("Cases with phenotype terms:", int((coverage["n_patient_terms"] > 0).sum()))
print("Cases with recognized terms:", int((coverage["n_recognized_terms"] > 0).sum()))
valid = coverage.loc[coverage["n_patient_terms"] > 0, "term_coverage"]
print("Mean HPO vocabulary coverage:", round(valid.mean(), 4) if len(valid) else "N/A")
print("Median HPO vocabulary coverage:", round(valid.median(), 4) if len(valid) else "N/A")

print("\nCoverage summary:")
print(coverage[["n_patient_terms", "n_recognized_terms", "term_coverage"]].describe().round(4))

JSON files indexed: 10427

Test cases: 1959
Missing JSON files: 0
Cases with phenotype terms: 1959
Cases with recognized terms: 1949
Mean HPO vocabulary coverage: 0.9904
Median HPO vocabulary coverage: 1.0

Coverage summary:
       n_patient_terms  n_recognized_terms  term_coverage
count        1959.0000           1959.0000      1959.0000
mean           23.7989             23.7147         0.9904
std            19.6747             19.6504         0.0777
min             1.0000              0.0000         0.0000
25%            10.0000             10.0000         1.0000
50%            18.0000             18.0000         1.0000
75%            31.0000             31.0000         1.0000
max           105.0000            105.0000         1.0000


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
PACKETS = PROJECT / "all_phenopackets"

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
test_case = split.loc[split["split"] == "test"].iloc[0]

json_path = next(
    p for p in PACKETS.rglob("*.json")
    if p.stem == test_case["case_id"]
)

with json_path.open(encoding="utf-8") as f:
    packet = json.load(f)

print("Case ID:", test_case["case_id"])
print("Expected diagnosis:", test_case["diagnosis_ids"])
print("JSON file:", json_path)
print("\nTop-level keys:", list(packet.keys()))

print("\nPhenotype-related structure:")
for key in ["phenotypicFeatures", "subject", "diseases", "interpretations"]:
    if key in packet:
        value = packet[key]
        print(f"\n{key}:")
        print(json.dumps(value[:3] if isinstance(value, list) else value,
                         indent=2)[:4000])

Case ID: PMID_28239884_Family_1_proband
Expected diagnosis: OMIM:148600
JSON file: /content/drive/MyDrive/Rare Disease ai/all_phenopackets/0.1.27/AAGAB/PMID_28239884_Family_1_proband.json

Top-level keys: ['id', 'subject', 'phenotypicFeatures', 'interpretations', 'diseases', 'metaData']

Phenotype-related structure:

phenotypicFeatures:
[
  {
    "type": {
      "id": "HP:0006482",
      "label": "Abnormal dental morphology"
    },
    "excluded": true
  },
  {
    "type": {
      "id": "HP:0007530",
      "label": "Punctate palmoplantar hyperkeratosis"
    },
    "onset": {
      "age": {
        "iso8601duration": "P18Y"
      }
    }
  },
  {
    "type": {
      "id": "HP:0001595",
      "label": "Abnormal hair morphology"
    },
    "excluded": true
  }
]

subject:
{
  "id": "Family 1 proband",
  "timeAtLastEncounter": {
    "age": {
      "iso8601duration": "P43Y"
    }
  },
  "sex": "FEMALE"
}

diseases:
[
  {
    "term": {
      "id": "OMIM:148600",
      "label": "Keratoderma, 

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
PACKETS = PROJECT / "all_phenopackets"

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
test_cases = split.loc[split["split"] == "test"].copy()

json_files = {p.stem: p for p in PACKETS.rglob("*.json")}

rows = []
missing = []

for row in test_cases.itertuples(index=False):
    path = json_files.get(row.case_id)
    if path is None:
        missing.append(row.case_id)
        continue

    with path.open(encoding="utf-8") as f:
        packet = json.load(f)

    positive, excluded = set(), set()

    for feature in packet.get("phenotypicFeatures", []):
        term = feature.get("type", {}).get("id")
        if not isinstance(term, str) or not term.startswith("HP:"):
            continue

        if feature.get("excluded") is True:
            excluded.add(term)
        else:
            positive.add(term)

    # Excluded and positive cannot both represent present findings.
    positive -= excluded

    rows.append({
        "case_id": row.case_id,
        "pmid": row.pmid,
        "diagnosis_ids": row.diagnosis_ids,
        "positive_hpo_terms": sorted(positive),
        "excluded_hpo_terms": sorted(excluded),
        "n_positive": len(positive),
        "n_excluded": len(excluded),
    })

patient_features = pd.DataFrame(rows)

print("Test cases in split:", len(test_cases))
print("Cases extracted:", len(patient_features))
print("Missing files:", len(missing))
print("Cases with positive phenotypes:",
      int((patient_features["n_positive"] > 0).sum()))
print("Total positive observations:",
      int(patient_features["n_positive"].sum()))
print("Total excluded observations:",
      int(patient_features["n_excluded"].sum()))

print("\nExample:")
print(patient_features[
    ["case_id", "diagnosis_ids", "n_positive", "n_excluded"]
].head(5).to_string(index=False))

Test cases in split: 1959
Cases extracted: 1959
Missing files: 0
Cases with positive phenotypes: 1959
Total positive observations: 18874
Total excluded observations: 26931

Example:
                       case_id diagnosis_ids  n_positive  n_excluded
PMID_28239884_Family_1_proband   OMIM:148600           1           3
PMID_28239884_Family_2_proband   OMIM:148600           1           3
PMID_28239884_Family_3_proband   OMIM:148600           1           3
          PMID_1680289_patient   OMIM:102700           6           0
 PMID_30733661_4_year_old_girl   OMIM:620475          11           0


In [ ]:
import torch
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

ckpt_path = DATA / "rare_disease_gnn_training_only.pt"
ckpt = torch.load(
    ckpt_path,
    map_location="cpu",
    weights_only=True
)

train_split = pd.read_csv(
    DATA / "disease_level_split.csv",
    dtype=str
)

print("Checkpoint disease IDs:", len(ckpt["disease_ids"]))
print("Checkpoint phenotype IDs:", len(ckpt["phenotype_ids"]))
print("Checkpoint node count:", ckpt["num_nodes"])
print("Checkpoint hidden dimension:", ckpt["hidden_dim"])

print("\nCheckpoint mapping examples:")
print("Disease IDs:", ckpt["disease_ids"][:5])
print("Phenotype IDs:", ckpt["phenotype_ids"][:5])

print("\nDisease split columns:", train_split.columns.tolist())
print(train_split.head(3).to_string(index=False))

print("\nCheckpoint disease index sample:")
for disease_id in ckpt["disease_ids"][:5]:
    print(disease_id, "->", ckpt["disease_to_idx"].get(disease_id))

print("\nCheckpoint phenotype index sample:")
for hpo_id in ckpt["phenotype_ids"][:5]:
    print(hpo_id, "->", ckpt["phenotype_to_idx"].get(hpo_id))

Checkpoint disease IDs: 7340
Checkpoint phenotype IDs: 9982
Checkpoint node count: 17322
Checkpoint hidden dimension: 64

Checkpoint mapping examples:
Disease IDs: ['OMIM:100300', 'OMIM:100800', 'OMIM:101000', 'OMIM:101200', 'OMIM:101400']
Phenotype IDs: ['HP:0000002', 'HP:0000003', 'HP:0000006', 'HP:0000007', 'HP:0000008']

Disease split columns: ['disease_id', 'split']
 disease_id split
OMIM:100100  test
OMIM:100300 train
OMIM:100800 train

Checkpoint disease index sample:
OMIM:100300 -> 0
OMIM:100800 -> 1
OMIM:101000 -> 2
OMIM:101200 -> 3
OMIM:101400 -> 4

Checkpoint phenotype index sample:
HP:0000002 -> 7340
HP:0000003 -> 7341
HP:0000006 -> 7342
HP:0000007 -> 7343
HP:0000008 -> 7344


In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch_geometric.nn import GCNConv

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
PACKETS = PROJECT / "all_phenopackets"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load checkpoint and test split
ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)
test_split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
test_split = test_split[test_split["split"] == "test"].copy()

# Reconstruct the checkpoint's GCN architecture
class RareDiseaseGCN(torch.nn.Module):
    def __init__(self, num_nodes, hidden_dim=64):
        super().__init__()
        self.embedding = torch.nn.Embedding(num_nodes, hidden_dim)
        self.conv1 = GCNConv(hidden_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

    def forward(self, edge_index):
        x = self.embedding.weight
        x = self.conv1(x, edge_index).relu()
        return self.conv2(x, edge_index)

model = RareDiseaseGCN(ckpt["num_nodes"], ckpt["hidden_dim"])
model.load_state_dict(ckpt["model_state_dict"], strict=True)
model = model.to(DEVICE).eval()

# Rebuild the disease-phenotype training graph from verified mappings
ann = pd.read_csv(DATA / "gnn_training_annotations.csv", dtype=str)
ann = ann.drop_duplicates()

disease_to_idx = ckpt["disease_to_idx"]
phenotype_to_idx = ckpt["phenotype_to_idx"]

pairs = []
for r in ann.itertuples(index=False):
    d = disease_to_idx.get(r.disease_id)
    h = phenotype_to_idx.get(r.hpo_id)
    if d is not None and h is not None:
        pairs.append((d, h))
        pairs.append((h, d))

edge_index = torch.tensor(pairs, dtype=torch.long).T.contiguous().to(DEVICE)

# Compute embeddings
with torch.no_grad():
    embeddings = model(edge_index).cpu().numpy()

disease_ids = ckpt["disease_ids"]
disease_vecs = embeddings[
    [disease_to_idx[d] for d in disease_ids]
]
phenotype_ids = ckpt["phenotype_ids"]
phenotype_vecs = embeddings[
    [phenotype_to_idx[h] for h in phenotype_ids]
]
hpo_to_row = {h: i for i, h in enumerate(phenotype_ids)}

# Index patient JSONs
json_files = {p.stem: p for p in PACKETS.rglob("*.json")}

results = []
for row in test_split.itertuples(index=False):
    path = json_files.get(row.case_id)
    if path is None:
        continue

    with path.open(encoding="utf-8") as f:
        packet = json.load(f)

    present, excluded = set(), set()
    for feature in packet.get("phenotypicFeatures", []):
        hpo = feature.get("type", {}).get("id")
        if not isinstance(hpo, str) or not hpo.startswith("HP:"):
            continue
        if feature.get("excluded") is True:
            excluded.add(hpo)
        else:
            present.add(hpo)
    present -= excluded

    rows = [hpo_to_row[h] for h in present if h in hpo_to_row]
    if not rows:
        continue

    # Mean cosine similarity between positive phenotype embeddings
    # and each candidate disease embedding.
    pvecs = phenotype_vecs[rows]
    pvecs = pvecs / np.maximum(
        np.linalg.norm(pvecs, axis=1, keepdims=True), 1e-12
    )
    dvecs = disease_vecs / np.maximum(
        np.linalg.norm(disease_vecs, axis=1, keepdims=True), 1e-12
    )
    scores = (pvecs @ dvecs.T).mean(axis=0)
    order = np.argsort(-scores)

    candidates = [disease_ids[i] for i in order]
    target_ids = set(str(row.diagnosis_ids).split(";"))

    ranks = [
        i + 1 for i, d in enumerate(candidates)
        if d in target_ids
    ]
    results.append({
        "case_id": row.case_id,
        "diagnosis_ids": row.diagnosis_ids,
        "n_positive_hpo": len(present),
        "n_recognized_hpo": len(rows),
        "target_in_candidate_set": bool(ranks),
        "rank": min(ranks) if ranks else np.nan,
        "top1": bool(ranks and min(ranks) <= 1),
        "top5": bool(ranks and min(ranks) <= 5),
        "top10": bool(ranks and min(ranks) <= 10),
        "mrr": 1 / min(ranks) if ranks else 0.0,
        "top_prediction": candidates[0],
    })

gnn_results = pd.DataFrame(results)
out_path = DATA / "patient_gnn_training_disease_rankings.csv"
gnn_results.to_csv(out_path, index=False)

print("Device:", DEVICE)
print("Cases ranked:", len(gnn_results))
print("Candidate diseases:", len(disease_ids))
print("Target coverage:", round(gnn_results["target_in_candidate_set"].mean(), 4))
print("Top-1:", round(gnn_results["top1"].mean(), 4))
print("Top-5:", round(gnn_results["top5"].mean(), 4))
print("Top-10:", round(gnn_results["top10"].mean(), 4))
print("MRR:", round(gnn_results["mrr"].mean(), 4))
print("Saved:", out_path)

AttributeError: 'Pandas' object has no attribute 'disease_id'

In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ann = pd.read_csv(DATA / "gnn_training_annotations.csv", nrows=5)

print("Columns:", ann.columns.tolist())
print("\nSample rows:")
print(ann.to_string(index=False))

Columns: ['database_id', 'disease_name', 'qualifier', 'hpo_id', 'reference', 'evidence', 'onset', 'frequency', 'sex', 'modifier', 'aspect', 'biocuration']

Sample rows:
database_id                                  disease_name  qualifier     hpo_id     reference evidence  onset frequency  sex  modifier aspect               biocuration
OMIM:619340 Developmental and epileptic encephalopathy 96        NaN HP:0011097 PMID:31675180      PCS    NaN       1/2  NaN       NaN      P HPO:probinson[2021-06-21]
OMIM:619340 Developmental and epileptic encephalopathy 96        NaN HP:0002187 PMID:31675180      PCS    NaN       1/1  NaN       NaN      P HPO:probinson[2021-06-21]
OMIM:619340 Developmental and epileptic encephalopathy 96        NaN HP:0001518 PMID:31675180      PCS    NaN       1/2  NaN       NaN      P HPO:probinson[2021-06-21]
OMIM:619340 Developmental and epileptic encephalopathy 96        NaN HP:0032792 PMID:31675180      PCS    NaN       1/2  NaN       NaN      P HPO:probinson[202

In [ ]:
import pandas as pd
import torch
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

ann = pd.read_csv(DATA / "gnn_training_annotations.csv", dtype=str)

# Keep explicitly negated annotations out of positive graph edges.
if "qualifier" in ann.columns:
    ann = ann[
        ann["qualifier"].isna()
        | ~ann["qualifier"].str.contains(
            "NOT", case=False, na=False
        )
    ]

ann = ann.drop_duplicates(subset=["database_id", "hpo_id"])

disease_to_idx = ckpt["disease_to_idx"]
phenotype_to_idx = ckpt["phenotype_to_idx"]

pairs = []
unmapped_diseases = set()
unmapped_phenotypes = set()

for r in ann.itertuples(index=False):
    d = disease_to_idx.get(r.database_id)
    h = phenotype_to_idx.get(r.hpo_id)

    if d is None:
        unmapped_diseases.add(r.database_id)
    if h is None:
        unmapped_phenotypes.add(r.hpo_id)

    if d is not None and h is not None:
        pairs.append((d, h))
        pairs.append((h, d))

edge_index = torch.tensor(
    pairs, dtype=torch.long
).T.contiguous().to(DEVICE)

print("Annotation pairs after filtering:", len(ann))
print("Mapped bidirectional edges:", edge_index.shape[1])
print("Unmapped disease IDs:", len(unmapped_diseases))
print("Unmapped HPO IDs:", len(unmapped_phenotypes))
print("Edge index shape:", tuple(edge_index.shape))
print("Device:", DEVICE)

Annotation pairs after filtering: 179287
Mapped bidirectional edges: 358574
Unmapped disease IDs: 0
Unmapped HPO IDs: 0
Edge index shape: (2, 358574)
Device: cuda


In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

original = pd.read_csv(DATA / "gnn_training_annotations.csv", dtype=str)
original_pairs = original[["database_id", "hpo_id"]].drop_duplicates()

print("Original unique disease-HPO pairs:", len(original_pairs))
print("Rows with qualifier values:")
print(original["qualifier"].value_counts(dropna=False).to_string())

print("\nPairs with a non-empty qualifier:")
print(
    original.loc[
        original["qualifier"].notna(),
        ["database_id", "hpo_id", "qualifier"]
    ].drop_duplicates().head(20).to_string(index=False)
)

print("\nCheckpoint association count:", 179653)

Original unique disease-HPO pairs: 179653
Rows with qualifier values:
qualifier
NaN    179287
NOT       366

Pairs with a non-empty qualifier:
 database_id     hpo_id qualifier
  ORPHA:1401 HP:0001270       NOT
  ORPHA:1401 HP:0200041       NOT
ORPHA:240094 HP:0002063       NOT
ORPHA:240094 HP:0002067       NOT
 ORPHA:79408 HP:0000982       NOT
 ORPHA:79408 HP:0002671       NOT
 ORPHA:79408 HP:0006297       NOT
 ORPHA:79408 HP:0012252       NOT
 ORPHA:79409 HP:0000478       NOT
 ORPHA:79409 HP:0000982       NOT
 ORPHA:79409 HP:0001965       NOT
 ORPHA:79409 HP:0002671       NOT
 ORPHA:79409 HP:0002860       NOT
 ORPHA:79409 HP:0006297       NOT
 ORPHA:79409 HP:0012056       NOT
 ORPHA:79409 HP:0012252       NOT
 ORPHA:79411 HP:0000079       NOT
 ORPHA:79411 HP:0000478       NOT
 ORPHA:79411 HP:0000982       NOT
 ORPHA:79411 HP:0001510       NOT

Checkpoint association count: 179653


In [ ]:
import pandas as pd
import torch
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

ann = pd.read_csv(DATA / "gnn_training_annotations.csv", dtype=str)
ann = ann.drop_duplicates(subset=["database_id", "hpo_id"])

disease_to_idx = ckpt["disease_to_idx"]
phenotype_to_idx = ckpt["phenotype_to_idx"]

pairs = []
unmapped = 0

for r in ann.itertuples(index=False):
    d = disease_to_idx.get(r.database_id)
    h = phenotype_to_idx.get(r.hpo_id)

    if d is None or h is None:
        unmapped += 1
        continue

    pairs.append((d, h))
    pairs.append((h, d))

edge_index = torch.tensor(
    pairs, dtype=torch.long
).T.contiguous().to(DEVICE)

print("Unique annotation pairs:", len(ann))
print("Expected checkpoint associations:", ckpt["training_association_count"])
print("Reconstructed associations:", edge_index.shape[1] // 2)
print("Directed edges:", edge_index.shape[1])
print("Unmapped pairs:", unmapped)
print("Edge index shape:", tuple(edge_index.shape))

Unique annotation pairs: 179653
Expected checkpoint associations: 179653
Reconstructed associations: 179653
Directed edges: 359306
Unmapped pairs: 0
Edge index shape: (2, 359306)


In [ ]:
# Use the verified graph reconstructed in Step 23D.
# Do not filter out NOT-qualified associations for this checkpoint reproduction.
assert edge_index.shape == (2, 359306)

model = RareDiseaseGCN(
    ckpt["num_nodes"],
    ckpt["hidden_dim"]
)
model.load_state_dict(ckpt["model_state_dict"], strict=True)
model = model.to(DEVICE).eval()

with torch.no_grad():
    embeddings = model(edge_index).cpu().numpy()

print("Embedding shape:", embeddings.shape)
print("All embeddings finite:", bool(__import__("numpy").isfinite(embeddings).all()))

Embedding shape: (17322, 64)
All embeddings finite: True


In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"
PACKETS = PROJECT / "all_phenopackets"

# Verify the expected in-memory variables
assert embeddings.shape == (ckpt["num_nodes"], ckpt["hidden_dim"])

disease_ids = ckpt["disease_ids"]
phenotype_ids = ckpt["phenotype_ids"]
disease_to_idx = ckpt["disease_to_idx"]
phenotype_to_idx = ckpt["phenotype_to_idx"]

disease_vecs = embeddings[[disease_to_idx[d] for d in disease_ids]]
phenotype_vecs = embeddings[[phenotype_to_idx[h] for h in phenotype_ids]]

def normalize_rows(x):
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-12)

disease_vecs = normalize_rows(disease_vecs)
hpo_to_row = {h: i for i, h in enumerate(phenotype_ids)}

test_split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
test_split = test_split[test_split["split"] == "test"].copy()
json_files = {p.stem: p for p in PACKETS.rglob("*.json")}

results = []

for row in test_split.itertuples(index=False):
    path = json_files.get(row.case_id)
    if path is None:
        continue

    with path.open(encoding="utf-8") as f:
        packet = json.load(f)

    positive, excluded = set(), set()

    for feature in packet.get("phenotypicFeatures", []):
        hpo = feature.get("type", {}).get("id")
        if not isinstance(hpo, str) or not hpo.startswith("HP:"):
            continue
        if feature.get("excluded") is True:
            excluded.add(hpo)
        else:
            positive.add(hpo)

    positive -= excluded
    recognized = [hpo_to_row[h] for h in positive if h in hpo_to_row]

    target_ids = set(str(row.diagnosis_ids).split(";"))
    candidate_indices = [
        i for i, disease_id in enumerate(disease_ids)
        if disease_id in target_ids
    ]

    if not recognized:
        rank = np.nan
        top_prediction = None
    else:
        patient_vecs = normalize_rows(phenotype_vecs[recognized])
        scores = (patient_vecs @ disease_vecs.T).mean(axis=0)
        order = np.argsort(-scores)

        rank_positions = [
            int(np.where(order == i)[0][0]) + 1
            for i in candidate_indices
        ]
        rank = min(rank_positions) if rank_positions else np.nan
        top_prediction = disease_ids[int(order[0])]

    results.append({
        "case_id": row.case_id,
        "diagnosis_ids": row.diagnosis_ids,
        "n_positive_hpo": len(positive),
        "n_recognized_hpo": len(recognized),
        "target_in_candidate_set": bool(candidate_indices),
        "rank": rank,
        "top1": bool(pd.notna(rank) and rank <= 1),
        "top5": bool(pd.notna(rank) and rank <= 5),
        "top10": bool(pd.notna(rank) and rank <= 10),
        "mrr": 1.0 / rank if pd.notna(rank) else 0.0,
        "top_prediction": top_prediction,
    })

gnn_results = pd.DataFrame(results)
out_path = DATA / "patient_gnn_training_disease_rankings.csv"
gnn_results.to_csv(out_path, index=False)

print("Cases evaluated:", len(gnn_results))
print("Candidate diseases:", len(disease_ids))
print("Target candidate coverage:",
      round(gnn_results["target_in_candidate_set"].mean(), 4))
print("Top-1:", round(gnn_results["top1"].mean(), 4))
print("Top-5:", round(gnn_results["top5"].mean(), 4))
print("Top-10:", round(gnn_results["top10"].mean(), 4))
print("MRR:", round(gnn_results["mrr"].mean(), 4))
print("Saved:", out_path)

Cases evaluated: 1959
Candidate diseases: 7340
Target candidate coverage: 0.7616
Top-1: 0.0316
Top-5: 0.0572
Top-10: 0.0781
MRR: 0.0476
Saved: /content/drive/MyDrive/Rare Disease ai/processed/patient_gnn_training_disease_rankings.csv


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

ann = pd.read_csv(DATA / "gnn_training_annotations.csv", dtype=str)
ann = ann.drop_duplicates(["database_id", "hpo_id"])

disease_to_idx = ckpt["disease_to_idx"]
phenotype_to_idx = ckpt["phenotype_to_idx"]

# Reuse the embeddings already computed.
E = embeddings
rng = np.random.default_rng(42)

positive_scores = []
negative_scores = []

# Sample positive pairs and randomly corrupted pairs.
sample = ann.sample(n=min(20000, len(ann)), random_state=42)

for r in sample.itertuples(index=False):
    d = disease_to_idx.get(r.database_id)
    h = phenotype_to_idx.get(r.hpo_id)
    if d is None or h is None:
        continue

    dv = E[d]
    hv = E[h]
    positive_scores.append(
        np.dot(dv, hv) /
        max(np.linalg.norm(dv) * np.linalg.norm(hv), 1e-12)
    )

    # Random disease–phenotype pair; may occasionally be a true pair.
    random_d = int(rng.integers(0, len(ckpt["disease_ids"])))
    random_d_id = ckpt["disease_ids"][random_d]

    if random_d_id == r.database_id:
        random_d = (random_d + 1) % len(ckpt["disease_ids"])

    ndv = E[random_d]
    negative_scores.append(
        np.dot(ndv, hv) /
        max(np.linalg.norm(ndv) * np.linalg.norm(hv), 1e-12)
    )

print("Positive pairs:", len(positive_scores))
print("Random comparison pairs:", len(negative_scores))
print("Mean positive cosine:", round(float(np.mean(positive_scores)), 4))
print("Mean random cosine:", round(float(np.mean(negative_scores)), 4))
print("Median positive cosine:", round(float(np.median(positive_scores)), 4))
print("Median random cosine:", round(float(np.median(negative_scores)), 4))
print("Positive score > random score:",
      round(float(np.mean(np.array(positive_scores) >
                          np.array(negative_scores))), 4))

Positive pairs: 20000
Random comparison pairs: 20000
Mean positive cosine: 0.7935
Mean random cosine: 0.5904
Median positive cosine: 0.9026
Median random cosine: 0.753
Positive score > random score: 0.7345


In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

baseline_path = DATA / "patient_phenotype_only_baseline.csv"
gnn_path = DATA / "patient_gnn_training_disease_rankings.csv"

baseline = pd.read_csv(baseline_path)
gnn = pd.read_csv(gnn_path)

print("Baseline columns:", baseline.columns.tolist())
print("GNN columns:", gnn.columns.tolist())

print("\nBaseline sample:")
print(baseline.head(3).to_string(index=False))

print("\nGNN sample:")
print(gnn.head(3).to_string(index=False))

if "case_id" in baseline.columns and "case_id" in gnn.columns:
    print("\nUnique baseline case IDs:", baseline["case_id"].nunique())
    print("Unique GNN case IDs:", gnn["case_id"].nunique())
    print("Overlapping case IDs:",
          baseline["case_id"].isin(gnn["case_id"]).sum())

Baseline columns: ['case_id', 'pmid', 'diagnosis_ids', 'rank', 'top1', 'top5', 'top10', 'candidate_covered']
GNN columns: ['case_id', 'diagnosis_ids', 'n_positive_hpo', 'n_recognized_hpo', 'target_in_candidate_set', 'rank', 'top1', 'top5', 'top10', 'mrr', 'top_prediction']

Baseline sample:
                       case_id     pmid diagnosis_ids  rank  top1  top5  top10  candidate_covered
PMID_28239884_Family_1_proband 28239884   OMIM:148600   3.0     0     1      1                  1
PMID_28239884_Family_2_proband 28239884   OMIM:148600   3.0     0     1      1                  1
PMID_28239884_Family_3_proband 28239884   OMIM:148600   3.0     0     1      1                  1

GNN sample:
                       case_id diagnosis_ids  n_positive_hpo  n_recognized_hpo  target_in_candidate_set  rank  top1  top5  top10  mrr top_prediction
PMID_28239884_Family_1_proband   OMIM:148600               1                 1                     True   1.0  True  True   True  1.0    OMIM:148600
PMID_

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

baseline = pd.read_csv(DATA / "patient_phenotype_only_baseline.csv")
gnn = pd.read_csv(DATA / "patient_gnn_training_disease_rankings.csv")

# Join by case ID and verify the recorded diagnosis agrees.
paired = baseline.merge(
    gnn,
    on="case_id",
    suffixes=("_baseline", "_gnn"),
    validate="one_to_one"
)

assert len(paired) == 1959
assert (
    paired["diagnosis_ids_baseline"].fillna("")
    == paired["diagnosis_ids_gnn"].fillna("")
).all()

def summarize(df, prefix):
    rank = pd.to_numeric(df[f"rank_{prefix}"], errors="coerce")
    return {
        "cases": len(df),
        "target_coverage": df[
            "candidate_covered" if prefix == "baseline"
            else "target_in_candidate_set"
        ].astype(bool).mean(),
        "Top-1": (rank <= 1).mean(),
        "Top-5": (rank <= 5).mean(),
        "Top-10": (rank <= 10).mean(),
        "MRR": (1 / rank).where(rank.notna(), 0).mean(),
    }

overall = pd.DataFrame({
    "Phenotype baseline": summarize(paired, "baseline"),
    "GNN embedding score": summarize(paired, "gnn"),
}).T

print("OVERALL RESULTS")
print(overall.round(4).to_string())

# Compare only cases where both methods contain the target diagnosis.
both_covered = paired[
    paired["candidate_covered"].astype(bool)
    & paired["target_in_candidate_set"].astype(bool)
].copy()

print("\nCASES COVERED BY BOTH METHODS:", len(both_covered))
conditional = pd.DataFrame({
    "Phenotype baseline": summarize(both_covered, "baseline"),
    "GNN embedding score": summarize(both_covered, "gnn"),
}).T
print(conditional.round(4).to_string())

# Paired win/tie/loss using reciprocal rank.
rb = pd.to_numeric(both_covered["rank_baseline"], errors="coerce")
rg = pd.to_numeric(both_covered["rank_gnn"], errors="coerce")
rr_baseline = (1 / rb).fillna(0)
rr_gnn = (1 / rg).fillna(0)

print("\nPAIRED RANK COMPARISON (both-covered cases)")
print("GNN better:", int((rr_gnn > rr_baseline).sum()))
print("Tie:", int((rr_gnn == rr_baseline).sum()))
print("Baseline better:", int((rr_gnn < rr_baseline).sum()))

out = DATA / "patient_baseline_vs_gnn_paired.csv"
paired.to_csv(out, index=False)
print("\nSaved paired results:", out)

OVERALL RESULTS
                      cases  target_coverage   Top-1   Top-5  Top-10     MRR
Phenotype baseline   1959.0           0.7917  0.1138  0.1960  0.2210  0.1527
GNN embedding score  1959.0           0.7616  0.0316  0.0572  0.0781  0.0476

CASES COVERED BY BOTH METHODS: 1181
                      cases  target_coverage   Top-1   Top-5  Top-10     MRR
Phenotype baseline   1181.0              1.0  0.1160  0.2235  0.2566  0.1659
GNN embedding score  1181.0              1.0  0.0212  0.0457  0.0754  0.0381

PAIRED RANK COMPARISON (both-covered cases)
GNN better: 528
Tie: 5
Baseline better: 648

Saved paired results: /content/drive/MyDrive/Rare Disease ai/processed/patient_baseline_vs_gnn_paired.csv


In [ ]:
import numpy as np
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

# These variables should still exist from the embedding evaluation.
assert "disease_vecs" in globals()
assert "phenotype_vecs" in globals()

# Normalize disease vectors.
D = disease_vecs / np.maximum(
    np.linalg.norm(disease_vecs, axis=1, keepdims=True), 1e-12
)

# Sample random pairs of distinct disease embeddings.
rng = np.random.default_rng(42)
n = min(50000, len(D) * 5)

i = rng.integers(0, len(D), size=n)
j = rng.integers(0, len(D), size=n)
keep = i != j

cosines = np.sum(D[i[keep]] * D[j[keep]], axis=1)

print("Disease embeddings:", D.shape)
print("Random distinct disease pairs:", len(cosines))
print("Mean cosine similarity:", round(float(cosines.mean()), 4))
print("Median cosine similarity:", round(float(np.median(cosines)), 4))
print("10th percentile:", round(float(np.percentile(cosines, 10)), 4))
print("90th percentile:", round(float(np.percentile(cosines, 90)), 4))

Disease embeddings: (7340, 64)
Random distinct disease pairs: 36695
Mean cosine similarity: 0.4671
Median cosine similarity: 0.5857
10th percentile: -0.1783
90th percentile: 0.9064


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

# Reuse the GNN ranking output and embeddings already in memory.
gnn = pd.read_csv(DATA / "patient_gnn_training_disease_rankings.csv")

D = disease_vecs / np.maximum(
    np.linalg.norm(disease_vecs, axis=1, keepdims=True), 1e-12
)
P = phenotype_vecs / np.maximum(
    np.linalg.norm(phenotype_vecs, axis=1, keepdims=True), 1e-12
)

disease_to_col = {d: i for i, d in enumerate(ckpt["disease_ids"])}
hpo_to_row = {h: i for i, h in enumerate(ckpt["phenotype_ids"])}

json_files = {p.stem: p for p in PACKETS.rglob("*.json")}
diagnostic = []

for row in gnn.itertuples(index=False):
    path = json_files.get(row.case_id)
    if path is None:
        continue

    import json
    with path.open(encoding="utf-8") as f:
        packet = json.load(f)

    positive, excluded = set(), set()
    for feature in packet.get("phenotypicFeatures", []):
        hpo = feature.get("type", {}).get("id")
        if not isinstance(hpo, str) or not hpo.startswith("HP:"):
            continue
        (excluded if feature.get("excluded") is True else positive).add(hpo)

    positive -= excluded
    ids = [hpo_to_row[h] for h in positive if h in hpo_to_row]
    if not ids:
        continue

    target_ids = set(str(row.diagnosis_ids).split(";"))
    target_cols = [disease_to_col[d] for d in target_ids if d in disease_to_col]
    if not target_cols:
        continue

    scores = (P[ids] @ D.T).mean(axis=0)
    target_score = max(scores[j] for j in target_cols)

    # Average score of all non-target candidates for this patient.
    mask = np.ones(len(scores), dtype=bool)
    mask[target_cols] = False
    other_scores = scores[mask]

    diagnostic.append({
        "case_id": row.case_id,
        "target_score": float(target_score),
        "mean_other_score": float(other_scores.mean()),
        "target_above_other_mean": bool(target_score > other_scores.mean()),
        "target_percentile": float((other_scores < target_score).mean()),
    })

diag = pd.DataFrame(diagnostic)

print("Cases with target in GNN candidate set:", len(diag))
print("Mean target score:", round(diag["target_score"].mean(), 4))
print("Mean non-target score:", round(diag["mean_other_score"].mean(), 4))
print("Target above non-target mean:",
      round(diag["target_above_other_mean"].mean(), 4))
print("Mean target percentile:",
      round(diag["target_percentile"].mean(), 4)).


SyntaxError: invalid syntax (597861943.py, line 73)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

# Reuse the GNN ranking output and embeddings already in memory.
gnn = pd.read_csv(DATA / "patient_gnn_training_disease_rankings.csv")

D = disease_vecs / np.maximum(
    np.linalg.norm(disease_vecs, axis=1, keepdims=True), 1e-12
)
P = phenotype_vecs / np.maximum(
    np.linalg.norm(phenotype_vecs, axis=1, keepdims=True), 1e-12
)

disease_to_col = {d: i for i, d in enumerate(ckpt["disease_ids"])}
hpo_to_row = {h: i for i, h in enumerate(ckpt["phenotype_ids"])}

json_files = {p.stem: p for p in PACKETS.rglob("*.json")}
diagnostic = []

for row in gnn.itertuples(index=False):
    path = json_files.get(row.case_id)
    if path is None:
        continue

    import json
    with path.open(encoding="utf-8") as f:
        packet = json.load(f)

    positive, excluded = set(), set()
    for feature in packet.get("phenotypicFeatures", []):
        hpo = feature.get("type", {}).get("id")
        if not isinstance(hpo, str) or not hpo.startswith("HP:"):
            continue
        (excluded if feature.get("excluded") is True else positive).add(hpo)

    positive -= excluded
    ids = [hpo_to_row[h] for h in positive if h in hpo_to_row]
    if not ids:
        continue

    target_ids = set(str(row.diagnosis_ids).split(";"))
    target_cols = [disease_to_col[d] for d in target_ids if d in disease_to_col]
    if not target_cols:
        continue

    scores = (P[ids] @ D.T).mean(axis=0)
    target_score = max(scores[j] for j in target_cols)

    # Average score of all non-target candidates for this patient.
    mask = np.ones(len(scores), dtype=bool)
    mask[target_cols] = False
    other_scores = scores[mask]

    diagnostic.append({
        "case_id": row.case_id,
        "target_score": float(target_score),
        "mean_other_score": float(other_scores.mean()),
        "target_above_other_mean": bool(target_score > other_scores.mean()),
        "target_percentile": float((other_scores < target_score).mean()),
    })

diag = pd.DataFrame(diagnostic)

print("Cases with target in GNN candidate set:", len(diag))
print("Mean target score:", round(diag["target_score"].mean(), 4))
print("Mean non-target score:", round(diag["mean_other_score"].mean(), 4))
print("Target above non-target mean:",
      round(diag["target_above_other_mean"].mean(), 4))
print("Mean target percentile:",
      round(diag["target_percentile"].mean(), 4))

Cases with target in GNN candidate set: 1492
Mean target score: 0.8527
Mean non-target score: 0.6097
Target above non-target mean: 0.9672
Mean target percentile: 0.8216


In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")
PACKETS = Path("/content/drive/MyDrive/Rare Disease ai/all_phenopackets")

gnn = pd.read_csv(DATA / "patient_gnn_training_disease_rankings.csv")
json_files = {p.stem: p for p in PACKETS.rglob("*.json")}

D = disease_vecs / np.maximum(
    np.linalg.norm(disease_vecs, axis=1, keepdims=True), 1e-12
)
P = phenotype_vecs / np.maximum(
    np.linalg.norm(phenotype_vecs, axis=1, keepdims=True), 1e-12
)
disease_to_col = {d: i for i, d in enumerate(ckpt["disease_ids"])}
hpo_to_row = {h: i for i, h in enumerate(ckpt["phenotype_ids"])}

hard_cases = []

for row in gnn.itertuples(index=False):
    if pd.isna(row.rank) or row.rank <= 10:
        continue

    path = json_files.get(row.case_id)
    if path is None:
        continue

    with path.open(encoding="utf-8") as f:
        packet = json.load(f)

    positive, excluded = set(), set()
    for feature in packet.get("phenotypicFeatures", []):
        hpo = feature.get("type", {}).get("id")
        if not isinstance(hpo, str) or not hpo.startswith("HP:"):
            continue
        (excluded if feature.get("excluded") is True else positive).add(hpo)

    positive -= excluded
    ids = [hpo_to_row[h] for h in positive if h in hpo_to_row]
    targets = [
        disease_to_col[d]
        for d in set(str(row.diagnosis_ids).split(";"))
        if d in disease_to_col
    ]

    if not ids or not targets:
        continue

    scores = (P[ids] @ D.T).mean(axis=0)
    order = np.argsort(-scores)

    target_score = max(scores[t] for t in targets)
    top_alternatives = [
        (ckpt["disease_ids"][i], round(float(scores[i]), 4))
        for i in order if i not in targets
    ][:5]

    hard_cases.append({
        "case_id": row.case_id,
        "diagnosis": row.diagnosis_ids,
        "target_rank": int(row.rank),
        "target_score": round(float(target_score), 4),
        "top_alternatives": top_alternatives,
        "positive_hpo_count": len(positive),
    })

hard_df = pd.DataFrame(hard_cases)
print("Cases with target rank > 10 and target represented:", len(hard_df))
print(hard_df.head(10).to_string(index=False))

Cases with target rank > 10 and target represented: 1339
                          case_id   diagnosis  target_rank  target_score                                                                                                      top_alternatives  positive_hpo_count
             PMID_1680289_patient OMIM:102700         3278        0.7151      [(ORPHA:394, 0.8285), (OMIM:618885, 0.8275), (OMIM:122470, 0.8274), (ORPHA:79282, 0.8273), (ORPHA:3253, 0.8245)]                   6
          PMID_21989597_Proband_2 OMIM:621196         1105        0.6168     [(OMIM:621080, 0.7609), (OMIM:619221, 0.7403), (ORPHA:99845, 0.7387), (OMIM:306955, 0.7316), (ORPHA:2248, 0.725)]                   2
          PMID_21989597_Proband_1 OMIM:621196         1105        0.6168     [(OMIM:621080, 0.7609), (OMIM:619221, 0.7403), (ORPHA:99845, 0.7387), (OMIM:306955, 0.7316), (ORPHA:2248, 0.725)]                   2
PMID_37951597_Family_22_Subject_1 OMIM:620785           23        0.9908  [(ORPHA:352665, 0.9938), 

In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

for filename in [
    "patient_phenotype_only_baseline.csv",
    "patient_publication_split.csv",
    "patient_test_disease_coverage.csv",
]:
    path = DATA / filename
    print(f"\n--- {filename} ---")
    if path.exists():
        df = pd.read_csv(path, nrows=5)
        print("Columns:", df.columns.tolist())
        print(df.head(3).to_string(index=False))
    else:
        print("File not found")

print("\nGNN result rows:", len(pd.read_csv(
    DATA / "patient_gnn_training_disease_rankings.csv"
)))



--- patient_phenotype_only_baseline.csv ---
Columns: ['case_id', 'pmid', 'diagnosis_ids', 'rank', 'top1', 'top5', 'top10', 'candidate_covered']
                       case_id     pmid diagnosis_ids  rank  top1  top5  top10  candidate_covered
PMID_28239884_Family_1_proband 28239884   OMIM:148600   3.0     0     1      1                  1
PMID_28239884_Family_2_proband 28239884   OMIM:148600   3.0     0     1      1                  1
PMID_28239884_Family_3_proband 28239884   OMIM:148600   3.0     0     1      1                  1

--- patient_publication_split.csv ---
Columns: ['case_id', 'pmid', 'split', 'diagnosis_ids']
                    case_id     pmid       split diagnosis_ids
PMID_19836009_Family_E_II_6 19836009 development   OMIM:277600
PMID_19836009_Family_E_II_3 19836009 development   OMIM:277600
PMID_19836009_Family_B_II_4 19836009 development   OMIM:277600

--- patient_test_disease_coverage.csv ---
Columns: ['test_cases', 'cases_with_diagnosis_profile', 'cases_without_dia

In [ ]:
import pandas as pd
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Rare Disease ai")
DATA = PROJECT / "processed"

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
ann = pd.read_csv(DATA / "patient_test_leakage_reduced_annotations.csv", dtype=str)
ckpt = torch.load(
    DATA / "rare_disease_gnn_training_only.pt",
    map_location="cpu",
    weights_only=True
)

development = split[split["split"] == "development"].copy()
gnn_diseases = set(ckpt["disease_ids"])
profile_diseases = set(ann["database_id"].dropna().unique())

development["target_ids"] = development["diagnosis_ids"].fillna("").str.split(";")
development["gnn_covered"] = development["target_ids"].apply(
    lambda ids: any(d in gnn_diseases for d in ids)
)
development["profile_covered"] = development["target_ids"].apply(
    lambda ids: any(d in profile_diseases for d in ids)
)

print("Development cases:", len(development))
print("Unique development publications:", development["pmid"].nunique())
print("GNN candidate coverage:", round(development["gnn_covered"].mean(), 4))
print("Phenotype-profile coverage:", round(development["profile_covered"].mean(), 4))
print("Cases covered by both:", int(
    (development["gnn_covered"] & development["profile_covered"]).sum()
))
print("Cases covered by neither:", int(
    (~development["gnn_covered"] & ~development["profile_covered"]).sum()
))


Development cases: 8386
Unique development publications: 1383
GNN candidate coverage: 0.7913
Phenotype-profile coverage: 0.9982
Cases covered by both: 6621
Cases covered by neither: 0


In [ ]:
import pandas as pd
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Rare Disease ai/processed")

split = pd.read_csv(DATA / "patient_publication_split.csv", dtype=str)
ann = pd.read_csv(DATA / "gnn_training_annotations.csv", dtype=str)

dev_pmids = set(
    split.loc[split["split"] == "development", "pmid"]
    .dropna().astype(str)
)
ann_pmids = set(
    ann["reference"].dropna().astype(str)
    .str.extract(r"PMID:(\d+)")[0].dropna()
)

overlap = dev_pmids & ann_pmids

print("Development publications:", len(dev_pmids))
print("Distinct annotation publication references:", len(ann_pmids))
print("Overlapping publication IDs:", len(overlap))
print("Development publications overlapping annotations:",
      round(len(overlap) / max(len(dev_pmids), 1), 4))
print("Example overlapping IDs:", sorted(overlap)[:15])

Development publications: 1383
Distinct annotation publication references: 7149
Overlapping publication IDs: 806
Development publications overlapping annotations: 0.5828
Example overlapping IDs: ['10077612', '10487826', '10498624', '10560675', '10842287', '10939567', '11050011', '11093277', '11179005', '11181649', '11344241', '11410597', '11479728', '11506412', '11555793']
